# India Quick-Commerce Profitability & Fulfilment Intelligence
## Dark Store, Inventory & Last-Mile Analytics

### End-to-End Analytics Portfolio Project

**Objective:**  
Build an operations intelligence system for a quick-commerce business that evaluates commercial performance, dark-store efficiency, inventory health, fulfilment SLA, last-mile productivity and unit economics.

### Technology Stack
- Python
- Pandas / NumPy
- SQL / DuckDB
- Scikit-learn
- Time-Series Analytics
- Power BI
- Git / GitHub

### Analytical Scope
1. Commercial Performance
2. Dark Store Operations
3. Inventory Intelligence
4. Fulfilment & SLA Analytics
5. Last-Mile Delivery Analytics
6. Cost-to-Serve & Unit Economics
7. Demand Forecasting
8. Stockout Risk Intelligence
9. Operational Prioritisation
10. Management Recommendations

> **Data Disclosure:** This portfolio project uses realistic practice data modelled on Indian quick-commerce operations. It does not contain proprietary operational or financial data from Blinkit, Zepto, Swiggy Instamart or any other quick-commerce company.

In [7]:
# ============================================================
# 00. PROJECT ENVIRONMENT & REPRODUCIBILITY
# ============================================================

# Install the analytical database engine used later for SQL
%pip -q install duckdb openpyxl pyarrow

# -----------------------------
# Core imports
# -----------------------------
import os
import sys
import platform
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import duckdb

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Project directory structure
# -----------------------------
PROJECT_ROOT = Path("/content/quick_commerce_intelligence")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SQL_DIR = PROJECT_ROOT / "sql"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

PROJECT_DIRS = [
    RAW_DIR,
    PROCESSED_DIR,
    SQL_DIR,
    OUTPUT_DIR,
    MODEL_DIR,
    REPORT_DIR
]

for folder in PROJECT_DIRS:
    folder.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Display configuration
# -----------------------------
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# -----------------------------
# Environment audit
# -----------------------------
environment_info = {
    "Project": "Quick-Commerce Profitability & Fulfilment Intelligence",
    "Execution Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Python": sys.version.split()[0],
    "Platform": platform.platform(),
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "DuckDB": duckdb.__version__,
    "Random Seed": SEED
}

print("=" * 72)
print(" QUICK-COMMERCE INTELLIGENCE — PROJECT INITIALIZATION")
print("=" * 72)

for key, value in environment_info.items():
    print(f"{key:<20}: {value}")

print("\nProject directories:")
for folder in PROJECT_DIRS:
    print(f"  ✓ {folder}")

print("\n" + "=" * 72)
print("PROJECT ENVIRONMENT INITIALIZED SUCCESSFULLY")
print("=" * 72)

 QUICK-COMMERCE INTELLIGENCE — PROJECT INITIALIZATION
Project             : Quick-Commerce Profitability & Fulfilment Intelligence
Execution Time      : 2026-08-30 17:07:52
Python              : 3.13.15
Platform            : Linux-6.6.122+-x86_64-with-glibc2.35
Pandas              : 2.2.3
NumPy               : 2.1.3
DuckDB              : 1.3.2
Random Seed         : 42

Project directories:
  ✓ /content/quick_commerce_intelligence/data/raw
  ✓ /content/quick_commerce_intelligence/data/processed
  ✓ /content/quick_commerce_intelligence/sql
  ✓ /content/quick_commerce_intelligence/outputs
  ✓ /content/quick_commerce_intelligence/models
  ✓ /content/quick_commerce_intelligence/reports

PROJECT ENVIRONMENT INITIALIZED SUCCESSFULLY


In [8]:
# ============================================================
# 01. SOURCE DATA INGESTION
# Preserve the original dataset package exactly as downloaded.
# ============================================================

from google.colab import files
import shutil
import zipfile
import hashlib

# ------------------------------------------------------------
# Remove the accidental sample file, if it exists
# ------------------------------------------------------------
for file_path in RAW_DIR.iterdir():
    if "sample" in file_path.name.lower():
        print(f"Removing old sample file: {file_path.name}")
        file_path.unlink()

# ------------------------------------------------------------
# Upload the original Kaggle ZIP
# ------------------------------------------------------------
print("Upload the original Zepto dataset ZIP file.\n")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded.")

uploaded_name = list(uploaded.keys())[0]

source_path = Path("/content") / uploaded_name
archive_path = RAW_DIR / uploaded_name

# Allow safe re-running of this cell
if archive_path.exists():
    archive_path.unlink()

shutil.move(str(source_path), str(archive_path))

# ------------------------------------------------------------
# File metadata + SHA-256 checksum
# ------------------------------------------------------------
file_size_bytes = archive_path.stat().st_size
file_size_mb = file_size_bytes / (1024 ** 2)

sha256 = hashlib.sha256()

with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

archive_hash = sha256.hexdigest()

print("\n" + "=" * 72)
print(" SOURCE PACKAGE RECEIVED")
print("=" * 72)

print(f"Filename      : {archive_path.name}")
print(f"Size          : {file_size_mb:.2f} MB")
print(f"Size (bytes)  : {file_size_bytes:,}")
print(f"SHA-256       : {archive_hash}")

# ------------------------------------------------------------
# Validate ZIP format
# ------------------------------------------------------------
if not zipfile.is_zipfile(archive_path):
    raise ValueError(
        "Uploaded file is not a valid ZIP archive. "
        "Please upload the original Kaggle archive."
    )

print("ZIP validation: PASS ✅")

print("=" * 72)

Upload the original Zepto dataset ZIP file.



Saving archive.zip to archive.zip

 SOURCE PACKAGE RECEIVED
Filename      : archive.zip
Size          : 2.81 MB
Size (bytes)  : 2,948,335
SHA-256       : 4b4fda7ec74d2274b7ddede45c4eeb686de1991314627ead118e81b024762803
ZIP validation: PASS ✅


In [9]:
# ============================================================
# 02. SOURCE PACKAGE EXTRACTION
# Extract the original archive into a controlled raw-data folder.
# ============================================================

EXTRACTED_DIR = RAW_DIR / "extracted"
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

# Clear previously extracted files if the cell is rerun
for item in EXTRACTED_DIR.iterdir():
    if item.is_file():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

with zipfile.ZipFile(archive_path, "r") as zip_ref:

    archive_contents = zip_ref.namelist()

    print("=" * 72)
    print(" ARCHIVE CONTENTS")
    print("=" * 72)

    for item in archive_contents:
        print(f"  • {item}")

    zip_ref.extractall(EXTRACTED_DIR)

print("\nExtraction completed successfully. ✅")

print(f"\nExtracted location:\n{EXTRACTED_DIR}")

 ARCHIVE CONTENTS
  • Zepto_Dataset.xlsx

Extraction completed successfully. ✅

Extracted location:
/content/quick_commerce_intelligence/data/raw/extracted


In [10]:
# ============================================================
# 03. WORKBOOK STRUCTURE AUDIT
# Discover all sheets before any cleaning or transformation.
# ============================================================

excel_files = list(EXTRACTED_DIR.glob("*.xlsx"))

if len(excel_files) == 0:
    raise FileNotFoundError("No Excel workbook found after extraction.")

if len(excel_files) > 1:
    print("Multiple Excel workbooks detected:")
    for f in excel_files:
        print("  •", f.name)

WORKBOOK_PATH = excel_files[0]

excel_file = pd.ExcelFile(WORKBOOK_PATH)

sheet_names = excel_file.sheet_names

print("=" * 72)
print(" WORKBOOK STRUCTURE")
print("=" * 72)

print(f"Workbook : {WORKBOOK_PATH.name}")
print(f"Sheets   : {len(sheet_names)}")

print("\nDetected tables:")

for i, sheet in enumerate(sheet_names, start=1):
    print(f"  {i}. {sheet}")

print("=" * 72)

 WORKBOOK STRUCTURE
Workbook : Zepto_Dataset.xlsx
Sheets   : 6

Detected tables:
  1. customer
  2. product
  3. order
  4. transaction
  5. rating
  6. delivery


In [11]:
# ============================================================
# 04. DATASET MANIFEST & INITIAL QUALITY AUDIT
# ============================================================

tables = {}
manifest_records = []

for sheet in sheet_names:

    df = pd.read_excel(WORKBOOK_PATH, sheet_name=sheet)

    tables[sheet] = df

    row_count = len(df)
    column_count = len(df.columns)
    missing_values = int(df.isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    manifest_records.append({
        "table_name": sheet,
        "rows": row_count,
        "columns": column_count,
        "missing_values": missing_values,
        "duplicate_rows": duplicate_rows
    })

dataset_manifest = pd.DataFrame(manifest_records)

print("=" * 80)
print(" DATASET MANIFEST")
print("=" * 80)

display(dataset_manifest)

print("\nTotal rows across all tables:",
      f"{dataset_manifest['rows'].sum():,}")

print("Total columns across all tables:",
      f"{dataset_manifest['columns'].sum():,}")

print("=" * 80)

 DATASET MANIFEST


,table_name,rows,columns,missing_values,duplicate_rows
0,customer,10000,8,0,0
1,product,1200,5,0,0
2,order,20000,4,0,0
3,transaction,50000,6,0,0
4,rating,20000,4,0,0
5,delivery,20000,5,0,0



Total rows across all tables: 121,200
Total columns across all tables: 32


In [12]:
# ============================================================
# 05. TABLE SCHEMA INSPECTION
# ============================================================

for table_name, df in tables.items():

    print("\n" + "=" * 80)
    print(f" TABLE: {table_name.upper()}")
    print("=" * 80)

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

    schema_df = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "nulls": df.isna().sum().values,
        "unique_values": df.nunique(dropna=True).values
    })

    display(schema_df)

    print("Sample records:")
    display(df.head(3))


 TABLE: CUSTOMER
Shape: 10,000 rows × 8 columns



,column,dtype,non_null,nulls,unique_values
0,customer_id,object,10000,0,10000
1,customer_name,object,10000,0,200
2,email,object,10000,0,9971
3,gender,object,10000,0,3
4,age,int64,10000,0,43
5,city,object,10000,0,10
6,state,object,10000,0,9
7,created_date,datetime64[ns],10000,0,731


Sample records:


,customer_id,customer_name,email,gender,age,city,state,created_date
0,C00001,Vihaan Sharma,vihaan.sharma4507@gmail.com,Male,26,Delhi,Delhi,2024-11-23
1,C00002,Meera Verma,meera.verma9675@gmail.com,Male,19,Delhi,Delhi,2023-08-12
2,C00003,Kabir Singh,kabir.singh9864@gmail.com,Male,30,Jaipur,Rajasthan,2024-03-05



 TABLE: PRODUCT
Shape: 1,200 rows × 5 columns



...

,rating_id,order_id,rating,review
0,R000001,O008814,5,"On-time delivery, good quality."
1,R000002,O017887,5,All items were available and fresh.
2,R000003,O011133,5,"On-time delivery, good quality."



 TABLE: DELIVERY
Shape: 20,000 rows × 5 columns



,column,dtype,non_null,nulls,unique_values
0,delivery_id,object,20000,0,20000
1,order_id,object,20000,0,20000
2,delivery_time_mins,int64,20000,0,103
3,delivery_status,object,20000,0,2
4,distance_km,float64,20000,0,96


Sample records:


,delivery_id,order_id,delivery_time_mins,delivery_status,distance_km
0,D000001,O000001,52,Delivered Late,5.00
1,D000002,O000002,66,Delivered Late,6.50
2,D000003,O000003,31,Delivered On Time,6.50


## 06. Relational Integrity & Business-Rule Audit

Before transforming the source data, the relational structure is validated to ensure that primary keys are unique, foreign keys resolve correctly, numeric fields fall within plausible ranges, and relationships between orders, transactions, deliveries and ratings are internally consistent.

No records are removed during this stage. Potential anomalies are first quantified and evaluated in their business context.

In [13]:
# ============================================================
# 06. RELATIONAL INTEGRITY & BUSINESS-RULE AUDIT
# ============================================================

customers = tables["customer"].copy()
products = tables["product"].copy()
orders = tables["order"].copy()
transactions = tables["transaction"].copy()
ratings = tables["rating"].copy()
deliveries = tables["delivery"].copy()

audit_results = []

def add_audit(category, test, value, expected, status):
    audit_results.append({
        "category": category,
        "test": test,
        "observed": value,
        "expected": expected,
        "status": status
    })


# ============================================================
# A. PRIMARY KEY UNIQUENESS
# ============================================================

primary_keys = {
    "customer": (customers, "customer_id"),
    "product": (products, "product_id"),
    "order": (orders, "order_id"),
    "transaction": (transactions, "transaction_id"),
    "rating": (ratings, "rating_id"),
    "delivery": (deliveries, "delivery_id")
}

for table_name, (df, key) in primary_keys.items():

    duplicate_pk = int(df[key].duplicated().sum())

    add_audit(
        "Primary Key",
        f"{table_name}.{key} duplicate count",
        duplicate_pk,
        0,
        "PASS" if duplicate_pk == 0 else "REVIEW"
    )


# ============================================================
# B. FOREIGN KEY INTEGRITY
# ============================================================

fk_tests = [
    (
        "order.customer_id → customer.customer_id",
        orders["customer_id"],
        customers["customer_id"]
    ),
    (
        "transaction.order_id → order.order_id",
        transactions["order_id"],
        orders["order_id"]
    ),
    (
        "transaction.product_id → product.product_id",
        transactions["product_id"],
        products["product_id"]
    ),
    (
        "rating.order_id → order.order_id",
        ratings["order_id"],
        orders["order_id"]
    ),
    (
        "delivery.order_id → order.order_id",
        deliveries["order_id"],
        orders["order_id"]
    )
]

for relationship, child, parent in fk_tests:

    orphan_count = int((~child.isin(parent)).sum())

    add_audit(
        "Foreign Key",
        relationship,
        orphan_count,
        0,
        "PASS" if orphan_count == 0 else "REVIEW"
    )


# ============================================================
# C. DOMAIN / BUSINESS-RULE CHECKS
# ============================================================

business_rules = [
    (
        "Customer age outside 18–100",
        int(((customers["age"] < 18) | (customers["age"] > 100)).sum())
    ),
    (
        "Product price <= 0",
        int((products["price"] <= 0).sum())
    ),
    (
        "Transaction quantity <= 0",
        int((transactions["quantity"] <= 0).sum())
    ),
    (
        "Transaction amount < 0",
        int((transactions["amount"] < 0).sum())
    ),
    (
        "Rating outside 1–5",
        int(((ratings["rating"] < 1) | (ratings["rating"] > 5)).sum())
    ),
    (
        "Delivery time <= 0 minutes",
        int((deliveries["delivery_time_mins"] <= 0).sum())
    ),
    (
        "Delivery distance < 0 km",
        int((deliveries["distance_km"] < 0).sum())
    )
]

for test, count in business_rules:

    add_audit(
        "Business Rule",
        test,
        count,
        0,
        "PASS" if count == 0 else "REVIEW"
    )


# ============================================================
# D. ORDER COVERAGE ACROSS CHILD TABLES
# ============================================================

all_order_ids = set(orders["order_id"])

transaction_order_ids = set(transactions["order_id"])
rating_order_ids = set(ratings["order_id"])
delivery_order_ids = set(deliveries["order_id"])

orders_without_transactions = len(
    all_order_ids - transaction_order_ids
)

orders_without_ratings = len(
    all_order_ids - rating_order_ids
)

orders_without_delivery = len(
    all_order_ids - delivery_order_ids
)

add_audit(
    "Relationship Coverage",
    "Orders without transaction rows",
    orders_without_transactions,
    "Investigate by order status",
    "REVIEW" if orders_without_transactions > 0 else "PASS"
)

add_audit(
    "Relationship Coverage",
    "Orders without ratings",
    orders_without_ratings,
    "May be valid",
    "INFO"
)

add_audit(
    "Relationship Coverage",
    "Orders without delivery record",
    orders_without_delivery,
    0,
    "PASS" if orders_without_delivery == 0 else "REVIEW"
)


# ============================================================
# E. CARDINALITY CHECKS
# ============================================================

transactions_per_order = (
    transactions.groupby("order_id")
    .size()
)

ratings_per_order = (
    ratings.groupby("order_id")
    .size()
)

deliveries_per_order = (
    deliveries.groupby("order_id")
    .size()
)

multi_rating_orders = int(
    (ratings_per_order > 1).sum()
)

multi_delivery_orders = int(
    (deliveries_per_order > 1).sum()
)

add_audit(
    "Cardinality",
    "Orders with multiple ratings",
    multi_rating_orders,
    "Investigate",
    "INFO"
)

add_audit(
    "Cardinality",
    "Orders with multiple delivery records",
    multi_delivery_orders,
    0,
    "PASS" if multi_delivery_orders == 0 else "REVIEW"
)


# ============================================================
# F. TRANSACTION AMOUNT CONSISTENCY
# Expected line amount = product price × quantity
# ============================================================

amount_check = transactions.merge(
    products[["product_id", "price"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

amount_check["expected_amount"] = (
    amount_check["price"] *
    amount_check["quantity"]
)

amount_check["amount_difference"] = (
    amount_check["amount"] -
    amount_check["expected_amount"]
)

amount_mismatch_count = int(
    (~np.isclose(
        amount_check["amount"],
        amount_check["expected_amount"]
    )).sum()
)

add_audit(
    "Financial Consistency",
    "Transaction amount != price × quantity",
    amount_mismatch_count,
    0,
    "PASS" if amount_mismatch_count == 0 else "REVIEW"
)


# ============================================================
# G. CUSTOMER EMAIL UNIQUENESS
# ============================================================

duplicate_email_rows = int(
    customers["email"].duplicated(keep=False).sum()
)

duplicate_unique_emails = int(
    customers.loc[
        customers["email"].duplicated(keep=False),
        "email"
    ].nunique()
)

add_audit(
    "Customer Quality",
    "Rows involved in duplicated email addresses",
    duplicate_email_rows,
    "Investigate",
    "INFO"
)


# ============================================================
# H. FINAL AUDIT REPORT
# ============================================================

relational_audit = pd.DataFrame(audit_results)

status_order = {
    "REVIEW": 0,
    "INFO": 1,
    "PASS": 2
}

relational_audit["_sort"] = (
    relational_audit["status"]
    .map(status_order)
)

relational_audit = (
    relational_audit
    .sort_values(
        ["_sort", "category", "test"]
    )
    .drop(columns="_sort")
    .reset_index(drop=True)
)

print("=" * 92)
print(" RELATIONAL INTEGRITY & BUSINESS-RULE AUDIT")
print("=" * 92)

display(relational_audit)

print("\nAudit summary:")
display(
    relational_audit["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="tests")
)

print("\n" + "=" * 92)
print(" RELATIONSHIP COVERAGE")
print("=" * 92)

print(f"Total orders                  : {len(orders):,}")
print(f"Orders represented in sales  : {len(transaction_order_ids):,}")
print(f"Orders without transactions  : {orders_without_transactions:,}")
print(f"Orders represented in ratings: {len(rating_order_ids):,}")
print(f"Orders without ratings       : {orders_without_ratings:,}")
print(f"Orders represented in delivery: {len(delivery_order_ids):,}")
print(f"Orders without delivery      : {orders_without_delivery:,}")

print("\nTransaction lines per represented order:")
print(transactions_per_order.describe())

print("\nRatings per rated order:")
print(ratings_per_order.describe())

print("\nDuplicate email diagnostics:")
print(f"Rows involved              : {duplicate_email_rows:,}")
print(f"Distinct duplicated emails : {duplicate_unique_emails:,}")

 RELATIONAL INTEGRITY & BUSINESS-RULE AUDIT


,category,test,observed,expected,status
0,Relationship Coverage,Orders without transaction rows,1628,Investigate by order status,REVIEW
1,Cardinality,Orders with multiple ratings,5319,Investigate,INFO
2,Customer Quality,Rows involved in duplicated email addresses,58,Investigate,INFO
3,Relationship Coverage,Orders without ratings,7445,May be valid,INFO
4,Business Rule,Customer age outside 18–100,0,0,PASS
5,Business Rule,Delivery distance < 0 km,0,0,PASS
6,Business Rule,Delivery time <= 0 minutes,0,0,PASS
7,Business Rule,Product price <= 0,0,0,PASS
8,Business Rule,Rating outside 1–5,0,0,PASS
9,Business Rule,Transaction amount < 0,0,0,PASS



Audit summary:


,status,tests
0,PASS,21
1,INFO,3
2,REVIEW,1



 RELATIONSHIP COVERAGE
Total orders                  : 20,000
Orders represented in sales  : 18,372
Orders without transactions  : 1,628
Orders represented in ratings: 12,555
Orders without ratings       : 7,445
Orders represented in delivery: 20,000
Orders without delivery      : 0

Transaction lines per represented order:
count   18,372.00
mean         2.72
std          1.45
min          1.00
25%          2.00
50%          2.00
75%          4.00
max         10.00
dtype: float64

Ratings per rated order:
count   12,555.00
mean         1.59
std          0.82
min          1.00
25%          1.00
50%          1.00
75%          2.00
max          7.00
dtype: float64

Duplicate email diagnostics:
Rows involved              : 58
Distinct duplicated emails : 29


In [14]:
# ============================================================
# 07. STATUS & CROSS-TABLE CONSISTENCY DIAGNOSTICS
# ============================================================

print("=" * 88)
print(" ORDER STATUS DISTRIBUTION")
print("=" * 88)

display(
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="orders")
)


print("\n" + "=" * 88)
print(" DELIVERY STATUS DISTRIBUTION")
print("=" * 88)

display(
    deliveries["delivery_status"]
    .value_counts(dropna=False)
    .rename_axis("delivery_status")
    .reset_index(name="deliveries")
)


# ------------------------------------------------------------
# Identify whether orders without transactions are associated
# with specific order statuses
# ------------------------------------------------------------

orders_tx_flag = orders.copy()

orders_tx_flag["has_transaction"] = (
    orders_tx_flag["order_id"]
    .isin(transaction_order_ids)
)

tx_coverage_by_status = (
    orders_tx_flag
    .groupby(
        ["order_status", "has_transaction"],
        dropna=False
    )
    .size()
    .reset_index(name="orders")
)

print("\n" + "=" * 88)
print(" TRANSACTION COVERAGE BY ORDER STATUS")
print("=" * 88)

display(tx_coverage_by_status)


# ------------------------------------------------------------
# Order status vs delivery status
# ------------------------------------------------------------

order_delivery_status = orders.merge(
    deliveries[
        ["order_id", "delivery_status"]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

status_matrix = pd.crosstab(
    order_delivery_status["order_status"],
    order_delivery_status["delivery_status"],
    margins=True
)

print("\n" + "=" * 88)
print(" ORDER STATUS × DELIVERY STATUS")
print("=" * 88)

display(status_matrix)

 ORDER STATUS DISTRIBUTION


,order_status,orders
0,Delivered,15618
1,Cancelled,2027
2,Out for Delivery,1366
3,Returned,989



 DELIVERY STATUS DISTRIBUTION


,delivery_status,deliveries
0,Delivered On Time,10064
1,Delivered Late,9936



 TRANSACTION COVERAGE BY ORDER STATUS


,order_status,has_transaction,orders
0,Cancelled,False,153
1,Cancelled,True,1874
2,Delivered,False,1285
3,Delivered,True,14333
4,Out for Delivery,False,127
5,Out for Delivery,True,1239
6,Returned,False,63
7,Returned,True,926



 ORDER STATUS × DELIVERY STATUS


delivery_status,Delivered Late,Delivered On Time,All
order_status,,,
Cancelled,1012,1015,2027
Delivered,7729,7889,15618
Out for Delivery,707,659,1366
Returned,488,501,989
All,9936,10064,20000


## 08. Semantic Consistency Diagnostics

Relational integrity alone does not guarantee business consistency. This stage evaluates cross-table semantics, temporal validity, payment behaviour, product-line structure, customer feedback coverage and the reliability of source-provided delivery SLA labels.

The objective is to distinguish valid business behaviour from synthetic-data artefacts before constructing the analytical model.

In [15]:
# ============================================================
# 08. SEMANTIC CONSISTENCY DIAGNOSTICS
# ============================================================


# ============================================================
# A. CUSTOMER CREATION DATE VS ORDER DATE
# ============================================================

customer_order_dates = orders.merge(
    customers[
        ["customer_id", "created_date"]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

customer_order_dates["order_before_customer_created"] = (
    customer_order_dates["order_date"] <
    customer_order_dates["created_date"]
)

invalid_customer_timeline = int(
    customer_order_dates[
        "order_before_customer_created"
    ].sum()
)

print("=" * 92)
print(" TEMPORAL CONSISTENCY")
print("=" * 92)

print(
    f"Orders placed before customer creation date : "
    f"{invalid_customer_timeline:,}"
)

print(
    f"Total orders                                : "
    f"{len(orders):,}"
)


# ============================================================
# B. PAYMENT-MODE CONSISTENCY WITHIN ORDERS
# ============================================================

payment_modes_per_order = (
    transactions
    .groupby("order_id")["payment_mode"]
    .nunique()
)

multi_payment_orders = int(
    (payment_modes_per_order > 1).sum()
)

print("\n" + "=" * 92)
print(" PAYMENT MODE CONSISTENCY")
print("=" * 92)

print(
    f"Orders represented in transactions : "
    f"{len(payment_modes_per_order):,}"
)

print(
    f"Orders with multiple payment modes : "
    f"{multi_payment_orders:,}"
)

print("\nNumber of payment modes per order:")

display(
    payment_modes_per_order
    .value_counts()
    .sort_index()
    .rename_axis("payment_modes_per_order")
    .reset_index(name="orders")
)


# ============================================================
# C. PRODUCT-LINE DUPLICATION WITHIN ORDERS
# ============================================================

order_product_counts = (
    transactions
    .groupby(
        ["order_id", "product_id"]
    )
    .size()
    .reset_index(name="line_count")
)

repeated_product_pairs = (
    order_product_counts[
        order_product_counts["line_count"] > 1
    ]
)

orders_with_repeated_products = (
    repeated_product_pairs["order_id"]
    .nunique()
)

print("\n" + "=" * 92)
print(" PRODUCT-LINE CONSISTENCY")
print("=" * 92)

print(
    f"Order-product combinations repeated : "
    f"{len(repeated_product_pairs):,}"
)

print(
    f"Orders containing repeated product lines : "
    f"{orders_with_repeated_products:,}"
)


# ============================================================
# D. DELIVERY-TIME DISTRIBUTION BY SOURCE STATUS
# ============================================================

delivery_time_summary = (
    deliveries
    .groupby("delivery_status")
    ["delivery_time_mins"]
    .agg(
        orders="count",
        minimum="min",
        median="median",
        mean="mean",
        maximum="max"
    )
    .reset_index()
)

print("\n" + "=" * 92)
print(" DELIVERY TIME BY SOURCE DELIVERY STATUS")
print("=" * 92)

display(delivery_time_summary)


# ============================================================
# E. TEST WHETHER SOURCE DELIVERY STATUS CAN BE EXPLAINED
#    BY DELIVERY TIME
# ============================================================

delivery_test = deliveries[
    [
        "delivery_time_mins",
        "delivery_status"
    ]
].copy()

delivery_test["actual_on_time"] = (
    delivery_test["delivery_status"] ==
    "Delivered On Time"
)

candidate_thresholds = sorted(
    delivery_test[
        "delivery_time_mins"
    ].unique()
)

threshold_results = []

for threshold in candidate_thresholds:

    predicted_on_time = (
        delivery_test["delivery_time_mins"]
        <= threshold
    )

    accuracy = (
        predicted_on_time ==
        delivery_test["actual_on_time"]
    ).mean()

    threshold_results.append(
        (threshold, accuracy)
    )

best_threshold, best_accuracy = max(
    threshold_results,
    key=lambda x: x[1]
)

print("\n" + "=" * 92)
print(" SOURCE SLA LABEL VALIDATION")
print("=" * 92)

print(
    f"Best delivery-time threshold found : "
    f"{best_threshold} minutes"
)

print(
    f"Best classification accuracy       : "
    f"{best_accuracy:.2%}"
)

if best_accuracy >= 0.90:
    print(
        "\nInterpretation: Source delivery status appears "
        "strongly related to delivery time."
    )
else:
    print(
        "\nInterpretation: Source delivery status is NOT "
        "reliably explained by delivery time."
    )


# ============================================================
# F. RATINGS BY ORDER STATUS
# ============================================================

rating_status = ratings.merge(
    orders[
        ["order_id", "order_status"]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

ratings_by_status = (
    rating_status
    .groupby("order_status")
    .agg(
        rating_records=("rating_id", "count"),
        rated_orders=("order_id", "nunique"),
        avg_rating=("rating", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 92)
print(" RATINGS BY ORDER STATUS")
print("=" * 92)

display(ratings_by_status)


# ============================================================
# G. TRANSACTION COVERAGE RATE BY ORDER STATUS
# ============================================================

coverage = orders.copy()

coverage["has_transaction"] = (
    coverage["order_id"]
    .isin(transaction_order_ids)
)

transaction_coverage_summary = (
    coverage
    .groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        orders_with_transactions=(
            "has_transaction",
            "sum"
        )
    )
    .reset_index()
)

transaction_coverage_summary[
    "orders_without_transactions"
] = (
    transaction_coverage_summary[
        "total_orders"
    ]
    -
    transaction_coverage_summary[
        "orders_with_transactions"
    ]
)

transaction_coverage_summary[
    "transaction_coverage_pct"
] = (
    transaction_coverage_summary[
        "orders_with_transactions"
    ]
    /
    transaction_coverage_summary[
        "total_orders"
    ]
    * 100
)

print("\n" + "=" * 92)
print(" TRANSACTION COVERAGE RATE BY ORDER STATUS")
print("=" * 92)

display(transaction_coverage_summary)

 TEMPORAL CONSISTENCY
Orders placed before customer creation date : 9,965
Total orders                                : 20,000

 PAYMENT MODE CONSISTENCY
Orders represented in transactions : 18,372
Orders with multiple payment modes : 11,938

Number of payment modes per order:


,payment_modes_per_order,orders
0,1,6434
1,2,7347
2,3,3621
3,4,855
4,5,110
5,6,5



 PRODUCT-LINE CONSISTENCY
Order-product combinations repeated : 50
Orders containing repeated product lines : 50

 DELIVERY TIME BY SOURCE DELIVERY STATUS


,delivery_status,orders,minimum,median,mean,maximum
0,Delivered Late,9936,8,47.00,48.51,107
1,Delivered On Time,10064,5,33.00,34.21,90



 SOURCE SLA LABEL VALIDATION
Best delivery-time threshold found : 35 minutes
Best classification accuracy       : 61.36%

Interpretation: Source delivery status is NOT reliably explained by delivery time.

 RATINGS BY ORDER STATUS


,order_status,rating_records,rated_orders,avg_rating
0,Cancelled,1979,1257,3.85
1,Delivered,15714,9826,3.88
2,Out for Delivery,1380,857,3.86
3,Returned,927,615,3.89



 TRANSACTION COVERAGE RATE BY ORDER STATUS


,order_status,total_orders,orders_with_transactions,orders_without_transactions,transaction_coverage_pct
0,Cancelled,2027,1874,153,92.45
1,Delivered,15618,14333,1285,91.77
2,Out for Delivery,1366,1239,127,90.70
3,Returned,989,926,63,93.63


## 09. Data Quality Treatment Policy

The source dataset demonstrates strong relational integrity but contains several semantic artefacts typical of synthetic data. Rather than deleting or silently correcting source records, the analytical layer applies transparent eligibility rules.

### Treatment Rules

1. **Customer creation date**
   - `created_date` is retained for provenance.
   - It is not used for customer tenure, acquisition cohort, or lifecycle analysis because a material share of orders predates the recorded creation date.

2. **Transaction coverage**
   - Orders without transaction lines remain available for operational order counts.
   - They are excluded from revenue, basket, product, inventory-demand, and profitability calculations.

3. **Delivery status**
   - The source `delivery_status` is retained for lineage only.
   - Operational SLA is derived independently from `delivery_time_mins`.

4. **Ratings**
   - Multiple ratings are aggregated to one order-level average.
   - Customer-experience KPIs use ratings for completed delivered orders only.

5. **Payment mode**
   - Payment mode is retained at transaction-line level.
   - It is not treated as a reliable one-payment-per-order attribute because many orders contain multiple source payment modes.

6. **Duplicate email addresses**
   - No customer records are merged based on email.
   - `customer_id` remains the authoritative customer identifier.

7. **Source preservation**
   - Raw tables are never overwritten.
   - All analytical transformations are performed on derived canonical tables.

In [16]:
# ============================================================
# 09. ORDER-LEVEL QUALITY & ANALYTICAL ELIGIBILITY FLAGS
# ============================================================

# ------------------------------------------------------------
# Transaction aggregates
# ------------------------------------------------------------

order_transaction_summary = (
    transactions
    .groupby("order_id")
    .agg(
        transaction_lines=("transaction_id", "count"),
        distinct_products=("product_id", "nunique"),
        total_units=("quantity", "sum"),
        gross_merchandise_value=("amount", "sum"),
        payment_mode_count=("payment_mode", "nunique")
    )
    .reset_index()
)


# ------------------------------------------------------------
# Rating aggregates
# ------------------------------------------------------------

order_rating_summary = (
    ratings
    .groupby("order_id")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating_id", "count")
    )
    .reset_index()
)


# ------------------------------------------------------------
# Begin canonical order-level table
# ------------------------------------------------------------

fact_order = (
    orders
    .merge(
        deliveries,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        order_transaction_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        order_rating_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Replace transaction absence with explicit zeros
# ------------------------------------------------------------

transaction_zero_cols = [
    "transaction_lines",
    "distinct_products",
    "total_units",
    "gross_merchandise_value",
    "payment_mode_count"
]

fact_order[transaction_zero_cols] = (
    fact_order[transaction_zero_cols]
    .fillna(0)
)


# ------------------------------------------------------------
# Quality flags
# ------------------------------------------------------------

fact_order["has_transaction"] = (
    fact_order["transaction_lines"] > 0
)

fact_order["has_rating"] = (
    fact_order["rating_count"].fillna(0) > 0
)

fact_order["is_delivered"] = (
    fact_order["order_status"] == "Delivered"
)

fact_order["is_cancelled"] = (
    fact_order["order_status"] == "Cancelled"
)

fact_order["is_returned"] = (
    fact_order["order_status"] == "Returned"
)

fact_order["is_open_order"] = (
    fact_order["order_status"] == "Out for Delivery"
)


# ------------------------------------------------------------
# Analytical eligibility
# ------------------------------------------------------------

fact_order["commercial_eligible"] = (
    fact_order["has_transaction"]
)

fact_order["completed_sales_eligible"] = (
    fact_order["has_transaction"]
    &
    fact_order["is_delivered"]
)

fact_order["cx_eligible"] = (
    fact_order["is_delivered"]
    &
    fact_order["has_rating"]
)

fact_order["sla_eligible"] = (
    fact_order["is_delivered"]
    &
    fact_order["delivery_time_mins"].notna()
)


# ------------------------------------------------------------
# Data quality classification
# ------------------------------------------------------------

def classify_order_quality(row):

    if not row["has_transaction"]:
        return "Missing Transaction Detail"

    if row["payment_mode_count"] > 1:
        return "Multi-Payment Source Artifact"

    return "Analytically Complete"


fact_order["data_quality_flag"] = (
    fact_order.apply(
        classify_order_quality,
        axis=1
    )
)


print("=" * 92)
print(" CANONICAL ORDER QUALITY LAYER")
print("=" * 92)

print(f"Total orders                  : {len(fact_order):,}")
print(f"Commercial eligible           : {fact_order['commercial_eligible'].sum():,}")
print(f"Completed-sales eligible      : {fact_order['completed_sales_eligible'].sum():,}")
print(f"Customer-experience eligible  : {fact_order['cx_eligible'].sum():,}")
print(f"SLA eligible                  : {fact_order['sla_eligible'].sum():,}")

print("\nData-quality classification:")

display(
    fact_order[
        "data_quality_flag"
    ]
    .value_counts()
    .rename_axis("quality_flag")
    .reset_index(name="orders")
)

print("\nCanonical order sample:")
display(fact_order.head())

 CANONICAL ORDER QUALITY LAYER
Total orders                  : 20,000
Commercial eligible           : 18,372
Completed-sales eligible      : 14,333
Customer-experience eligible  : 9,826
SLA eligible                  : 15,618

Data-quality classification:


,quality_flag,orders
0,Multi-Payment Source Artifact,11938
1,Analytically Complete,6434
2,Missing Transaction Detail,1628



Canonical order sample:


,order_id,customer_id,order_date,order_status,delivery_id,delivery_time_mins,delivery_status,distance_km,transaction_lines,distinct_products,total_units,gross_merchandise_value,payment_mode_count,avg_rating,rating_count,has_transaction,has_rating,is_delivered,is_cancelled,is_returned,is_open_order,commercial_eligible,completed_sales_eligible,cx_eligible,sla_eligible,data_quality_flag
0,O000001,C09732,2024-03-24,Delivered,D000001,52,Delivered Late,5.00,5.00,5.00,15.00,"2,560.00",3.00,5.00,1.00,True,True,True,False,False,False,True,True,True,True,Multi-Payment Source Artifact
1,O000002,C06239,2023-06-27,Delivered,D000002,66,Delivered Late,6.50,1.00,1.00,1.00,180.00,1.00,4.25,4.00,True,True,True,False,False,False,True,True,True,True,Analytically Complete
2,O000003,C08023,2023-04-07,Delivered,D000003,31,Delivered On Time,6.50,3.00,3.00,8.00,660.00,2.00,4.33,3.00,True,True,True,False,False,False,True,True,True,True,Multi-Payment Source Artifact
3,O000004,C01051,2024-03-19,Cancelled,D000004,68,Delivered On Time,7.70,0.00,0.00,0.00,0.00,0.00,NaN,NaN,False,False,False,True,False,False,False,False,False,False,Missing Transaction Detail
4,O000005,C02847,2024-02-21,Delivered,D000005,44,Delivered Late,7.10,3.00,3.00,9.00,640.00,2.00,NaN,NaN,True,False,True,False,False,False,True,True,False,True,Multi-Payment Source Artifact


In [17]:
# ============================================================
# 10. DERIVED OPERATIONAL SLA
# ============================================================

SLA_TARGET_MINUTES = 35
SLA_RISK_LIMIT_MINUTES = 50

fact_order["derived_sla_status"] = np.select(
    [
        fact_order["delivery_time_mins"] <= SLA_TARGET_MINUTES,
        fact_order["delivery_time_mins"] <= SLA_RISK_LIMIT_MINUTES
    ],
    [
        "On Time",
        "At Risk"
    ],
    default="Late"
)

# Only completed delivered orders should contribute
# to final SLA KPIs.
fact_order["derived_sla_status"] = np.where(
    fact_order["sla_eligible"],
    fact_order["derived_sla_status"],
    "Not Eligible"
)

fact_order["sla_minutes_over_target"] = np.where(
    fact_order["sla_eligible"],
    np.maximum(
        fact_order["delivery_time_mins"] - SLA_TARGET_MINUTES,
        0
    ),
    np.nan
)

print("=" * 92)
print(" DERIVED SLA FRAMEWORK")
print("=" * 92)

print(f"Target SLA          : <= {SLA_TARGET_MINUTES} minutes")
print(
    f"At-Risk Band        : "
    f"{SLA_TARGET_MINUTES + 1}–{SLA_RISK_LIMIT_MINUTES} minutes"
)
print(
    f"Late                : "
    f"> {SLA_RISK_LIMIT_MINUTES} minutes"
)

sla_distribution = (
    fact_order[
        fact_order["sla_eligible"]
    ]
    ["derived_sla_status"]
    .value_counts()
    .rename_axis("sla_status")
    .reset_index(name="orders")
)

sla_distribution["share_pct"] = (
    sla_distribution["orders"]
    /
    sla_distribution["orders"].sum()
    * 100
)

print("\nCompleted-order SLA distribution:")
display(sla_distribution)

print(
    "\nMedian completed delivery time:",
    round(
        fact_order.loc[
            fact_order["sla_eligible"],
            "delivery_time_mins"
        ].median(),
        2
    ),
    "minutes"
)

print(
    "P90 completed delivery time:",
    round(
        fact_order.loc[
            fact_order["sla_eligible"],
            "delivery_time_mins"
        ].quantile(0.90),
        2
    ),
    "minutes"
)

 DERIVED SLA FRAMEWORK
Target SLA          : <= 35 minutes
At-Risk Band        : 36–50 minutes
Late                : > 50 minutes

Completed-order SLA distribution:


,sla_status,orders,share_pct
0,On Time,6637,42.50
1,Late,5349,34.25
2,At Risk,3632,23.26



Median completed delivery time: 40.0 minutes
P90 completed delivery time: 71.0 minutes


In [18]:
# ============================================================
# 11. CANONICAL ANALYTICAL DATA MODEL
# ============================================================

# ------------------------------------------------------------
# Dimensions
# ------------------------------------------------------------

dim_customer = customers[
    [
        "customer_id",
        "customer_name",
        "gender",
        "age",
        "city",
        "state"
    ]
].copy()

dim_product = products.copy()


# ------------------------------------------------------------
# Date dimension
# ------------------------------------------------------------

date_min = orders["order_date"].min()
date_max = orders["order_date"].max()

dim_date = pd.DataFrame({
    "date": pd.date_range(
        start=date_min,
        end=date_max,
        freq="D"
    )
})

dim_date["date_key"] = (
    dim_date["date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["week_number"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day_of_month"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = (
    dim_date["date"].dt.dayofweek >= 5
)


# ------------------------------------------------------------
# Canonical order-item fact
# ------------------------------------------------------------

fact_order_item = (
    transactions
    .merge(
        orders[
            [
                "order_id",
                "customer_id",
                "order_date",
                "order_status"
            ]
        ],
        on="order_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        products[
            [
                "product_id",
                "product_name",
                "category",
                "sub_category",
                "price"
            ]
        ],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
)


fact_order_item["line_revenue"] = (
    fact_order_item["amount"]
)

fact_order_item["commercial_eligible"] = True

fact_order_item["completed_sales_eligible"] = (
    fact_order_item["order_status"] ==
    "Delivered"
)


# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------

print("=" * 92)
print(" CANONICAL ANALYTICAL MODEL")
print("=" * 92)

model_summary = pd.DataFrame([
    {
        "table": "dim_customer",
        "rows": len(dim_customer),
        "grain": "1 row per customer"
    },
    {
        "table": "dim_product",
        "rows": len(dim_product),
        "grain": "1 row per product/SKU"
    },
    {
        "table": "dim_date",
        "rows": len(dim_date),
        "grain": "1 row per calendar date"
    },
    {
        "table": "fact_order",
        "rows": len(fact_order),
        "grain": "1 row per order"
    },
    {
        "table": "fact_order_item",
        "rows": len(fact_order_item),
        "grain": "1 row per transaction line"
    }
])

display(model_summary)

 CANONICAL ANALYTICAL MODEL


,table,rows,grain
0,dim_customer,10000,1 row per customer
1,dim_product,1200,1 row per product/SKU
2,dim_date,731,1 row per calendar date
3,fact_order,20000,1 row per order
4,fact_order_item,50000,1 row per transaction line


## 12. Dark Store Network Engineering

The source dataset contains customer geography and delivery metrics but does not provide a native dark-store identifier.

To support dark-store-level operational analysis, a simulated fulfilment network is constructed using city-level completed-order demand. Store allocation is deterministic and reproducible, ensuring that the same customer is consistently mapped to the same modelled fulfilment node within a city.

These dark stores are analytical simulation entities and do not represent actual Zepto facilities.

In [19]:
# ============================================================
# 12. GEOGRAPHIC DEMAND PROFILE
# ============================================================

# Remove previously generated geography fields if this cell
# is rerun.
fact_order_geo = fact_order.drop(
    columns=[
        "city",
        "state",
        "dark_store_id",
        "dark_store_name"
    ],
    errors="ignore"
).copy()

# ------------------------------------------------------------
# Attach customer geography
# ------------------------------------------------------------

fact_order_geo = fact_order_geo.merge(
    dim_customer[
        [
            "customer_id",
            "city",
            "state"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# Helper analytical columns
# ------------------------------------------------------------

fact_order_geo["completed_gmv"] = np.where(
    fact_order_geo["completed_sales_eligible"],
    fact_order_geo["gross_merchandise_value"],
    0
)

fact_order_geo["completed_units"] = np.where(
    fact_order_geo["completed_sales_eligible"],
    fact_order_geo["total_units"],
    0
)

fact_order_geo["sla_on_time"] = (
    fact_order_geo["derived_sla_status"] == "On Time"
).astype(int)

fact_order_geo["delivery_time_for_sla"] = np.where(
    fact_order_geo["sla_eligible"],
    fact_order_geo["delivery_time_mins"],
    np.nan
)

fact_order_geo["distance_for_sla"] = np.where(
    fact_order_geo["sla_eligible"],
    fact_order_geo["distance_km"],
    np.nan
)

fact_order_geo["rating_for_cx"] = np.where(
    fact_order_geo["cx_eligible"],
    fact_order_geo["avg_rating"],
    np.nan
)


# ------------------------------------------------------------
# City-level demand profile
# ------------------------------------------------------------

city_profile = (
    fact_order_geo
    .groupby(
        ["city", "state"],
        dropna=False
    )
    .agg(
        total_orders=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        commercial_orders=("commercial_eligible", "sum"),
        completed_sales_orders=("completed_sales_eligible", "sum"),
        sla_eligible_orders=("sla_eligible", "sum"),
        on_time_orders=("sla_on_time", "sum"),
        completed_gmv=("completed_gmv", "sum"),
        completed_units=("completed_units", "sum"),
        avg_delivery_mins=("delivery_time_for_sla", "mean"),
        avg_delivery_distance_km=("distance_for_sla", "mean"),
        avg_customer_rating=("rating_for_cx", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# Derived geographic KPIs
# ------------------------------------------------------------

city_profile["completed_aov"] = np.where(
    city_profile["completed_sales_orders"] > 0,
    city_profile["completed_gmv"] /
    city_profile["completed_sales_orders"],
    np.nan
)

city_profile["sla_on_time_pct"] = np.where(
    city_profile["sla_eligible_orders"] > 0,
    city_profile["on_time_orders"] /
    city_profile["sla_eligible_orders"] * 100,
    np.nan
)

city_profile["gmv_share_pct"] = (
    city_profile["completed_gmv"] /
    city_profile["completed_gmv"].sum() * 100
)

city_profile = (
    city_profile
    .sort_values(
        "completed_sales_orders",
        ascending=False
    )
    .reset_index(drop=True)
)

city_profile["city_rank"] = (
    np.arange(1, len(city_profile) + 1)
)


print("=" * 96)
print(" CITY-LEVEL QUICK-COMMERCE DEMAND PROFILE")
print("=" * 96)

print(f"Cities represented       : {city_profile['city'].nunique():,}")
print(f"States represented       : {city_profile['state'].nunique():,}")
print(f"Total completed orders   : {city_profile['completed_sales_orders'].sum():,.0f}")
print(f"Completed GMV            : ₹{city_profile['completed_gmv'].sum():,.2f}")

print("\nCity performance:")

display(
    city_profile[
        [
            "city_rank",
            "city",
            "state",
            "total_orders",
            "completed_sales_orders",
            "completed_gmv",
            "completed_aov",
            "avg_delivery_mins",
            "sla_on_time_pct",
            "avg_delivery_distance_km",
            "avg_customer_rating"
        ]
    ]
)

 CITY-LEVEL QUICK-COMMERCE DEMAND PROFILE
Cities represented       : 10
States represented       : 9
Total completed orders   : 14,333
Completed GMV            : ₹28,224,720.00

City performance:


,city_rank,city,state,total_orders,completed_sales_orders,completed_gmv,completed_aov,avg_delivery_mins,sla_on_time_pct,avg_delivery_distance_km,avg_customer_rating
0,1,Kolkata,West Bengal,2055,1497,"2,836,860.00","1,895.03",42.74,39.69,5.40,3.86
1,2,Delhi,Delhi,2057,1497,"3,009,330.00","2,010.24",41.10,42.89,5.20,3.89
2,3,Ahmedabad,Gujarat,2086,1489,"2,889,040.00","1,940.26",42.10,38.93,5.34,3.87
3,4,Chennai,Tamil Nadu,2060,1459,"2,818,560.00","1,931.84",41.30,42.94,5.25,3.90
4,5,Pune,Maharashtra,2026,1438,"2,870,290.00","1,996.03",41.41,42.61,5.29,3.92
5,6,Hyderabad,Telangana,1989,1435,"2,760,630.00","1,923.78",41.45,43.55,5.18,3.87
6,7,Jaipur,Rajasthan,1982,1426,"2,817,930.00","1,976.11",40.02,44.96,5.07,3.86
7,8,Bengaluru,Karnataka,1943,1388,"2,745,830.00","1,978.26",40.87,43.61,5.20,3.92
8,9,Lucknow,Uttar Pradesh,1907,1371,"2,799,830.00","2,042.18",41.44,41.64,5.27,3.90
9,10,Mumbai,Maharashtra,1895,1333,"2,676,420.00","2,007.82",40.36,44.51,5.13,3.81


In [20]:
# ============================================================
# 13. SIMULATED DARK STORE NETWORK
# ============================================================

# ------------------------------------------------------------
# Portfolio modelling assumption
# ------------------------------------------------------------
#
# This is NOT intended to represent Zepto's real store capacity.
# It is a reproducible network-sizing heuristic that creates
# enough operational nodes to analyse city/store performance.
#
# Approximately one analytical fulfilment node is created for
# every 1,200 completed orders observed during the dataset
# horizon, with at least one node per represented city.
# ------------------------------------------------------------

TARGET_COMPLETED_ORDERS_PER_STORE = 1200
MAX_STORES_PER_CITY = 6


city_network_plan = city_profile.copy()

city_network_plan["modelled_store_count"] = np.ceil(
    city_network_plan["completed_sales_orders"] /
    TARGET_COMPLETED_ORDERS_PER_STORE
).astype(int)

city_network_plan["modelled_store_count"] = (
    city_network_plan["modelled_store_count"]
    .clip(lower=1, upper=MAX_STORES_PER_CITY)
)


# ------------------------------------------------------------
# Demand-network tier
# ------------------------------------------------------------

city_network_plan["network_tier"] = pd.cut(
    city_network_plan["gmv_share_pct"],
    bins=[
        -np.inf,
        5,
        10,
        np.inf
    ],
    labels=[
        "Tier C",
        "Tier B",
        "Tier A"
    ]
)


# ------------------------------------------------------------
# Build dark-store dimension
# ------------------------------------------------------------

dark_store_records = []

for _, city_row in city_network_plan.iterrows():

    city_rank = int(city_row["city_rank"])
    store_count = int(city_row["modelled_store_count"])

    for store_number in range(1, store_count + 1):

        dark_store_records.append({
            "dark_store_id":
                f"DS_{city_rank:02d}_{store_number:02d}",

            "dark_store_name":
                f"{city_row['city']} Dark Store {store_number:02d}",

            "city":
                city_row["city"],

            "state":
                city_row["state"],

            "city_rank":
                city_rank,

            "network_tier":
                str(city_row["network_tier"]),

            "store_number":
                store_number
        })


dim_dark_store = pd.DataFrame(
    dark_store_records
)


print("=" * 96)
print(" SIMULATED DARK STORE NETWORK")
print("=" * 96)

print(
    f"Modelled dark stores                 : "
    f"{len(dim_dark_store):,}"
)

print(
    f"Target completed orders / store      : "
    f"{TARGET_COMPLETED_ORDERS_PER_STORE:,}"
)

print(
    f"Maximum modelled stores / city       : "
    f"{MAX_STORES_PER_CITY}"
)

print("\nCity network plan:")

display(
    city_network_plan[
        [
            "city_rank",
            "city",
            "state",
            "completed_sales_orders",
            "completed_gmv",
            "gmv_share_pct",
            "modelled_store_count",
            "network_tier"
        ]
    ]
)

print("\nDark-store dimension:")

display(
    dim_dark_store.head(20)
)

 SIMULATED DARK STORE NETWORK
Modelled dark stores                 : 20
Target completed orders / store      : 1,200
Maximum modelled stores / city       : 6

City network plan:


,city_rank,city,state,completed_sales_orders,completed_gmv,gmv_share_pct,modelled_store_count,network_tier
0,1,Kolkata,West Bengal,1497,"2,836,860.00",10.05,2,Tier A
1,2,Delhi,Delhi,1497,"3,009,330.00",10.66,2,Tier A
2,3,Ahmedabad,Gujarat,1489,"2,889,040.00",10.24,2,Tier A
3,4,Chennai,Tamil Nadu,1459,"2,818,560.00",9.99,2,Tier B
4,5,Pune,Maharashtra,1438,"2,870,290.00",10.17,2,Tier A
5,6,Hyderabad,Telangana,1435,"2,760,630.00",9.78,2,Tier B
6,7,Jaipur,Rajasthan,1426,"2,817,930.00",9.98,2,Tier B
7,8,Bengaluru,Karnataka,1388,"2,745,830.00",9.73,2,Tier B
8,9,Lucknow,Uttar Pradesh,1371,"2,799,830.00",9.92,2,Tier B
9,10,Mumbai,Maharashtra,1333,"2,676,420.00",9.48,2,Tier B



Dark-store dimension:


,dark_store_id,dark_store_name,city,state,city_rank,network_tier,store_number
0,DS_01_01,Kolkata Dark Store 01,Kolkata,West Bengal,1,Tier A,1
1,DS_01_02,Kolkata Dark Store 02,Kolkata,West Bengal,1,Tier A,2
2,DS_02_01,Delhi Dark Store 01,Delhi,Delhi,2,Tier A,1
3,DS_02_02,Delhi Dark Store 02,Delhi,Delhi,2,Tier A,2
4,DS_03_01,Ahmedabad Dark Store 01,Ahmedabad,Gujarat,3,Tier A,1
5,DS_03_02,Ahmedabad Dark Store 02,Ahmedabad,Gujarat,3,Tier A,2
6,DS_04_01,Chennai Dark Store 01,Chennai,Tamil Nadu,4,Tier B,1
7,DS_04_02,Chennai Dark Store 02,Chennai,Tamil Nadu,4,Tier B,2
8,DS_05_01,Pune Dark Store 01,Pune,Maharashtra,5,Tier A,1
9,DS_05_02,Pune Dark Store 02,Pune,Maharashtra,5,Tier A,2


In [21]:
# ============================================================
# 14. DETERMINISTIC ORDER-TO-DARK-STORE ASSIGNMENT
# ============================================================

import hashlib


# ------------------------------------------------------------
# Create city → store mapping
# ------------------------------------------------------------

city_store_map = (
    dim_dark_store
    .groupby("city")["dark_store_id"]
    .apply(list)
    .to_dict()
)


def assign_dark_store(customer_id, city):
    """
    Deterministically assign a customer to one modelled
    dark store inside their city.

    MD5 is used only as a stable allocation mechanism,
    not for security.
    """

    available_stores = city_store_map.get(city)

    if not available_stores:
        return None

    stable_hash = int(
        hashlib.md5(
            str(customer_id).encode("utf-8")
        ).hexdigest(),
        16
    )

    store_index = (
        stable_hash %
        len(available_stores)
    )

    return available_stores[store_index]


fact_order_geo["dark_store_id"] = (
    fact_order_geo.apply(
        lambda row: assign_dark_store(
            row["customer_id"],
            row["city"]
        ),
        axis=1
    )
)


# ------------------------------------------------------------
# Add dark store names
# ------------------------------------------------------------

fact_order_geo = fact_order_geo.merge(
    dim_dark_store[
        [
            "dark_store_id",
            "dark_store_name"
        ]
    ],
    on="dark_store_id",
    how="left",
    validate="many_to_one"
)


# Make this our new canonical fact_order
fact_order = fact_order_geo.copy()


# ------------------------------------------------------------
# Attach dark store to order-item fact
# ------------------------------------------------------------

fact_order_item = fact_order_item.drop(
    columns=[
        "dark_store_id",
        "dark_store_name",
        "city",
        "state"
    ],
    errors="ignore"
)

fact_order_item = fact_order_item.merge(
    fact_order[
        [
            "order_id",
            "dark_store_id",
            "dark_store_name",
            "city",
            "state"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

unassigned_orders = int(
    fact_order["dark_store_id"]
    .isna()
    .sum()
)

stores_receiving_orders = (
    fact_order["dark_store_id"]
    .nunique()
)

print("=" * 96)
print(" DARK STORE ASSIGNMENT VALIDATION")
print("=" * 96)

print(f"Total orders                 : {len(fact_order):,}")
print(f"Unassigned orders            : {unassigned_orders:,}")
print(f"Modelled dark stores         : {len(dim_dark_store):,}")
print(f"Stores receiving orders      : {stores_receiving_orders:,}")

if unassigned_orders == 0:
    print("\nAssignment validation: PASS ✅")
else:
    print("\nAssignment validation: REVIEW ⚠️")

 DARK STORE ASSIGNMENT VALIDATION
Total orders                 : 20,000
Unassigned orders            : 0
Modelled dark stores         : 20
Stores receiving orders      : 20

Assignment validation: PASS ✅


In [22]:
# ============================================================
# 15. DARK STORE BASELINE PERFORMANCE
# ============================================================

dark_store_performance = (
    fact_order
    .groupby(
        [
            "dark_store_id",
            "dark_store_name",
            "city",
            "state"
        ]
    )
    .agg(
        total_orders=("order_id", "count"),

        unique_customers=(
            "customer_id",
            "nunique"
        ),

        commercial_orders=(
            "commercial_eligible",
            "sum"
        ),

        completed_orders=(
            "completed_sales_eligible",
            "sum"
        ),

        sla_eligible_orders=(
            "sla_eligible",
            "sum"
        ),

        on_time_orders=(
            "sla_on_time",
            "sum"
        ),

        completed_gmv=(
            "completed_gmv",
            "sum"
        ),

        completed_units=(
            "completed_units",
            "sum"
        ),

        avg_delivery_mins=(
            "delivery_time_for_sla",
            "mean"
        ),

        avg_delivery_distance_km=(
            "distance_for_sla",
            "mean"
        ),

        avg_customer_rating=(
            "rating_for_cx",
            "mean"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Store-level KPIs
# ------------------------------------------------------------

dark_store_performance["aov"] = np.where(
    dark_store_performance[
        "completed_orders"
    ] > 0,

    dark_store_performance[
        "completed_gmv"
    ] /
    dark_store_performance[
        "completed_orders"
    ],

    np.nan
)


dark_store_performance["units_per_order"] = np.where(
    dark_store_performance[
        "completed_orders"
    ] > 0,

    dark_store_performance[
        "completed_units"
    ] /
    dark_store_performance[
        "completed_orders"
    ],

    np.nan
)


dark_store_performance["sla_on_time_pct"] = np.where(
    dark_store_performance[
        "sla_eligible_orders"
    ] > 0,

    dark_store_performance[
        "on_time_orders"
    ] /
    dark_store_performance[
        "sla_eligible_orders"
    ] * 100,

    np.nan
)


dark_store_performance["commercial_coverage_pct"] = (
    dark_store_performance[
        "commercial_orders"
    ] /
    dark_store_performance[
        "total_orders"
    ] * 100
)


dark_store_performance = (
    dark_store_performance
    .sort_values(
        "completed_gmv",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 110)
print(" DARK STORE BASELINE PERFORMANCE")
print("=" * 110)

display(
    dark_store_performance[
        [
            "dark_store_id",
            "dark_store_name",
            "city",
            "total_orders",
            "completed_orders",
            "completed_gmv",
            "aov",
            "units_per_order",
            "avg_delivery_mins",
            "sla_on_time_pct",
            "avg_delivery_distance_km",
            "avg_customer_rating"
        ]
    ]
)

print("\nNetwork totals:")

print(
    f"Dark stores              : "
    f"{len(dark_store_performance):,}"
)

print(
    f"Completed orders         : "
    f"{dark_store_performance['completed_orders'].sum():,.0f}"
)

print(
    f"Completed GMV            : "
    f"₹{dark_store_performance['completed_gmv'].sum():,.2f}"
)

print(
    f"Network SLA On-Time      : "
    f"{dark_store_performance['on_time_orders'].sum() /
       dark_store_performance['sla_eligible_orders'].sum() * 100:.2f}%"
)

 DARK STORE BASELINE PERFORMANCE


,dark_store_id,dark_store_name,city,total_orders,completed_orders,completed_gmv,aov,units_per_order,avg_delivery_mins,sla_on_time_pct,avg_delivery_distance_km,avg_customer_rating
0,DS_02_02,Delhi Dark Store 02,Delhi,1039,760,"1,534,750.00","2,019.41",8.24,41.81,42.44,5.23,3.94
1,DS_05_01,Pune Dark Store 01,Pune,1036,738,"1,504,950.00","2,039.23",8.33,40.56,43.30,5.20,3.86
2,DS_07_02,Jaipur Dark Store 02,Jaipur,1053,755,"1,502,010.00","1,989.42",8.32,40.71,43.08,5.18,3.83
3,DS_06_01,Hyderabad Dark Store 01,Hyderabad,1054,762,"1,500,460.00","1,969.11",8.15,42.20,41.34,5.23,3.82
4,DS_02_01,Delhi Dark Store 01,Delhi,1018,737,"1,474,580.00","2,000.79",8.29,40.38,43.36,5.18,3.85
5,DS_09_02,Lucknow Dark Store 02,Lucknow,966,696,"1,458,320.00","2,095.29",8.84,41.72,41.05,5.36,3.87
6,DS_03_01,Ahmedabad Dark Store 01,Ahmedabad,1063,756,"1,450,020.00","1,918.02",8.13,42.10,39.13,5.36,3.92
7,DS_01_02,Kolkata Dark Store 02,Kolkata,1038,760,"1,446,140.00","1,902.82",7.74,41.93,40.63,5.25,3.82
8,DS_03_02,Ahmedabad Dark Store 02,Ahmedabad,1023,733,"1,439,020.00","1,963.19",8.05,42.09,38.72,5.32,3.82
9,DS_08_01,Bengaluru Dark Store 01,Bengaluru,999,725,"1,437,450.00","1,982.69",8.16,40.84,43.85,5.23,3.96



Network totals:
Dark stores              : 20
Completed orders         : 14,333
Completed GMV            : ₹28,224,720.00
Network SLA On-Time      : 42.50%


## 16. Inventory Demand & Replenishment Intelligence

The source data contains order-item demand but does not provide historical inventory snapshots, supplier lead times, safety stock, or replenishment records.

Therefore, the inventory layer is divided into two components:

### Observed Metrics
Derived directly from completed order-item transactions:
- SKU demand
- Units sold
- Revenue
- Order frequency
- Store-SKU demand velocity
- Category contribution

### Modelled Metrics
Derived using documented portfolio assumptions:
- Lead time
- Safety stock
- Reorder point
- Modelled inventory on hand
- Days of supply
- Stockout risk
- Replenishment recommendation

All modelled fields are explicitly labelled and should not be interpreted as proprietary Zepto inventory data.

In [23]:
# ============================================================
# 16. COMPLETED ORDER-ITEM DEMAND BASE
# ============================================================

# Keep only transaction lines belonging to completed orders
completed_order_items = (
    fact_order_item[
        fact_order_item["completed_sales_eligible"]
    ]
    .copy()
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 96)
print(" COMPLETED ORDER-ITEM DEMAND BASE")
print("=" * 96)

print(
    f"Completed transaction lines : "
    f"{len(completed_order_items):,}"
)

print(
    f"Completed orders represented: "
    f"{completed_order_items['order_id'].nunique():,}"
)

print(
    f"Products represented        : "
    f"{completed_order_items['product_id'].nunique():,}"
)

print(
    f"Dark stores represented     : "
    f"{completed_order_items['dark_store_id'].nunique():,}"
)

print(
    f"Units sold                  : "
    f"{completed_order_items['quantity'].sum():,.0f}"
)

print(
    f"Revenue                     : "
    f"₹{completed_order_items['line_revenue'].sum():,.2f}"
)

print("\nObserved date range:")

print(
    completed_order_items["order_date"].min(),
    "to",
    completed_order_items["order_date"].max()
)

 COMPLETED ORDER-ITEM DEMAND BASE
Completed transaction lines : 38,986
Completed orders represented: 14,333
Products represented        : 1,200
Dark stores represented     : 20
Units sold                  : 117,468
Revenue                     : ₹28,224,720.00

Observed date range:
2023-01-01 00:00:00 to 2024-12-31 00:00:00


In [24]:
# ============================================================
# 17. PRODUCT CATALOGUE & CATEGORY PROFILE
# ============================================================

category_profile = (
    completed_order_items
    .groupby(
        ["category"],
        dropna=False
    )
    .agg(
        unique_products=("product_id", "nunique"),
        completed_orders=("order_id", "nunique"),
        units_sold=("quantity", "sum"),
        revenue=("line_revenue", "sum")
    )
    .reset_index()
)

category_profile["revenue_share_pct"] = (
    category_profile["revenue"]
    /
    category_profile["revenue"].sum()
    * 100
)

category_profile["unit_share_pct"] = (
    category_profile["units_sold"]
    /
    category_profile["units_sold"].sum()
    * 100
)

category_profile = (
    category_profile
    .sort_values(
        "revenue",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 96)
print(" PRODUCT CATEGORY PROFILE")
print("=" * 96)

print(
    f"Catalogue products : "
    f"{dim_product['product_id'].nunique():,}"
)

print(
    f"Active products    : "
    f"{completed_order_items['product_id'].nunique():,}"
)

print(
    f"Categories         : "
    f"{dim_product['category'].nunique():,}"
)

print(
    f"Sub-categories     : "
    f"{dim_product['sub_category'].nunique():,}"
)

display(category_profile)

 PRODUCT CATEGORY PROFILE
Catalogue products : 1,200
Active products    : 1,200
Categories         : 8
Sub-categories     : 29


,category,unique_products,completed_orders,units_sold,revenue,revenue_share_pct,unit_share_pct
0,Baby Care,163,4401,15569,7179820,25.44,13.25
1,Personal Care,145,4080,14318,4688810,16.61,12.19
2,Grocery & Staples,153,4222,14986,4596430,16.29,12.76
3,Household,139,3911,13668,2863030,10.14,11.64
4,Fruits & Vegetables,161,4547,15871,2740140,9.71,13.51
5,Dairy & Bakery,166,4610,16678,2657910,9.42,14.20
6,Beverages,138,3921,13431,1794010,6.36,11.43
7,Snacks & Branded Foods,135,3769,12947,1704570,6.04,11.02


In [25]:
# ============================================================
# 18. DAILY DARK STORE × SKU DEMAND
# ============================================================

daily_store_sku = (
    completed_order_items
    .groupby(
        [
            "order_date",
            "dark_store_id",
            "product_id",
            "product_name",
            "category",
            "sub_category"
        ],
        as_index=False
    )
    .agg(
        daily_units=("quantity", "sum"),
        daily_revenue=("line_revenue", "sum"),
        daily_orders=("order_id", "nunique")
    )
)


ANALYSIS_START_DATE = (
    completed_order_items["order_date"].min()
)

ANALYSIS_END_DATE = (
    completed_order_items["order_date"].max()
)

ANALYSIS_DAYS = (
    ANALYSIS_END_DATE -
    ANALYSIS_START_DATE
).days + 1


print("=" * 96)
print(" DAILY STORE × SKU DEMAND")
print("=" * 96)

print(
    f"Analysis horizon     : "
    f"{ANALYSIS_DAYS:,} days"
)

print(
    f"Daily demand records : "
    f"{len(daily_store_sku):,}"
)

print(
    f"Store-SKU pairs      : "
    f"{daily_store_sku[['dark_store_id', 'product_id']].drop_duplicates().shape[0]:,}"
)

print("\nSample:")
display(daily_store_sku.head(10))

 DAILY STORE × SKU DEMAND
Analysis horizon     : 731 days
Daily demand records : 38,892
Store-SKU pairs      : 19,280

Sample:


,order_date,dark_store_id,product_id,product_name,category,sub_category,daily_units,daily_revenue,daily_orders
0,2023-01-01,DS_01_02,P00509,Green Floor Cleaner 2kg,Baby Care,Baby Skin Care,5,3800,1
1,2023-01-01,DS_01_02,P00852,Good Coffee 1L,Snacks & Branded Foods,Biscuits & Cookies,2,80,1
2,2023-01-01,DS_01_02,P01143,Metro Toor Dal 200g,Fruits & Vegetables,Herbs,4,120,1
3,2023-01-01,DS_02_01,P00016,Urban Eggs 1L,Dairy & Bakery,Milk,5,700,1
4,2023-01-01,DS_02_01,P00878,Smart Eggs 1kg,Fruits & Vegetables,Fresh Vegetables,1,30,1
5,2023-01-01,DS_04_01,P00252,Good Banana 2kg,Beverages,Energy Drinks,5,1200,1
6,2023-01-01,DS_04_01,P00858,Happy Juice 12 pcs,Fruits & Vegetables,Herbs,4,840,1
7,2023-01-01,DS_05_01,P01121,Pure Namkeen 200g,Grocery & Staples,Atta & Flour,3,1710,1
8,2023-01-01,DS_05_02,P00853,Good Namkeen 2kg,Fruits & Vegetables,Herbs,3,210,1
9,2023-01-01,DS_05_02,P00879,Happy Tea 1L,Household,Disposables,5,1500,1


In [26]:
# ============================================================
# 19. STORE × SKU DEMAND INTELLIGENCE
# ============================================================

# ------------------------------------------------------------
# Aggregate observed demand
# ------------------------------------------------------------

store_sku_base = (
    daily_store_sku
    .groupby(
        [
            "dark_store_id",
            "product_id",
            "product_name",
            "category",
            "sub_category"
        ]
    )
    .agg(
        active_demand_days=("order_date", "nunique"),
        total_units=("daily_units", "sum"),
        total_revenue=("daily_revenue", "sum"),
        total_orders=("daily_orders", "sum"),
        sum_daily_units_squared=(
            "daily_units",
            lambda x: np.square(x).sum()
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Demand statistics across full horizon
#
# Missing dates implicitly represent zero demand.
# ------------------------------------------------------------

store_sku_base["avg_daily_demand"] = (
    store_sku_base["total_units"]
    / ANALYSIS_DAYS
)

store_sku_base["avg_daily_revenue"] = (
    store_sku_base["total_revenue"]
    / ANALYSIS_DAYS
)


# Population variance:
# E[X²] - E[X]²
store_sku_base["daily_demand_variance"] = (
    store_sku_base["sum_daily_units_squared"]
    / ANALYSIS_DAYS
) - (
    store_sku_base["avg_daily_demand"] ** 2
)

# Protect against tiny floating-point negatives
store_sku_base["daily_demand_variance"] = (
    store_sku_base[
        "daily_demand_variance"
    ].clip(lower=0)
)

store_sku_base["daily_demand_std"] = np.sqrt(
    store_sku_base[
        "daily_demand_variance"
    ]
)


# ------------------------------------------------------------
# Demand frequency
# ------------------------------------------------------------

store_sku_base["demand_frequency_pct"] = (
    store_sku_base["active_demand_days"]
    / ANALYSIS_DAYS
    * 100
)


# ------------------------------------------------------------
# Average units when demand occurs
# ------------------------------------------------------------

store_sku_base["units_per_active_day"] = (
    store_sku_base["total_units"]
    /
    store_sku_base["active_demand_days"]
)


print("=" * 105)
print(" STORE × SKU DEMAND INTELLIGENCE")
print("=" * 105)

print(
    f"Store-SKU combinations : "
    f"{len(store_sku_base):,}"
)

print(
    f"Average daily demand    : "
    f"{store_sku_base['avg_daily_demand'].mean():.4f}"
)

display(
    store_sku_base[
        [
            "dark_store_id",
            "product_id",
            "product_name",
            "category",
            "total_orders",
            "total_units",
            "total_revenue",
            "active_demand_days",
            "avg_daily_demand",
            "daily_demand_std",
            "demand_frequency_pct"
        ]
    ]
    .sort_values(
        "total_revenue",
        ascending=False
    )
    .head(20)
)

 STORE × SKU DEMAND INTELLIGENCE
Store-SKU combinations : 19,280
Average daily demand    : 0.0083


,dark_store_id,product_id,product_name,category,total_orders,total_units,total_revenue,active_demand_days,avg_daily_demand,daily_demand_std,demand_frequency_pct
14559,DS_08_02,P00047,Fresh Almonds 1L,Baby Care,5,20,15400,5,0.03,0.35,0.68
3320,DS_02_02,P00508,Quick Coffee 1kg,Baby Care,6,22,14740,6,0.03,0.36,0.82
7356,DS_04_02,P00744,Daily Floor Cleaner 12 pcs,Baby Care,5,22,14520,5,0.03,0.38,0.68
4597,DS_03_01,P00880,Happy Almonds 6 pcs,Baby Care,7,20,14400,7,0.03,0.31,0.96
6562,DS_04_01,P00953,Green Tomato 500ml,Baby Care,7,19,13870,7,0.03,0.28,0.96
10410,DS_06_01,P00882,Pure Milk 12 pcs,Baby Care,6,21,13860,6,0.03,0.33,0.82
16378,DS_09_01,P01178,Daily Cookies 12 pcs,Baby Care,5,20,13800,5,0.03,0.35,0.68
8129,DS_05_01,P00477,Urban Coffee 6 pcs,Baby Care,6,23,13570,6,0.03,0.37,0.82
10047,DS_06_01,P00450,Urban Cashew 1kg,Baby Care,4,18,13500,4,0.02,0.34,0.55
6404,DS_04_01,P00770,Metro Milk 500ml,Baby Care,8,20,13200,7,0.03,0.35,0.96


In [27]:
# ============================================================
# 20. ABC INVENTORY CLASSIFICATION
# ============================================================

inventory_intelligence = (
    store_sku_base.copy()
)


# ------------------------------------------------------------
# Rank SKUs independently within each dark store
# ------------------------------------------------------------

inventory_intelligence = (
    inventory_intelligence
    .sort_values(
        [
            "dark_store_id",
            "total_revenue"
        ],
        ascending=[
            True,
            False
        ]
    )
)


inventory_intelligence["store_total_revenue"] = (
    inventory_intelligence
    .groupby("dark_store_id")
    ["total_revenue"]
    .transform("sum")
)


inventory_intelligence["revenue_share"] = (
    inventory_intelligence["total_revenue"]
    /
    inventory_intelligence["store_total_revenue"]
)


inventory_intelligence[
    "cumulative_revenue_share"
] = (
    inventory_intelligence
    .groupby("dark_store_id")
    ["revenue_share"]
    .cumsum()
)


def abc_class(cumulative_share):

    if cumulative_share <= 0.80:
        return "A"

    elif cumulative_share <= 0.95:
        return "B"

    return "C"


inventory_intelligence["abc_class"] = (
    inventory_intelligence[
        "cumulative_revenue_share"
    ].apply(abc_class)
)


abc_summary = (
    inventory_intelligence
    .groupby("abc_class")
    .agg(
        store_sku_pairs=(
            "product_id",
            "count"
        ),
        revenue=(
            "total_revenue",
            "sum"
        ),
        units=(
            "total_units",
            "sum"
        )
    )
    .reset_index()
)


abc_summary["revenue_share_pct"] = (
    abc_summary["revenue"]
    /
    abc_summary["revenue"].sum()
    * 100
)


print("=" * 96)
print(" ABC INVENTORY CLASSIFICATION")
print("=" * 96)

display(abc_summary)

 ABC INVENTORY CLASSIFICATION


,abc_class,store_sku_pairs,revenue,units,revenue_share_pct
0,A,8594,22569580,71844,79.96
1,B,5439,4240700,28171,15.02
2,C,5247,1414440,17453,5.01


### Inventory Grain Refinement

Initial Store × SKU demand diagnostics revealed substantial sparsity: individual SKU-store combinations are active on very few days across the two-year observation horizon.

Accordingly, inventory analytics use a fit-for-purpose grain:

- **Network × SKU** for product-value (ABC) classification
- **Dark Store × Category × Week** for demand variability, replenishment and forecasting

This avoids generating unstable inventory recommendations from statistically sparse SKU-store time series.

In [28]:
# ============================================================
# 22. NETWORK-LEVEL SKU ABC CLASSIFICATION
# ============================================================

sku_network_profile = (
    completed_order_items
    .groupby(
        [
            "product_id",
            "product_name",
            "category",
            "sub_category"
        ],
        as_index=False
    )
    .agg(
        completed_orders=("order_id", "nunique"),
        units_sold=("quantity", "sum"),
        revenue=("line_revenue", "sum"),
        active_days=("order_date", "nunique"),
        dark_stores_selling=("dark_store_id", "nunique")
    )
)


# ------------------------------------------------------------
# Revenue ranking
# ------------------------------------------------------------

sku_network_profile = (
    sku_network_profile
    .sort_values(
        "revenue",
        ascending=False
    )
    .reset_index(drop=True)
)

sku_network_profile["revenue_share"] = (
    sku_network_profile["revenue"]
    /
    sku_network_profile["revenue"].sum()
)

sku_network_profile["cumulative_revenue_share"] = (
    sku_network_profile["revenue_share"]
    .cumsum()
)


# ------------------------------------------------------------
# ABC classification
# ------------------------------------------------------------

sku_network_profile["abc_class"] = np.select(
    [
        sku_network_profile[
            "cumulative_revenue_share"
        ] <= 0.80,

        sku_network_profile[
            "cumulative_revenue_share"
        ] <= 0.95
    ],
    [
        "A",
        "B"
    ],
    default="C"
)


sku_abc_summary = (
    sku_network_profile
    .groupby("abc_class")
    .agg(
        products=("product_id", "count"),
        revenue=("revenue", "sum"),
        units=("units_sold", "sum")
    )
    .reset_index()
)


sku_abc_summary["product_share_pct"] = (
    sku_abc_summary["products"]
    /
    sku_abc_summary["products"].sum()
    * 100
)

sku_abc_summary["revenue_share_pct"] = (
    sku_abc_summary["revenue"]
    /
    sku_abc_summary["revenue"].sum()
    * 100
)


print("=" * 100)
print(" NETWORK-LEVEL SKU ABC CLASSIFICATION")
print("=" * 100)

display(sku_abc_summary)

print("\nTop 20 products by observed revenue:")

display(
    sku_network_profile[
        [
            "product_id",
            "product_name",
            "category",
            "completed_orders",
            "units_sold",
            "revenue",
            "dark_stores_selling",
            "abc_class",
            "cumulative_revenue_share"
        ]
    ].head(20)
)

 NETWORK-LEVEL SKU ABC CLASSIFICATION


,abc_class,products,revenue,units,product_share_pct,revenue_share_pct
0,A,643,22572720,65587,53.58,79.98
1,B,315,4235410,29685,26.25,15.01
2,C,242,1416590,22196,20.17,5.02



Top 20 products by observed revenue:


,product_id,product_name,category,completed_orders,units_sold,revenue,dark_stores_selling,abc_class,cumulative_revenue_share
0,P00748,Daily Curd 6 pcs,Personal Care,48,156,92040,15,A,0.00
1,P00364,Smart Toothpaste 6 pcs,Baby Care,37,126,88200,16,A,0.01
2,P01178,Daily Cookies 12 pcs,Baby Care,41,126,86940,18,A,0.01
3,P00137,Daily Milk 12 pcs,Baby Care,34,117,86580,15,A,0.01
4,P01038,Urban Shampoo 500ml,Baby Care,33,108,86400,17,A,0.02
5,P00035,Urban Almonds 2kg,Baby Care,35,112,86240,17,A,0.02
6,P00047,Fresh Almonds 1L,Baby Care,38,108,83160,17,A,0.02
7,P00577,Urban Chocolate 2kg,Baby Care,40,129,82560,17,A,0.02
8,P00953,Green Tomato 500ml,Baby Care,35,111,81030,16,A,0.03
9,P00847,Quick Eggs 500g,Baby Care,34,108,79920,17,A,0.03


In [29]:
# ============================================================
# 23. WEEKLY DARK STORE × CATEGORY DEMAND
# ============================================================

weekly_source = completed_order_items.copy()

# ------------------------------------------------------------
# Create fixed 7-day periods from the analysis start date
# ------------------------------------------------------------

weekly_source["days_from_start"] = (
    weekly_source["order_date"] -
    ANALYSIS_START_DATE
).dt.days

weekly_source["week_index"] = (
    weekly_source["days_from_start"] // 7
) + 1

weekly_source["week_start"] = (
    ANALYSIS_START_DATE
    +
    pd.to_timedelta(
        (weekly_source["week_index"] - 1) * 7,
        unit="D"
    )
)


# ------------------------------------------------------------
# Observed weekly demand
# ------------------------------------------------------------

weekly_store_category_observed = (
    weekly_source
    .groupby(
        [
            "week_index",
            "week_start",
            "dark_store_id",
            "category"
        ],
        as_index=False
    )
    .agg(
        weekly_units=("quantity", "sum"),
        weekly_revenue=("line_revenue", "sum"),
        weekly_orders=("order_id", "nunique"),
        active_skus=("product_id", "nunique")
    )
)


# ------------------------------------------------------------
# Construct complete panel including zero-demand weeks
# ------------------------------------------------------------

TOTAL_WEEKS = int(
    np.ceil(
        ANALYSIS_DAYS / 7
    )
)

week_dimension = pd.DataFrame({
    "week_index":
        np.arange(1, TOTAL_WEEKS + 1)
})

week_dimension["week_start"] = (
    ANALYSIS_START_DATE
    +
    pd.to_timedelta(
        (week_dimension["week_index"] - 1) * 7,
        unit="D"
    )
)

all_stores = sorted(
    dim_dark_store["dark_store_id"].unique()
)

all_categories = sorted(
    dim_product["category"].unique()
)


complete_grid = pd.MultiIndex.from_product(
    [
        week_dimension["week_index"],
        all_stores,
        all_categories
    ],
    names=[
        "week_index",
        "dark_store_id",
        "category"
    ]
).to_frame(index=False)


complete_grid = complete_grid.merge(
    week_dimension,
    on="week_index",
    how="left"
)


weekly_store_category = complete_grid.merge(
    weekly_store_category_observed,
    on=[
        "week_index",
        "week_start",
        "dark_store_id",
        "category"
    ],
    how="left",
    validate="one_to_one"
)


zero_fill_columns = [
    "weekly_units",
    "weekly_revenue",
    "weekly_orders",
    "active_skus"
]

weekly_store_category[
    zero_fill_columns
] = (
    weekly_store_category[
        zero_fill_columns
    ].fillna(0)
)


print("=" * 100)
print(" WEEKLY STORE × CATEGORY DEMAND PANEL")
print("=" * 100)

print(f"Observation weeks     : {TOTAL_WEEKS:,}")
print(f"Dark stores           : {len(all_stores):,}")
print(f"Categories            : {len(all_categories):,}")
print(
    f"Expected panel rows   : "
    f"{TOTAL_WEEKS * len(all_stores) * len(all_categories):,}"
)
print(
    f"Actual panel rows     : "
    f"{len(weekly_store_category):,}"
)

print(
    f"Total units reconciled: "
    f"{weekly_store_category['weekly_units'].sum():,.0f}"
)

print(
    f"Total revenue         : "
    f"₹{weekly_store_category['weekly_revenue'].sum():,.2f}"
)

display(
    weekly_store_category.head(20)
)

 WEEKLY STORE × CATEGORY DEMAND PANEL
Observation weeks     : 105
Dark stores           : 20
Categories            : 8
Expected panel rows   : 16,800
Actual panel rows     : 16,800
Total units reconciled: 117,468
Total revenue         : ₹28,224,720.00


,week_index,dark_store_id,category,week_start,weekly_units,weekly_revenue,weekly_orders,active_skus
0,1,DS_01_01,Baby Care,2023-01-01,10.00,"5,000.00",2.00,2.00
1,1,DS_01_01,Beverages,2023-01-01,7.00,510.00,2.00,2.00
2,1,DS_01_01,Dairy & Bakery,2023-01-01,30.00,"5,330.00",6.00,8.00
3,1,DS_01_01,Fruits & Vegetables,2023-01-01,5.00,450.00,1.00,1.00
4,1,DS_01_01,Grocery & Staples,2023-01-01,11.00,"3,890.00",3.00,4.00
5,1,DS_01_01,Household,2023-01-01,8.00,"1,000.00",4.00,4.00
6,1,DS_01_01,Personal Care,2023-01-01,0.00,0.00,0.00,0.00
7,1,DS_01_01,Snacks & Branded Foods,2023-01-01,5.00,750.00,1.00,1.00
8,1,DS_01_02,Baby Care,2023-01-01,18.00,"7,140.00",5.00,5.00
9,1,DS_01_02,Beverages,2023-01-01,11.00,890.00,3.00,3.00


In [30]:
# ============================================================
# 24. STORE × CATEGORY DEMAND STATISTICS
# ============================================================

store_category_demand = (
    weekly_store_category
    .groupby(
        [
            "dark_store_id",
            "category"
        ]
    )
    .agg(
        total_units=("weekly_units", "sum"),
        total_revenue=("weekly_revenue", "sum"),
        total_orders=("weekly_orders", "sum"),

        avg_weekly_demand=("weekly_units", "mean"),
        median_weekly_demand=("weekly_units", "median"),
        demand_std=("weekly_units", "std"),

        active_weeks=(
            "weekly_units",
            lambda x: int((x > 0).sum())
        ),

        max_weekly_demand=("weekly_units", "max")
    )
    .reset_index()
)


store_category_demand["demand_frequency_pct"] = (
    store_category_demand["active_weeks"]
    /
    TOTAL_WEEKS
    * 100
)


store_category_demand["demand_cv"] = np.where(
    store_category_demand[
        "avg_weekly_demand"
    ] > 0,

    store_category_demand[
        "demand_std"
    ]
    /
    store_category_demand[
        "avg_weekly_demand"
    ],

    np.nan
)


print("=" * 110)
print(" STORE × CATEGORY DEMAND STATISTICS")
print("=" * 110)

print(
    f"Store-category combinations : "
    f"{len(store_category_demand):,}"
)

print("\nDemand CV distribution:")

display(
    store_category_demand[
        "demand_cv"
    ].describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90
        ]
    ).to_frame("value")
)

print("\nDemand frequency distribution:")

display(
    store_category_demand[
        "demand_frequency_pct"
    ].describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90
        ]
    ).to_frame("value")
)

print("\nHighest-volume store-category combinations:")

display(
    store_category_demand
    .sort_values(
        "total_units",
        ascending=False
    )
    .head(20)
)

 STORE × CATEGORY DEMAND STATISTICS
Store-category combinations : 160

Demand CV distribution:


,value
count,160.00
mean,0.83
std,0.08
min,0.57
10%,0.72
25%,0.78
50%,0.82
75%,0.88
90%,0.93
max,1.02



Demand frequency distribution:


,value
count,160.00
mean,85.52
std,4.02
min,76.19
10%,80.00
25%,82.86
50%,85.71
75%,88.57
90%,91.43
max,96.19



Highest-volume store-category combinations:


,dark_store_id,category,total_units,total_revenue,total_orders,avg_weekly_demand,median_weekly_demand,demand_std,active_weeks,max_weekly_demand,demand_frequency_pct,demand_cv
18,DS_02_01,Dairy & Bakery,952.00,"153,420.00",269.00,9.07,9.00,5.12,101,23.00,96.19,0.57
106,DS_07_02,Dairy & Bakery,947.00,"139,040.00",239.00,9.02,7.00,7.05,93,31.00,88.57,0.78
2,DS_01_01,Dairy & Bakery,927.00,"150,320.00",260.00,8.83,8.00,6.58,96,31.00,91.43,0.75
138,DS_09_02,Dairy & Bakery,916.00,"142,460.00",236.00,8.72,7.00,7.19,95,31.00,90.48,0.82
26,DS_02_02,Dairy & Bakery,898.00,"146,620.00",246.00,8.55,7.00,7.33,95,40.00,90.48,0.86
104,DS_07_02,Baby Care,898.00,"410,890.00",249.00,8.55,7.00,7.46,91,35.00,86.67,0.87
58,DS_04_02,Dairy & Bakery,885.00,"142,790.00",247.00,8.43,8.00,6.65,96,31.00,91.43,0.79
35,DS_03_01,Fruits & Vegetables,874.00,"152,620.00",235.00,8.32,8.00,5.96,93,27.00,88.57,0.72
42,DS_03_02,Dairy & Bakery,859.00,"136,120.00",230.00,8.18,7.00,5.71,96,27.00,91.43,0.70
112,DS_08_01,Baby Care,858.00,"418,970.00",254.00,8.17,7.00,6.12,97,34.00,92.38,0.75


## 25. Relative XYZ Demand Variability Classification

Weekly Store × Category demand provides sufficient density for inventory variability analysis.

Because the observed coefficient-of-variation range is relatively compressed, fixed textbook XYZ thresholds would provide limited analytical separation. A relative, data-driven XYZ framework is therefore applied:

- **X:** Lowest 25% of demand variability — relatively predictable
- **Y:** Middle 50% — moderate variability
- **Z:** Highest 25% — relatively volatile

These classes describe predictability relative to this analytical network and are not claimed as universal industry thresholds.

In [31]:
# ============================================================
# 25. RELATIVE XYZ DEMAND VARIABILITY CLASSIFICATION
# ============================================================

# ------------------------------------------------------------
# Data-driven CV thresholds
# ------------------------------------------------------------

XYZ_X_MAX = (
    store_category_demand["demand_cv"]
    .quantile(0.25)
)

XYZ_Y_MAX = (
    store_category_demand["demand_cv"]
    .quantile(0.75)
)


def assign_xyz(cv):
    """
    Relative demand predictability classification.

    X = lowest variability quartile
    Y = middle variability range
    Z = highest variability quartile
    """

    if pd.isna(cv):
        return "Unknown"

    if cv <= XYZ_X_MAX:
        return "X"

    elif cv <= XYZ_Y_MAX:
        return "Y"

    return "Z"


store_category_demand["xyz_class"] = (
    store_category_demand["demand_cv"]
    .apply(assign_xyz)
)


# ------------------------------------------------------------
# XYZ summary
# ------------------------------------------------------------

xyz_summary = (
    store_category_demand
    .groupby(
        "xyz_class",
        observed=True
    )
    .agg(
        store_category_pairs=(
            "category",
            "count"
        ),
        total_units=(
            "total_units",
            "sum"
        ),
        total_revenue=(
            "total_revenue",
            "sum"
        ),
        avg_cv=(
            "demand_cv",
            "mean"
        ),
        avg_demand_frequency_pct=(
            "demand_frequency_pct",
            "mean"
        )
    )
    .reset_index()
)


xyz_summary["pair_share_pct"] = (
    xyz_summary["store_category_pairs"]
    /
    xyz_summary["store_category_pairs"].sum()
    * 100
)


xyz_summary["revenue_share_pct"] = (
    xyz_summary["total_revenue"]
    /
    xyz_summary["total_revenue"].sum()
    * 100
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 105)
print(" RELATIVE XYZ DEMAND CLASSIFICATION")
print("=" * 105)

print(
    f"X — relatively predictable : "
    f"CV <= {XYZ_X_MAX:.4f}"
)

print(
    f"Y — moderate variability   : "
    f"{XYZ_X_MAX:.4f} < CV <= {XYZ_Y_MAX:.4f}"
)

print(
    f"Z — relatively volatile    : "
    f"CV > {XYZ_Y_MAX:.4f}"
)

print("\nXYZ Summary:")

display(xyz_summary)


# ------------------------------------------------------------
# Category × XYZ matrix
# ------------------------------------------------------------

xyz_category_matrix = pd.crosstab(
    store_category_demand["category"],
    store_category_demand["xyz_class"],
    margins=True
)

print("\n" + "=" * 105)
print(" CATEGORY × XYZ DISTRIBUTION")
print("=" * 105)

display(xyz_category_matrix)


# ------------------------------------------------------------
# Most volatile Store × Category combinations
# ------------------------------------------------------------

print("\n" + "=" * 105)
print(" MOST VOLATILE STORE × CATEGORY POSITIONS")
print("=" * 105)

display(
    store_category_demand[
        [
            "dark_store_id",
            "category",
            "total_units",
            "total_revenue",
            "avg_weekly_demand",
            "demand_std",
            "demand_cv",
            "demand_frequency_pct",
            "xyz_class"
        ]
    ]
    .sort_values(
        "demand_cv",
        ascending=False
    )
    .head(20)
)

 RELATIVE XYZ DEMAND CLASSIFICATION
X — relatively predictable : CV <= 0.7760
Y — moderate variability   : 0.7760 < CV <= 0.8787
Z — relatively volatile    : CV > 0.8787

XYZ Summary:


,xyz_class,store_category_pairs,total_units,total_revenue,avg_cv,avg_demand_frequency_pct,pair_share_pct,revenue_share_pct
0,X,40,"31,243.00","7,359,810.00",0.73,89.45,25.00,26.08
1,Y,80,"59,050.00","14,429,880.00",0.83,85.57,50.00,51.12
2,Z,40,"27,175.00","6,435,030.00",0.93,81.48,25.00,22.80



 CATEGORY × XYZ DISTRIBUTION


xyz_class,X,Y,Z,All
category,,,,
Baby Care,4,11,5,20
Beverages,3,13,4,20
Dairy & Bakery,7,9,4,20
Fruits & Vegetables,9,9,2,20
Grocery & Staples,6,10,4,20
Household,3,10,7,20
Personal Care,4,11,5,20
Snacks & Branded Foods,4,7,9,20
All,40,80,40,160



 MOST VOLATILE STORE × CATEGORY POSITIONS


,dark_store_id,category,total_units,total_revenue,avg_weekly_demand,demand_std,demand_cv,demand_frequency_pct,xyz_class
146,DS_10_01,Dairy & Bakery,682.00,"111,600.00",6.50,6.60,1.02,79.05,Z
135,DS_09_01,Snacks & Branded Foods,492.00,"65,310.00",4.69,4.75,1.01,79.05,Z
0,DS_01_01,Baby Care,679.00,"306,240.00",6.47,6.49,1.00,80.95,Z
62,DS_04_02,Personal Care,724.00,"230,320.00",6.90,6.88,1.00,84.76,Z
53,DS_04_01,Household,659.00,"132,860.00",6.28,6.21,0.99,76.19,Z
125,DS_08_02,Household,614.00,"127,180.00",5.85,5.73,0.98,80.00,Z
77,DS_05_02,Household,662.00,"144,090.00",6.30,6.17,0.98,85.71,Z
100,DS_07_01,Grocery & Staples,649.00,"201,030.00",6.18,6.04,0.98,80.00,Z
101,DS_07_01,Household,700.00,"147,910.00",6.67,6.41,0.96,80.95,Z
116,DS_08_01,Grocery & Staples,740.00,"213,470.00",7.05,6.78,0.96,83.81,Z


## 26. Inventory Operating Assumptions

The source dataset does not provide supplier lead times, warehouse replenishment cycles, service-level policies, or physical inventory snapshots.

To enable inventory decision modelling, category-specific operating assumptions are introduced transparently.

The assumptions reflect relative replenishment characteristics:

- Perishable categories receive shorter lead times and coverage periods.
- Ambient and non-perishable categories receive longer replenishment cycles.
- These parameters are scenario assumptions for portfolio modelling and do not represent proprietary Zepto operating policies.

All assumptions remain isolated from observed source fields so that measured and modelled metrics can be distinguished throughout the project.

In [32]:
# ============================================================
# 26. INVENTORY OPERATING ASSUMPTIONS
# ============================================================

inventory_assumptions = pd.DataFrame([
    {
        "category": "Fruits & Vegetables",
        "lead_time_days": 1,
        "target_coverage_days": 3
    },
    {
        "category": "Dairy & Bakery",
        "lead_time_days": 1,
        "target_coverage_days": 3
    },
    {
        "category": "Beverages",
        "lead_time_days": 2,
        "target_coverage_days": 5
    },
    {
        "category": "Snacks & Branded Foods",
        "lead_time_days": 2,
        "target_coverage_days": 6
    },
    {
        "category": "Grocery & Staples",
        "lead_time_days": 3,
        "target_coverage_days": 7
    },
    {
        "category": "Baby Care",
        "lead_time_days": 4,
        "target_coverage_days": 8
    },
    {
        "category": "Personal Care",
        "lead_time_days": 4,
        "target_coverage_days": 8
    },
    {
        "category": "Household",
        "lead_time_days": 5,
        "target_coverage_days": 10
    }
])

print("=" * 100)
print(" INVENTORY OPERATING ASSUMPTIONS")
print("=" * 100)

display(inventory_assumptions)


# ------------------------------------------------------------
# Validate category coverage
# ------------------------------------------------------------

source_categories = set(
    dim_product["category"]
    .dropna()
    .unique()
)

assumption_categories = set(
    inventory_assumptions["category"]
)

missing_assumptions = (
    source_categories -
    assumption_categories
)

extra_assumptions = (
    assumption_categories -
    source_categories
)

print("\nValidation:")

print(
    "Source categories               :",
    len(source_categories)
)

print(
    "Categories with assumptions     :",
    len(assumption_categories)
)

print(
    "Categories missing assumptions  :",
    missing_assumptions
)

print(
    "Unused assumption categories    :",
    extra_assumptions
)


if (
    len(missing_assumptions) == 0
    and
    len(extra_assumptions) == 0
):
    print(
        "\nCategory assumption validation: PASS ✅"
    )
else:
    print(
        "\nCategory assumption validation: REVIEW ⚠️"
    )

 INVENTORY OPERATING ASSUMPTIONS


,category,lead_time_days,target_coverage_days
0,Fruits & Vegetables,1,3
1,Dairy & Bakery,1,3
2,Beverages,2,5
3,Snacks & Branded Foods,2,6
4,Grocery & Staples,3,7
5,Baby Care,4,8
6,Personal Care,4,8
7,Household,5,10



Validation:
Source categories               : 8
Categories with assumptions     : 8
Categories missing assumptions  : set()
Unused assumption categories    : set()

Category assumption validation: PASS ✅


## 27. Store × Category Commercial Value Segmentation

Inventory service levels should reflect both demand uncertainty and commercial importance.

Each dark store's categories are therefore ranked by observed completed-order revenue and classified into relative commercial value tiers:

- **High Value:** Categories contributing to the first ~80% of store revenue
- **Medium Value:** Categories contributing to the next ~15%
- **Low Value:** Remaining categories

The category crossing a cumulative threshold remains in the higher tier so that strategically important revenue is not artificially split.

Modelled service-level targets are then assigned:

- **High Value:** 97%
- **Medium Value:** 95%
- **Low Value:** 92%

These service-level targets are portfolio modelling assumptions and do not represent proprietary operating policies of any quick-commerce company.

In [34]:
# ============================================================
# 27. STORE × CATEGORY COMMERCIAL VALUE TIERS — REVISED
# ============================================================

inventory_model = (
    store_category_demand
    .copy()
    .sort_values(
        ["dark_store_id", "total_revenue"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Store total revenue
# ------------------------------------------------------------

inventory_model["store_total_revenue"] = (
    inventory_model
    .groupby("dark_store_id")["total_revenue"]
    .transform("sum")
)


# ------------------------------------------------------------
# Category contribution within each store
# ------------------------------------------------------------

inventory_model["category_revenue_share"] = (
    inventory_model["total_revenue"]
    /
    inventory_model["store_total_revenue"]
)


inventory_model["cumulative_revenue_share"] = (
    inventory_model
    .groupby("dark_store_id")["category_revenue_share"]
    .cumsum()
)


# ------------------------------------------------------------
# Commercial value classification
#
# High   = cumulative revenue <= 80%
# Medium = cumulative revenue <= 95%
# Low    = remaining revenue
# ------------------------------------------------------------

inventory_model["value_tier"] = np.select(
    [
        inventory_model["cumulative_revenue_share"] <= 0.80,
        inventory_model["cumulative_revenue_share"] <= 0.95
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)


# ------------------------------------------------------------
# Service-level modelling assumptions
# ------------------------------------------------------------

SERVICE_LEVEL_MAP = {
    "High": 0.97,
    "Medium": 0.95,
    "Low": 0.92
}

inventory_model["service_level_target"] = (
    inventory_model["value_tier"]
    .map(SERVICE_LEVEL_MAP)
)


# ------------------------------------------------------------
# Revenue-share validation
# ------------------------------------------------------------

store_share_validation = (
    inventory_model
    .groupby("dark_store_id")["category_revenue_share"]
    .sum()
)

max_share_error = float(
    (store_share_validation - 1).abs().max()
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

value_tier_summary = (
    inventory_model
    .groupby("value_tier")
    .agg(
        store_category_pairs=("category", "count"),
        total_units=("total_units", "sum"),
        total_revenue=("total_revenue", "sum"),
        avg_demand_cv=("demand_cv", "mean")
    )
    .reset_index()
)


value_tier_summary["pair_share_pct"] = (
    value_tier_summary["store_category_pairs"]
    /
    value_tier_summary["store_category_pairs"].sum()
    * 100
)


value_tier_summary["revenue_share_pct"] = (
    value_tier_summary["total_revenue"]
    /
    value_tier_summary["total_revenue"].sum()
    * 100
)


print("=" * 105)
print(" STORE × CATEGORY COMMERCIAL VALUE TIERS — REVISED")
print("=" * 105)

print(
    f"Store-category positions : {len(inventory_model):,}"
)

print(
    f"Dark stores represented  : "
    f"{inventory_model['dark_store_id'].nunique():,}"
)

print(
    f"Maximum store share error: "
    f"{max_share_error:.10f}"
)

if max_share_error < 1e-9:
    print("Revenue-share validation : PASS ✅")
else:
    print("Revenue-share validation : REVIEW ⚠️")


print("\nValue-tier summary:")
display(value_tier_summary)


# ------------------------------------------------------------
# Value × variability matrix
# ------------------------------------------------------------

print("\n" + "=" * 105)
print(" COMMERCIAL VALUE × DEMAND VARIABILITY MATRIX")
print("=" * 105)

value_xyz_matrix = pd.crosstab(
    inventory_model["value_tier"],
    inventory_model["xyz_class"],
    margins=True
)

display(value_xyz_matrix)


# ------------------------------------------------------------
# Critical strategic segment
# ------------------------------------------------------------

high_value_volatile = (
    inventory_model[
        (inventory_model["value_tier"] == "High")
        &
        (inventory_model["xyz_class"] == "Z")
    ]
    .sort_values(
        "total_revenue",
        ascending=False
    )
)


print("\n" + "=" * 105)
print(" HIGH-VALUE × HIGH-VARIABILITY POSITIONS")
print("=" * 105)

print(
    f"High-Value + Z positions: "
    f"{len(high_value_volatile):,}"
)

display(
    high_value_volatile[
        [
            "dark_store_id",
            "category",
            "total_revenue",
            "category_revenue_share",
            "cumulative_revenue_share",
            "avg_weekly_demand",
            "demand_cv",
            "demand_frequency_pct",
            "xyz_class",
            "value_tier",
            "service_level_target"
        ]
    ]
    .head(20)
)

 STORE × CATEGORY COMMERCIAL VALUE TIERS — REVISED
Store-category positions : 160
Dark stores represented  : 20
Maximum store share error: 0.0000000000
Revenue-share validation : PASS ✅

Value-tier summary:


,value_tier,store_category_pairs,total_units,total_revenue,avg_demand_cv,pair_share_pct,revenue_share_pct
0,High,100,"75,754.00","22,152,060.00",0.82,62.50,78.48
1,Low,21,"13,350.00","1,755,400.00",0.84,13.12,6.22
2,Medium,39,"28,364.00","4,317,260.00",0.84,24.38,15.30



 COMMERCIAL VALUE × DEMAND VARIABILITY MATRIX


xyz_class,X,Y,Z,All
value_tier,,,,
High,27,51,22,100
Low,5,10,6,21
Medium,8,19,12,39
All,40,80,40,160



 HIGH-VALUE × HIGH-VARIABILITY POSITIONS
High-Value + Z positions: 22


,dark_store_id,category,total_revenue,category_revenue_share,cumulative_revenue_share,avg_weekly_demand,demand_cv,demand_frequency_pct,xyz_class,value_tier,service_level_target
24,DS_02_02,Baby Care,"377,070.00",0.25,0.25,7.63,0.89,83.81,Z,High,0.97
56,DS_04_02,Baby Care,"361,980.00",0.25,0.25,7.87,0.88,86.67,Z,High,0.97
40,DS_03_02,Baby Care,"361,830.00",0.25,0.25,7.18,0.92,82.86,Z,High,0.97
88,DS_06_02,Baby Care,"310,440.00",0.25,0.25,6.55,0.90,80.00,Z,High,0.97
0,DS_01_01,Baby Care,"306,240.00",0.22,0.22,6.47,1.00,80.95,Z,High,0.97
113,DS_08_01,Personal Care,"241,650.00",0.17,0.46,7.26,0.90,80.00,Z,High,0.97
57,DS_04_02,Personal Care,"230,320.00",0.16,0.41,6.90,1.00,84.76,Z,High,0.97
50,DS_04_01,Personal Care,"222,410.00",0.16,0.58,6.43,0.92,80.95,Z,High,0.97
146,DS_10_01,Grocery & Staples,"218,530.00",0.16,0.60,6.63,0.92,79.05,Z,High,0.97
121,DS_08_02,Personal Care,"213,660.00",0.16,0.43,6.26,0.95,76.19,Z,High,0.97


## 28. Safety Stock & Reorder Point Model

Observed weekly demand is combined with modelled lead-time and service-level assumptions to estimate inventory protection requirements.

For each Dark Store × Category position:

**Average Daily Demand**

Average Weekly Demand / 7

**Expected Lead-Time Demand**

Average Daily Demand × Lead Time

**Daily Demand Standard Deviation**

Weekly Demand Standard Deviation / √7

**Lead-Time Demand Uncertainty**

Daily Demand Standard Deviation × √Lead Time

**Safety Stock**

Service-Level Z Score × Lead-Time Demand Standard Deviation

**Reorder Point**

Expected Lead-Time Demand + Safety Stock

This model assumes approximately independent demand increments when translating weekly variability to daily and lead-time variability. It is a portfolio simulation and does not represent proprietary inventory policy.

In [36]:
# ============================================================
# 28. SAFETY STOCK & REORDER POINT MODEL — CORRECTED
# ============================================================

from scipy.stats import norm


# ------------------------------------------------------------
# Make reruns safe
# ------------------------------------------------------------

assumption_columns = [
    "lead_time_days",
    "target_coverage_days"
]

inventory_model = inventory_model.drop(
    columns=assumption_columns,
    errors="ignore"
)


# ------------------------------------------------------------
# Join validated category assumptions
# ------------------------------------------------------------

inventory_model = inventory_model.merge(
    inventory_assumptions,
    on="category",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# Validate assumption join
# ------------------------------------------------------------

missing_lead_times = int(
    inventory_model["lead_time_days"]
    .isna()
    .sum()
)

missing_coverage = int(
    inventory_model["target_coverage_days"]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# Convert weekly demand statistics to daily equivalents
#
# Approximation:
# weekly variance ≈ 7 × daily variance
# therefore:
# daily std ≈ weekly std / sqrt(7)
# ------------------------------------------------------------

inventory_model["avg_daily_demand"] = (
    inventory_model["avg_weekly_demand"]
    / 7
)

inventory_model["daily_demand_std"] = (
    inventory_model["demand_std"]
    / np.sqrt(7)
)


# ------------------------------------------------------------
# Expected demand during replenishment lead time
# ------------------------------------------------------------

inventory_model["mean_lead_time_demand"] = (
    inventory_model["avg_daily_demand"]
    *
    inventory_model["lead_time_days"]
)


# ------------------------------------------------------------
# Demand uncertainty during lead time
# ------------------------------------------------------------

inventory_model["lead_time_demand_std"] = (
    inventory_model["daily_demand_std"]
    *
    np.sqrt(
        inventory_model["lead_time_days"]
    )
)


# ------------------------------------------------------------
# Convert service level to Z-score
#
# High   97% → approx 1.88
# Medium 95% → approx 1.64
# Low    92% → approx 1.41
# ------------------------------------------------------------

inventory_model["service_z"] = norm.ppf(
    inventory_model["service_level_target"]
)


# ------------------------------------------------------------
# Safety stock
# ------------------------------------------------------------

inventory_model["safety_stock_units"] = np.ceil(
    inventory_model["service_z"]
    *
    inventory_model["lead_time_demand_std"]
).astype(int)


# ------------------------------------------------------------
# Reorder point
# ------------------------------------------------------------

inventory_model["reorder_point_units"] = np.ceil(
    inventory_model["mean_lead_time_demand"]
    +
    inventory_model["safety_stock_units"]
).astype(int)


# ------------------------------------------------------------
# Target inventory level
#
# Enough inventory for target coverage period
# plus safety stock.
# ------------------------------------------------------------

inventory_model["coverage_stock_units"] = (
    inventory_model["avg_daily_demand"]
    *
    inventory_model["target_coverage_days"]
)


inventory_model["target_inventory_units"] = np.ceil(
    np.maximum(
        inventory_model["reorder_point_units"],
        inventory_model["coverage_stock_units"]
        +
        inventory_model["safety_stock_units"]
    )
).astype(int)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

negative_safety_stock = int(
    (
        inventory_model["safety_stock_units"]
        < 0
    ).sum()
)

invalid_reorder_points = int(
    (
        inventory_model["reorder_point_units"]
        <
        inventory_model["mean_lead_time_demand"]
    ).sum()
)

invalid_target_inventory = int(
    (
        inventory_model["target_inventory_units"]
        <
        inventory_model["reorder_point_units"]
    ).sum()
)


print("=" * 112)
print(" SAFETY STOCK & REORDER POINT MODEL")
print("=" * 112)

print(
    f"Store-category positions       : "
    f"{len(inventory_model):,}"
)

print(
    f"Missing lead-time assumptions  : "
    f"{missing_lead_times:,}"
)

print(
    f"Missing coverage assumptions   : "
    f"{missing_coverage:,}"
)

print(
    f"Negative safety-stock values   : "
    f"{negative_safety_stock:,}"
)

print(
    f"Invalid reorder points         : "
    f"{invalid_reorder_points:,}"
)

print(
    f"Invalid target inventory       : "
    f"{invalid_target_inventory:,}"
)


if (
    missing_lead_times == 0
    and
    missing_coverage == 0
    and
    negative_safety_stock == 0
    and
    invalid_reorder_points == 0
    and
    invalid_target_inventory == 0
):
    print(
        "\nInventory mathematics validation: PASS ✅"
    )
else:
    print(
        "\nInventory mathematics validation: REVIEW ⚠️"
    )


# ------------------------------------------------------------
# Overall inventory-policy summary
# ------------------------------------------------------------

print("\n" + "=" * 112)
print(" INVENTORY POLICY SUMMARY")
print("=" * 112)

inventory_policy_summary = (
    inventory_model
    .groupby("value_tier")
    .agg(
        positions=("category", "count"),
        avg_daily_demand=(
            "avg_daily_demand",
            "mean"
        ),
        avg_safety_stock=(
            "safety_stock_units",
            "mean"
        ),
        avg_reorder_point=(
            "reorder_point_units",
            "mean"
        ),
        avg_target_inventory=(
            "target_inventory_units",
            "mean"
        )
    )
    .reset_index()
)

display(inventory_policy_summary)


# ------------------------------------------------------------
# Detailed sample — corrected
# ------------------------------------------------------------

print("\n" + "=" * 112)
print(" SAMPLE INVENTORY PARAMETERS")
print("=" * 112)

inventory_parameter_sample = (
    inventory_model
    .sort_values(
        [
            "reorder_point_units",
            "total_revenue"
        ],
        ascending=[
            False,
            False
        ]
    )
    [
        [
            "dark_store_id",
            "category",
            "value_tier",
            "xyz_class",
            "total_revenue",
            "avg_weekly_demand",
            "demand_cv",
            "lead_time_days",
            "target_coverage_days",
            "service_level_target",
            "service_z",
            "mean_lead_time_demand",
            "lead_time_demand_std",
            "safety_stock_units",
            "reorder_point_units",
            "target_inventory_units"
        ]
    ]
    .head(30)
    .reset_index(drop=True)
)

display(inventory_parameter_sample)

 SAFETY STOCK & REORDER POINT MODEL
Store-category positions       : 160
Missing lead-time assumptions  : 0
Missing coverage assumptions   : 0
Negative safety-stock values   : 0
Invalid reorder points         : 0
Invalid target inventory       : 0

Inventory mathematics validation: PASS ✅

 INVENTORY POLICY SUMMARY


,value_tier,positions,avg_daily_demand,avg_safety_stock,avg_reorder_point,avg_target_inventory
0,High,100,1.03,7.85,11.76,15.48
1,Low,21,0.86,4.33,6.33,9.81
2,Medium,39,0.99,5.15,7.38,10.13



 SAMPLE INVENTORY PARAMETERS


,dark_store_id,category,value_tier,xyz_class,total_revenue,avg_weekly_demand,demand_cv,lead_time_days,target_coverage_days,service_level_target,service_z,mean_lead_time_demand,lead_time_demand_std,safety_stock_units,reorder_point_units,target_inventory_units
0,DS_02_02,Household,High,Z,"156,020.00",7.10,0.89,5,10,0.97,1.88,5.07,5.32,11,17,22
1,DS_07_02,Baby Care,High,Y,"410,890.00",8.55,0.87,4,8,0.97,1.88,4.89,5.64,11,16,21
2,DS_05_01,Household,High,Y,"157,530.00",7.09,0.86,5,10,0.97,1.88,5.06,5.13,10,16,21
3,DS_03_01,Household,High,Y,"150,990.00",7.03,0.81,5,10,0.97,1.88,5.02,4.83,10,16,21
4,DS_07_01,Household,High,Z,"147,910.00",6.67,0.96,5,10,0.97,1.88,4.76,5.42,11,16,21
5,DS_05_01,Baby Care,High,Y,"396,830.00",8.03,0.85,4,8,0.97,1.88,4.59,5.17,10,15,20
6,DS_02_02,Baby Care,High,Z,"377,070.00",7.63,0.89,4,8,0.97,1.88,4.36,5.12,10,15,19
7,DS_04_02,Baby Care,High,Z,"361,980.00",7.87,0.88,4,8,0.97,1.88,4.50,5.23,10,15,19
8,DS_03_02,Baby Care,High,Z,"361,830.00",7.18,0.92,4,8,0.97,1.88,4.10,5.00,10,15,19
9,DS_05_01,Personal Care,High,Y,"248,520.00",7.48,0.87,4,8,0.97,1.88,4.27,4.92,10,15,19


## 29. Modelled Inventory Snapshot & Stockout Risk

The source dataset does not contain physical inventory-on-hand snapshots.

To demonstrate inventory decision intelligence, a reproducible scenario-based inventory position is generated for every Dark Store × Category combination.

The simulated inventory position is derived as a deterministic proportion of the calculated reorder point. This allows the project to evaluate:

- Days of Supply
- Reorder requirement
- Lead-time stockout probability
- Replenishment quantity
- Inventory risk band
- Recommended operational action

These inventory positions are modelled scenario values and do not represent actual Zepto stock levels.

In [37]:
# ============================================================
# 29. MODELLED INVENTORY SNAPSHOT & STOCKOUT RISK
# ============================================================

import hashlib


# ------------------------------------------------------------
# Deterministic inventory-position generator
#
# Range:
# 40% to 130% of calculated reorder point.
#
# The deterministic hash ensures that the same store-category
# receives the same inventory position every time the notebook
# is rerun.
# ------------------------------------------------------------

def deterministic_inventory_factor(
    dark_store_id,
    category,
    seed=SEED
):
    """
    Create a reproducible inventory-position multiplier.

    This is a portfolio simulation only and does NOT represent
    actual company inventory.
    """

    key = (
        f"{dark_store_id}|"
        f"{category}|"
        f"{seed}"
    )

    digest = hashlib.sha256(
        key.encode("utf-8")
    ).hexdigest()

    raw_value = (
        int(digest[:8], 16)
        /
        0xFFFFFFFF
    )

    # Scale from 0–1 into 0.40–1.30
    return 0.40 + (
        raw_value * 0.90
    )


# ------------------------------------------------------------
# Generate deterministic inventory position
# ------------------------------------------------------------

inventory_model["inventory_position_factor"] = (
    inventory_model.apply(
        lambda row:
            deterministic_inventory_factor(
                row["dark_store_id"],
                row["category"]
            ),
        axis=1
    )
)


inventory_model["modelled_inventory_on_hand"] = np.maximum(
    0,
    np.round(
        inventory_model["reorder_point_units"]
        *
        inventory_model["inventory_position_factor"]
    )
).astype(int)


# ------------------------------------------------------------
# Days of Supply
# ------------------------------------------------------------

inventory_model["days_of_supply"] = np.where(
    inventory_model["avg_daily_demand"] > 0,

    inventory_model["modelled_inventory_on_hand"]
    /
    inventory_model["avg_daily_demand"],

    np.nan
)


# ------------------------------------------------------------
# Inventory gap versus reorder point
#
# Negative = below reorder point
# Positive = above reorder point
# ------------------------------------------------------------

inventory_model["inventory_vs_reorder_gap"] = (
    inventory_model["modelled_inventory_on_hand"]
    -
    inventory_model["reorder_point_units"]
)


# ------------------------------------------------------------
# Reorder flag
# ------------------------------------------------------------

inventory_model["reorder_required"] = (
    inventory_model["modelled_inventory_on_hand"]
    <=
    inventory_model["reorder_point_units"]
)


# ------------------------------------------------------------
# Lead-time stockout probability
#
# Lead-time demand is approximated as Normal:
#
# Demand ~ N(
#     mean_lead_time_demand,
#     lead_time_demand_std
# )
#
# Stockout occurs when demand exceeds inventory on hand.
# ------------------------------------------------------------

def calculate_stockout_probability(row):

    mean_demand = (
        row["mean_lead_time_demand"]
    )

    demand_std = (
        row["lead_time_demand_std"]
    )

    inventory = (
        row["modelled_inventory_on_hand"]
    )

    # Deterministic-demand edge case
    if demand_std <= 0:

        return (
            1.0
            if inventory < mean_demand
            else 0.0
        )

    z = (
        inventory - mean_demand
    ) / demand_std

    probability = (
        1 - norm.cdf(z)
    )

    return float(
        np.clip(
            probability,
            0,
            1
        )
    )


inventory_model["stockout_probability"] = (
    inventory_model.apply(
        calculate_stockout_probability,
        axis=1
    )
)


inventory_model["stockout_probability_pct"] = (
    inventory_model[
        "stockout_probability"
    ] * 100
)


# ------------------------------------------------------------
# Stockout-risk band
# ------------------------------------------------------------

inventory_model["stockout_risk_band"] = pd.cut(
    inventory_model[
        "stockout_probability"
    ],
    bins=[
        -np.inf,
        0.10,
        0.30,
        0.50,
        np.inf
    ],
    labels=[
        "Low",
        "Medium",
        "High",
        "Critical"
    ]
)


# ------------------------------------------------------------
# Recommended replenishment quantity
#
# Replenish toward target inventory level.
# ------------------------------------------------------------

inventory_model[
    "recommended_replenishment_units"
] = np.maximum(
    inventory_model[
        "target_inventory_units"
    ]
    -
    inventory_model[
        "modelled_inventory_on_hand"
    ],
    0
).astype(int)


# ------------------------------------------------------------
# Replenishment urgency score
#
# Used later for operational prioritisation.
#
# Components:
# 50% stockout risk
# 30% commercial importance
# 20% demand volatility
# ------------------------------------------------------------

VALUE_SCORE_MAP = {
    "High": 1.00,
    "Medium": 0.60,
    "Low": 0.30
}

XYZ_SCORE_MAP = {
    "X": 0.30,
    "Y": 0.60,
    "Z": 1.00
}


inventory_model["value_score"] = (
    inventory_model[
        "value_tier"
    ].map(VALUE_SCORE_MAP)
)


inventory_model["variability_score"] = (
    inventory_model[
        "xyz_class"
    ].map(XYZ_SCORE_MAP)
)


inventory_model["replenishment_priority_score"] = (
    (
        0.50
        *
        inventory_model[
            "stockout_probability"
        ]
    )
    +
    (
        0.30
        *
        inventory_model[
            "value_score"
        ]
    )
    +
    (
        0.20
        *
        inventory_model[
            "variability_score"
        ]
    )
) * 100


# ------------------------------------------------------------
# Recommended management action
# ------------------------------------------------------------

def assign_inventory_action(row):

    risk = row[
        "stockout_risk_band"
    ]

    if (
        risk == "Critical"
        and
        row["value_tier"] == "High"
    ):
        return "Immediate Replenishment"

    elif risk == "Critical":
        return "Urgent Replenishment"

    elif risk == "High":
        return "Priority Replenishment"

    elif row["reorder_required"]:
        return "Planned Replenishment"

    return "Monitor"


inventory_model["recommended_action"] = (
    inventory_model.apply(
        assign_inventory_action,
        axis=1
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

missing_inventory = int(
    inventory_model[
        "modelled_inventory_on_hand"
    ]
    .isna()
    .sum()
)

negative_inventory = int(
    (
        inventory_model[
            "modelled_inventory_on_hand"
        ] < 0
    ).sum()
)

invalid_probability = int(
    (
        (
            inventory_model[
                "stockout_probability"
            ] < 0
        )
        |
        (
            inventory_model[
                "stockout_probability"
            ] > 1
        )
    ).sum()
)


print("=" * 116)
print(" MODELLED INVENTORY SNAPSHOT & STOCKOUT RISK")
print("=" * 116)

print(
    f"Inventory positions generated : "
    f"{len(inventory_model):,}"
)

print(
    f"Missing inventory values      : "
    f"{missing_inventory:,}"
)

print(
    f"Negative inventory values     : "
    f"{negative_inventory:,}"
)

print(
    f"Invalid probabilities         : "
    f"{invalid_probability:,}"
)


if (
    missing_inventory == 0
    and
    negative_inventory == 0
    and
    invalid_probability == 0
):
    print(
        "\nInventory snapshot validation: PASS ✅"
    )
else:
    print(
        "\nInventory snapshot validation: REVIEW ⚠️"
    )


# ------------------------------------------------------------
# Network inventory snapshot
# ------------------------------------------------------------

print("\n" + "=" * 116)
print(" NETWORK INVENTORY SNAPSHOT")
print("=" * 116)

print(
    f"Modelled inventory on hand     : "
    f"{inventory_model['modelled_inventory_on_hand'].sum():,.0f} units"
)

print(
    f"Target inventory requirement   : "
    f"{inventory_model['target_inventory_units'].sum():,.0f} units"
)

print(
    f"Positions requiring reorder    : "
    f"{inventory_model['reorder_required'].sum():,}"
)

print(
    f"Recommended replenishment      : "
    f"{inventory_model['recommended_replenishment_units'].sum():,.0f} units"
)

print(
    f"Average days of supply         : "
    f"{inventory_model['days_of_supply'].mean():.2f} days"
)

print(
    f"Average stockout probability   : "
    f"{inventory_model['stockout_probability_pct'].mean():.2f}%"
)

 MODELLED INVENTORY SNAPSHOT & STOCKOUT RISK
Inventory positions generated : 160
Missing inventory values      : 0
Negative inventory values     : 0
Invalid probabilities         : 0

Inventory snapshot validation: PASS ✅

 NETWORK INVENTORY SNAPSHOT
Modelled inventory on hand     : 1,309 units
Target inventory requirement   : 2,149 units
Positions requiring reorder    : 130
Recommended replenishment      : 841 units
Average days of supply         : 8.29 days
Average stockout probability   : 10.64%


In [39]:
# ============================================================
# 29A. REPLENISHMENT DECISION CONSISTENCY CORRECTION
# ============================================================

# ------------------------------------------------------------
# Replenishment should only be recommended once inventory
# reaches or falls below the calculated reorder point.
# ------------------------------------------------------------

inventory_model["recommended_replenishment_units"] = np.where(
    inventory_model["reorder_required"],

    np.maximum(
        inventory_model["target_inventory_units"]
        -
        inventory_model["modelled_inventory_on_hand"],
        0
    ),

    0
).astype(int)


# ------------------------------------------------------------
# Replenishment priority applies only to positions requiring
# an inventory intervention.
# ------------------------------------------------------------

raw_priority_score = (
    (
        0.50
        *
        inventory_model["stockout_probability"]
    )
    +
    (
        0.30
        *
        inventory_model["value_score"]
    )
    +
    (
        0.20
        *
        inventory_model["variability_score"]
    )
) * 100


inventory_model["replenishment_priority_score"] = np.where(
    inventory_model["reorder_required"],
    raw_priority_score,
    0
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

monitor_with_replenishment = int(
    (
        (~inventory_model["reorder_required"])
        &
        (
            inventory_model[
                "recommended_replenishment_units"
            ] > 0
        )
    ).sum()
)


reorder_without_replenishment = int(
    (
        inventory_model["reorder_required"]
        &
        (
            inventory_model[
                "recommended_replenishment_units"
            ] <= 0
        )
    ).sum()
)


print("=" * 112)
print(" REPLENISHMENT DECISION CONSISTENCY")
print("=" * 112)

print(
    f"Positions requiring reorder       : "
    f"{inventory_model['reorder_required'].sum():,}"
)

print(
    f"Non-reorder positions             : "
    f"{(~inventory_model['reorder_required']).sum():,}"
)

print(
    f"Monitor positions with replenishment: "
    f"{monitor_with_replenishment:,}"
)

print(
    f"Reorder positions without quantity  : "
    f"{reorder_without_replenishment:,}"
)


if (
    monitor_with_replenishment == 0
    and
    reorder_without_replenishment == 0
):
    print(
        "\nReplenishment consistency: PASS ✅"
    )
else:
    print(
        "\nReplenishment consistency: REVIEW ⚠️"
    )


print(
    "\nUpdated recommended replenishment:",
    f"{inventory_model['recommended_replenishment_units'].sum():,.0f}",
    "units"
)

 REPLENISHMENT DECISION CONSISTENCY
Positions requiring reorder       : 130
Non-reorder positions             : 30
Monitor positions with replenishment: 0
Reorder positions without quantity  : 0

Replenishment consistency: PASS ✅

Updated recommended replenishment: 800 units


In [40]:
# ============================================================
# 30. INVENTORY CONTROL TOWER
# ============================================================

# ------------------------------------------------------------
# Risk distribution
# ------------------------------------------------------------

risk_summary = (
    inventory_model
    .groupby(
        "stockout_risk_band",
        observed=True
    )
    .agg(
        positions=(
            "category",
            "count"
        ),

        modelled_inventory_units=(
            "modelled_inventory_on_hand",
            "sum"
        ),

        replenishment_units=(
            "recommended_replenishment_units",
            "sum"
        ),

        attributed_revenue=(
            "total_revenue",
            "sum"
        ),

        avg_stockout_probability_pct=(
            "stockout_probability_pct",
            "mean"
        )
    )
    .reset_index()
)


risk_summary["position_share_pct"] = (
    risk_summary["positions"]
    /
    risk_summary["positions"].sum()
    * 100
)


risk_summary["revenue_share_pct"] = (
    risk_summary["attributed_revenue"]
    /
    risk_summary["attributed_revenue"].sum()
    * 100
)


print("=" * 118)
print(" INVENTORY CONTROL TOWER")
print("=" * 118)

print("\nStockout-risk distribution:")

display(
    risk_summary
)


# ------------------------------------------------------------
# Management action summary
# ------------------------------------------------------------

action_summary = (
    inventory_model
    .groupby(
        "recommended_action"
    )
    .agg(
        positions=("category", "count"),

        replenishment_units=(
            "recommended_replenishment_units",
            "sum"
        ),

        attributed_revenue=(
            "total_revenue",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "attributed_revenue",
        ascending=False
    )
)


print("\n" + "=" * 118)
print(" MANAGEMENT ACTION SUMMARY")
print("=" * 118)

display(
    action_summary
)


# ------------------------------------------------------------
# Highest-priority inventory interventions
# ------------------------------------------------------------

inventory_priority = (
    inventory_model
    .sort_values(
        [
            "replenishment_priority_score",
            "total_revenue"
        ],
        ascending=[
            False,
            False
        ]
    )
    [
        [
            "dark_store_id",
            "category",
            "value_tier",
            "xyz_class",
            "total_revenue",
            "avg_weekly_demand",
            "lead_time_days",
            "modelled_inventory_on_hand",
            "safety_stock_units",
            "reorder_point_units",
            "target_inventory_units",
            "days_of_supply",
            "stockout_probability_pct",
            "stockout_risk_band",
            "recommended_replenishment_units",
            "replenishment_priority_score",
            "recommended_action"
        ]
    ]
    .reset_index(drop=True)
)


print("\n" + "=" * 118)
print(" TOP INVENTORY INTERVENTION PRIORITIES")
print("=" * 118)

display(
    inventory_priority.head(25)
)

 INVENTORY CONTROL TOWER

Stockout-risk distribution:


,stockout_risk_band,positions,modelled_inventory_units,replenishment_units,attributed_revenue,avg_stockout_probability_pct,position_share_pct,revenue_share_pct
0,Low,92,954,292,"17,084,360.00",3.03,57.50,60.53
1,Medium,56,300,395,"9,072,820.00",18.39,35.00,32.14
2,High,12,55,113,"2,067,540.00",32.78,7.50,7.33



 MANAGEMENT ACTION SUMMARY


,recommended_action,positions,replenishment_units,attributed_revenue
1,Planned Replenishment,118,687,"20,476,860.00"
0,Monitor,30,0,"5,680,320.00"
2,Priority Replenishment,12,113,"2,067,540.00"



 TOP INVENTORY INTERVENTION PRIORITIES


,dark_store_id,category,value_tier,xyz_class,total_revenue,avg_weekly_demand,lead_time_days,modelled_inventory_on_hand,safety_stock_units,reorder_point_units,target_inventory_units,days_of_supply,stockout_probability_pct,stockout_risk_band,recommended_replenishment_units,replenishment_priority_score,recommended_action
0,DS_05_02,Household,High,Z,"144,090.00",6.30,5,7,10,15,20,7.77,31.60,High,13,65.80,Priority Replenishment
1,DS_06_02,Baby Care,High,Z,"310,440.00",6.55,4,6,9,13,17,6.41,30.57,High,11,65.29,Priority Replenishment
2,DS_10_01,Grocery & Staples,High,Z,"218,530.00",6.63,3,5,8,11,15,5.28,29.47,Medium,10,64.74,Planned Replenishment
3,DS_04_02,Baby Care,High,Z,"361,980.00",7.87,4,8,10,15,19,7.12,25.13,Medium,11,62.56,Planned Replenishment
4,DS_01_01,Baby Care,High,Z,"306,240.00",6.47,4,7,10,14,18,7.58,25.02,Medium,11,62.51,Planned Replenishment
5,DS_09_02,Baby Care,High,Y,"357,070.00",7.68,4,6,9,14,18,5.47,36.13,High,12,60.07,Priority Replenishment
6,DS_09_01,Baby Care,High,Y,"339,720.00",7.10,4,6,9,14,18,5.91,33.86,High,12,58.93,Priority Replenishment
7,DS_02_01,Grocery & Staples,High,Y,"239,330.00",7.25,3,5,7,11,15,4.83,30.38,High,10,57.19,Priority Replenishment
8,DS_10_02,Dairy & Bakery,High,Z,"126,270.00",7.71,1,4,6,8,10,3.63,13.85,Medium,6,56.92,Planned Replenishment
9,DS_07_02,Baby Care,High,Y,"410,890.00",8.55,4,8,11,16,21,6.55,29.05,Medium,13,56.52,Planned Replenishment


## 31. DuckDB Analytical Database

The validated analytical tables are materialised into DuckDB to create a portable SQL analytics layer.

The database separates:

### Core Dimensions
- Customer
- Product
- Date
- Dark Store

### Transaction Facts
- Orders
- Order Items

### Operational Intelligence
- Weekly Store × Category Demand
- Inventory Intelligence
- Network SKU Classification

DuckDB is used to demonstrate relational SQL, CTEs, joins, window functions, aggregations and management-oriented analytical views before the final Power BI layer is created.

In [41]:
# ============================================================
# 31. DUCKDB ANALYTICAL DATABASE
# ============================================================

import duckdb
from pathlib import Path


# ------------------------------------------------------------
# Database location
# ------------------------------------------------------------

DATABASE_DIR = PROJECT_ROOT / "database"
DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DB_PATH = (
    DATABASE_DIR /
    "quick_commerce_intelligence.duckdb"
)


# ------------------------------------------------------------
# Open DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect(
    str(DB_PATH)
)


# ------------------------------------------------------------
# Tables to materialise
# ------------------------------------------------------------

database_tables = {
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_date": dim_date,
    "dim_dark_store": dim_dark_store,

    "fact_order": fact_order,
    "fact_order_item": fact_order_item,

    "weekly_store_category":
        weekly_store_category,

    "inventory_intelligence":
        inventory_model,

    "sku_network_profile":
        sku_network_profile,

    "inventory_assumptions":
        inventory_assumptions
}


# ------------------------------------------------------------
# Materialise each Pandas DataFrame into DuckDB
# ------------------------------------------------------------

for table_name, dataframe in database_tables.items():

    temp_view_name = (
        f"df_{table_name}"
    )

    con.register(
        temp_view_name,
        dataframe
    )

    con.execute(
        f"""
        CREATE OR REPLACE TABLE
        {table_name}
        AS
        SELECT *
        FROM {temp_view_name}
        """
    )

    con.unregister(
        temp_view_name
    )


# ------------------------------------------------------------
# Database metadata
# ------------------------------------------------------------

database_inventory = con.execute(
    """
    SELECT
        table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type = 'BASE TABLE'
    ORDER BY table_name
    """
).df()


# ------------------------------------------------------------
# Row-count validation
# ------------------------------------------------------------

row_count_records = []

for table_name in database_inventory[
    "table_name"
]:

    row_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM {table_name}
        """
    ).fetchone()[0]

    row_count_records.append(
        {
            "table_name": table_name,
            "rows": row_count
        }
    )


database_row_counts = pd.DataFrame(
    row_count_records
)


print("=" * 105)
print(" DUCKDB ANALYTICAL DATABASE")
print("=" * 105)

print(
    f"Database file : "
    f"{DB_PATH}"
)

print(
    f"Tables created: "
    f"{len(database_inventory):,}"
)

print("\nDatabase tables:")

display(
    database_row_counts
)


# ------------------------------------------------------------
# Expected counts
# ------------------------------------------------------------

expected_counts = {
    "dim_customer":
        len(dim_customer),

    "dim_product":
        len(dim_product),

    "dim_date":
        len(dim_date),

    "dim_dark_store":
        len(dim_dark_store),

    "fact_order":
        len(fact_order),

    "fact_order_item":
        len(fact_order_item),

    "weekly_store_category":
        len(weekly_store_category),

    "inventory_intelligence":
        len(inventory_model),

    "sku_network_profile":
        len(sku_network_profile),

    "inventory_assumptions":
        len(inventory_assumptions)
}


database_row_counts[
    "expected_rows"
] = (
    database_row_counts[
        "table_name"
    ].map(expected_counts)
)


database_row_counts[
    "row_count_match"
] = (
    database_row_counts["rows"]
    ==
    database_row_counts[
        "expected_rows"
    ]
)


print("\n" + "=" * 105)
print(" DATABASE ROW-COUNT VALIDATION")
print("=" * 105)

display(
    database_row_counts
)


if database_row_counts[
    "row_count_match"
].all():

    print(
        "\nDatabase row-count validation: PASS ✅"
    )

else:

    print(
        "\nDatabase row-count validation: REVIEW ⚠️"
    )


# ------------------------------------------------------------
# File size
# ------------------------------------------------------------

con.execute(
    "CHECKPOINT"
)

database_size_mb = (
    DB_PATH.stat().st_size /
    (1024 ** 2)
)

print(
    f"\nDuckDB database size: "
    f"{database_size_mb:.2f} MB"
)

 DUCKDB ANALYTICAL DATABASE
Database file : /content/quick_commerce_intelligence/database/quick_commerce_intelligence.duckdb
Tables created: 10

Database tables:


,table_name,rows
0,dim_customer,10000
1,dim_dark_store,20
2,dim_date,731
3,dim_product,1200
4,fact_order,20000
5,fact_order_item,50000
6,inventory_assumptions,8
7,inventory_intelligence,160
8,sku_network_profile,1200
9,weekly_store_category,16800



 DATABASE ROW-COUNT VALIDATION


,table_name,rows,expected_rows,row_count_match
0,dim_customer,10000,10000,True
1,dim_dark_store,20,20,True
2,dim_date,731,731,True
3,dim_product,1200,1200,True
4,fact_order,20000,20000,True
5,fact_order_item,50000,50000,True
6,inventory_assumptions,8,8,True
7,inventory_intelligence,160,160,True
8,sku_network_profile,1200,1200,True
9,weekly_store_category,16800,16800,True



Database row-count validation: PASS ✅

DuckDB database size: 5.51 MB


In [42]:
# ============================================================
# 32. SQL RELATIONSHIP & DATABASE INTEGRITY VALIDATION
# ============================================================

sql_integrity_checks = con.execute(
    """
    WITH checks AS (

        SELECT
            'fact_order → dim_customer'
                AS relationship,

            COUNT(*) FILTER (
                WHERE c.customer_id IS NULL
            ) AS orphan_records

        FROM fact_order o

        LEFT JOIN dim_customer c
            ON o.customer_id =
               c.customer_id


        UNION ALL


        SELECT
            'fact_order → dim_dark_store',

            COUNT(*) FILTER (
                WHERE d.dark_store_id IS NULL
            )

        FROM fact_order o

        LEFT JOIN dim_dark_store d
            ON o.dark_store_id =
               d.dark_store_id


        UNION ALL


        SELECT
            'fact_order_item → fact_order',

            COUNT(*) FILTER (
                WHERE o.order_id IS NULL
            )

        FROM fact_order_item i

        LEFT JOIN fact_order o
            ON i.order_id =
               o.order_id


        UNION ALL


        SELECT
            'fact_order_item → dim_product',

            COUNT(*) FILTER (
                WHERE p.product_id IS NULL
            )

        FROM fact_order_item i

        LEFT JOIN dim_product p
            ON i.product_id =
               p.product_id


        UNION ALL


        SELECT
            'inventory → dim_dark_store',

            COUNT(*) FILTER (
                WHERE d.dark_store_id IS NULL
            )

        FROM inventory_intelligence i

        LEFT JOIN dim_dark_store d
            ON i.dark_store_id =
               d.dark_store_id
    )

    SELECT *
    FROM checks
    ORDER BY relationship
    """
).df()


sql_integrity_checks[
    "status"
] = np.where(
    sql_integrity_checks[
        "orphan_records"
    ] == 0,
    "PASS",
    "REVIEW"
)


print("=" * 105)
print(" SQL RELATIONSHIP VALIDATION")
print("=" * 105)

display(
    sql_integrity_checks
)


if (
    sql_integrity_checks[
        "orphan_records"
    ].sum()
    == 0
):

    print(
        "\nSQL relationship validation: PASS ✅"
    )

else:

    print(
        "\nSQL relationship validation: REVIEW ⚠️"
    )

 SQL RELATIONSHIP VALIDATION


,relationship,orphan_records,status
0,fact_order → dim_customer,0,PASS
1,fact_order → dim_dark_store,0,PASS
2,fact_order_item → dim_product,0,PASS
3,fact_order_item → fact_order,0,PASS
4,inventory → dim_dark_store,0,PASS



SQL relationship validation: PASS ✅


In [43]:
# ============================================================
# 33. SQL COMMERCIAL KPI RECONCILIATION
# ============================================================

sql_commercial_kpis = con.execute(
    """
    SELECT

        COUNT(*)
            AS total_orders,

        COUNT(*) FILTER (
            WHERE commercial_eligible = TRUE
        )
            AS commercial_orders,

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        )
            AS completed_orders,

        ROUND(
            SUM(
                CASE
                    WHEN completed_sales_eligible = TRUE
                    THEN gross_merchandise_value
                    ELSE 0
                END
            ),
            2
        )
            AS completed_gmv,

        ROUND(
            SUM(
                CASE
                    WHEN completed_sales_eligible = TRUE
                    THEN gross_merchandise_value
                    ELSE 0
                END
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE completed_sales_eligible = TRUE
                ),
                0
            ),
            2
        )
            AS completed_aov,

        ROUND(
            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END
            ),
            2
        )
            AS avg_delivery_minutes,

        ROUND(
            100.0
            *
            COUNT(*) FILTER (
                WHERE
                    sla_eligible = TRUE
                    AND
                    derived_sla_status = 'On Time'
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE sla_eligible = TRUE
                ),
                0
            ),
            2
        )
            AS sla_on_time_pct

    FROM fact_order
    """
).df()


print("=" * 105)
print(" SQL COMMERCIAL KPI RECONCILIATION")
print("=" * 105)

display(
    sql_commercial_kpis
)

 SQL COMMERCIAL KPI RECONCILIATION


,total_orders,commercial_orders,completed_orders,completed_gmv,completed_aov,avg_delivery_minutes,sla_on_time_pct
0,20000,18372,14333,"28,224,720.00","1,969.21",41.30,42.50


## 34. SQL Analytics Layer

The validated DuckDB warehouse is now queried through a standalone SQL analytics layer.

The SQL portfolio is organised by business domain so that recruiters and reviewers can inspect the analytical logic independently of the Python notebook.

The SQL layer demonstrates:

- CTEs
- conditional aggregation
- joins
- window functions
- ranking
- rolling metrics
- period-over-period analysis
- operational segmentation
- management prioritisation

All scripts are written to the project `sql/` directory and executed against the same DuckDB database used for reconciliation.

In [45]:
# ============================================================
# 34. SQL PORTFOLIO FILE GENERATION
# ============================================================

SQL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

sql_files = {}

In [46]:
# ============================================================
# 35. SQL FILE 01 — SCHEMA & DATA QUALITY
# ============================================================

sql_01 = r"""
-- ==========================================================
-- 01_schema_and_quality.sql
-- Quick-Commerce Profitability & Fulfilment Intelligence
-- ==========================================================
--
-- Purpose:
-- Validate core warehouse dimensions, fact relationships,
-- analytical eligibility and data-quality flags.
-- ==========================================================


-- ----------------------------------------------------------
-- 1. TABLE ROW COUNTS
-- ----------------------------------------------------------

SELECT
    'dim_customer' AS table_name,
    COUNT(*) AS row_count
FROM dim_customer

UNION ALL

SELECT
    'dim_product',
    COUNT(*)
FROM dim_product

UNION ALL

SELECT
    'dim_date',
    COUNT(*)
FROM dim_date

UNION ALL

SELECT
    'dim_dark_store',
    COUNT(*)
FROM dim_dark_store

UNION ALL

SELECT
    'fact_order',
    COUNT(*)
FROM fact_order

UNION ALL

SELECT
    'fact_order_item',
    COUNT(*)
FROM fact_order_item

UNION ALL

SELECT
    'weekly_store_category',
    COUNT(*)
FROM weekly_store_category

UNION ALL

SELECT
    'inventory_intelligence',
    COUNT(*)
FROM inventory_intelligence
;


-- ----------------------------------------------------------
-- 2. ORDER-LEVEL DATA QUALITY PROFILE
-- ----------------------------------------------------------

SELECT
    data_quality_flag,
    COUNT(*) AS orders,

    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS order_share_pct

FROM fact_order

GROUP BY
    data_quality_flag

ORDER BY
    orders DESC
;


-- ----------------------------------------------------------
-- 3. ANALYTICAL ELIGIBILITY
-- ----------------------------------------------------------

SELECT

    COUNT(*) AS total_orders,

    COUNT(*) FILTER (
        WHERE commercial_eligible = TRUE
    ) AS commercial_eligible_orders,

    COUNT(*) FILTER (
        WHERE completed_sales_eligible = TRUE
    ) AS completed_sales_orders,

    COUNT(*) FILTER (
        WHERE cx_eligible = TRUE
    ) AS customer_experience_orders,

    COUNT(*) FILTER (
        WHERE sla_eligible = TRUE
    ) AS sla_eligible_orders

FROM fact_order
;


-- ----------------------------------------------------------
-- 4. REFERENTIAL INTEGRITY
-- ----------------------------------------------------------

WITH integrity_checks AS (

    SELECT
        'fact_order → dim_customer' AS relationship,
        COUNT(*) FILTER (
            WHERE c.customer_id IS NULL
        ) AS orphan_records

    FROM fact_order o

    LEFT JOIN dim_customer c
        ON o.customer_id = c.customer_id


    UNION ALL


    SELECT
        'fact_order → dim_dark_store',
        COUNT(*) FILTER (
            WHERE d.dark_store_id IS NULL
        )

    FROM fact_order o

    LEFT JOIN dim_dark_store d
        ON o.dark_store_id = d.dark_store_id


    UNION ALL


    SELECT
        'fact_order_item → fact_order',
        COUNT(*) FILTER (
            WHERE o.order_id IS NULL
        )

    FROM fact_order_item i

    LEFT JOIN fact_order o
        ON i.order_id = o.order_id


    UNION ALL


    SELECT
        'fact_order_item → dim_product',
        COUNT(*) FILTER (
            WHERE p.product_id IS NULL
        )

    FROM fact_order_item i

    LEFT JOIN dim_product p
        ON i.product_id = p.product_id

)

SELECT *
FROM integrity_checks
ORDER BY relationship
;
"""

sql_files[
    "01_schema_and_quality.sql"
] = sql_01

(SQL_DIR / "01_schema_and_quality.sql").write_text(
    sql_01,
    encoding="utf-8"
)

print(
    "Created:",
    SQL_DIR / "01_schema_and_quality.sql"
)

Created: /content/quick_commerce_intelligence/sql/01_schema_and_quality.sql


In [47]:
# ============================================================
# 36. SQL FILE 02 — COMMERCIAL PERFORMANCE
# ============================================================

sql_02 = r"""
-- ==========================================================
-- 02_commercial_performance.sql
-- Quick-Commerce Profitability & Fulfilment Intelligence
-- ==========================================================


-- ----------------------------------------------------------
-- 1. EXECUTIVE COMMERCIAL KPIs
-- ----------------------------------------------------------

SELECT

    COUNT(*) AS total_orders,

    COUNT(*) FILTER (
        WHERE completed_sales_eligible = TRUE
    ) AS completed_orders,

    ROUND(
        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN gross_merchandise_value
                ELSE 0
            END
        ),
        2
    ) AS completed_gmv,

    ROUND(
        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN gross_merchandise_value
                ELSE 0
            END
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE completed_sales_eligible = TRUE
            ),
            0
        ),
        2
    ) AS completed_aov,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE order_status = 'Cancelled'
        )
        /
        COUNT(*),
        2
    ) AS cancellation_rate_pct,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE order_status = 'Returned'
        )
        /
        COUNT(*),
        2
    ) AS return_rate_pct

FROM fact_order
;


-- ----------------------------------------------------------
-- 2. MONTHLY COMMERCIAL PERFORMANCE
-- ----------------------------------------------------------

WITH monthly_sales AS (

    SELECT

        DATE_TRUNC(
            'month',
            order_date
        ) AS month,

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        ) AS completed_orders,

        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN gross_merchandise_value
                ELSE 0
            END
        ) AS completed_gmv,

        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN total_units
                ELSE 0
            END
        ) AS units_sold

    FROM fact_order

    GROUP BY 1
),

monthly_growth AS (

    SELECT

        *,

        LAG(completed_gmv) OVER (
            ORDER BY month
        ) AS previous_month_gmv

    FROM monthly_sales
)

SELECT

    month,
    completed_orders,

    ROUND(
        completed_gmv,
        2
    ) AS completed_gmv,

    units_sold,

    ROUND(
        completed_gmv /
        NULLIF(completed_orders, 0),
        2
    ) AS aov,

    ROUND(
        100.0 *
        (
            completed_gmv -
            previous_month_gmv
        )
        /
        NULLIF(
            previous_month_gmv,
            0
        ),
        2
    ) AS mom_gmv_growth_pct

FROM monthly_growth

ORDER BY month
;


-- ----------------------------------------------------------
-- 3. CATEGORY PERFORMANCE
-- ----------------------------------------------------------

SELECT

    category,

    COUNT(
        DISTINCT order_id
    ) AS completed_orders,

    SUM(quantity)
        AS units_sold,

    ROUND(
        SUM(line_revenue),
        2
    ) AS revenue,

    ROUND(
        100.0 *
        SUM(line_revenue)
        /
        SUM(SUM(line_revenue)) OVER (),
        2
    ) AS revenue_share_pct,

    ROUND(
        SUM(line_revenue)
        /
        NULLIF(
            COUNT(DISTINCT order_id),
            0
        ),
        2
    ) AS revenue_per_order

FROM fact_order_item

WHERE
    completed_sales_eligible = TRUE

GROUP BY category

ORDER BY revenue DESC
;


-- ----------------------------------------------------------
-- 4. TOP PRODUCTS BY NETWORK REVENUE
-- ----------------------------------------------------------

SELECT

    product_id,
    product_name,
    category,
    completed_orders,
    units_sold,

    ROUND(
        revenue,
        2
    ) AS revenue,

    abc_class,

    RANK() OVER (
        ORDER BY revenue DESC
    ) AS revenue_rank

FROM sku_network_profile

ORDER BY revenue_rank

LIMIT 50
;
"""

sql_files[
    "02_commercial_performance.sql"
] = sql_02

(SQL_DIR / "02_commercial_performance.sql").write_text(
    sql_02,
    encoding="utf-8"
)

print(
    "Created:",
    SQL_DIR / "02_commercial_performance.sql"
)

Created: /content/quick_commerce_intelligence/sql/02_commercial_performance.sql


In [48]:
# ============================================================
# 37. SQL FILE 03 — DARK STORE OPERATIONS
# ============================================================

sql_03 = r"""
-- ==========================================================
-- 03_dark_store_operations.sql
-- Quick-Commerce Profitability & Fulfilment Intelligence
-- ==========================================================


-- ----------------------------------------------------------
-- 1. DARK STORE KPI BASE
-- ----------------------------------------------------------

WITH store_kpis AS (

    SELECT

        dark_store_id,
        dark_store_name,
        city,
        state,

        COUNT(*) AS total_orders,

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        ) AS completed_orders,

        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN gross_merchandise_value
                ELSE 0
            END
        ) AS completed_gmv,

        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ) AS avg_delivery_minutes,

        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN distance_km
            END
        ) AS avg_delivery_distance_km,

        COUNT(*) FILTER (
            WHERE
                sla_eligible = TRUE
                AND
                derived_sla_status = 'On Time'
        ) AS on_time_orders,

        COUNT(*) FILTER (
            WHERE sla_eligible = TRUE
        ) AS sla_orders

    FROM fact_order

    GROUP BY
        dark_store_id,
        dark_store_name,
        city,
        state
)

SELECT

    *,

    ROUND(
        completed_gmv,
        2
    ) AS completed_gmv_rounded,

    ROUND(
        completed_gmv /
        NULLIF(
            completed_orders,
            0
        ),
        2
    ) AS aov,

    ROUND(
        avg_delivery_minutes,
        2
    ) AS avg_delivery_minutes_rounded,

    ROUND(
        avg_delivery_distance_km,
        2
    ) AS avg_delivery_distance_km_rounded,

    ROUND(
        100.0 *
        on_time_orders /
        NULLIF(
            sla_orders,
            0
        ),
        2
    ) AS sla_on_time_pct

FROM store_kpis

ORDER BY completed_gmv DESC
;


-- ----------------------------------------------------------
-- 2. DARK STORE PERFORMANCE RANKING
-- ----------------------------------------------------------

WITH store_performance AS (

    SELECT

        dark_store_id,
        dark_store_name,
        city,

        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN gross_merchandise_value
                ELSE 0
            END
        ) AS completed_gmv,

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        ) AS completed_orders,

        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ) AS avg_delivery_minutes,

        100.0 *
        COUNT(*) FILTER (
            WHERE
                sla_eligible = TRUE
                AND
                derived_sla_status = 'On Time'
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE sla_eligible = TRUE
            ),
            0
        ) AS sla_on_time_pct

    FROM fact_order

    GROUP BY
        dark_store_id,
        dark_store_name,
        city
)

SELECT

    *,

    RANK() OVER (
        ORDER BY completed_gmv DESC
    ) AS gmv_rank,

    RANK() OVER (
        ORDER BY sla_on_time_pct DESC
    ) AS sla_rank,

    RANK() OVER (
        ORDER BY avg_delivery_minutes ASC
    ) AS delivery_speed_rank

FROM store_performance

ORDER BY gmv_rank
;


-- ----------------------------------------------------------
-- 3. STORE VS NETWORK BENCHMARK
-- ----------------------------------------------------------

WITH store_metrics AS (

    SELECT

        dark_store_id,
        dark_store_name,

        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN gross_merchandise_value
                ELSE 0
            END
        ) AS store_gmv,

        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ) AS store_delivery_minutes

    FROM fact_order

    GROUP BY
        dark_store_id,
        dark_store_name
),

benchmarked AS (

    SELECT

        *,

        AVG(store_gmv) OVER ()
            AS network_avg_store_gmv,

        AVG(store_delivery_minutes) OVER ()
            AS network_avg_delivery_minutes

    FROM store_metrics
)

SELECT

    dark_store_id,
    dark_store_name,

    ROUND(
        store_gmv,
        2
    ) AS store_gmv,

    ROUND(
        network_avg_store_gmv,
        2
    ) AS network_avg_store_gmv,

    ROUND(
        100.0 *
        (
            store_gmv -
            network_avg_store_gmv
        )
        /
        NULLIF(
            network_avg_store_gmv,
            0
        ),
        2
    ) AS gmv_vs_network_pct,

    ROUND(
        store_delivery_minutes,
        2
    ) AS store_delivery_minutes,

    ROUND(
        store_delivery_minutes -
        network_avg_delivery_minutes,
        2
    ) AS delivery_minutes_vs_network

FROM benchmarked

ORDER BY
    gmv_vs_network_pct DESC
;
"""

sql_files[
    "03_dark_store_operations.sql"
] = sql_03

(SQL_DIR / "03_dark_store_operations.sql").write_text(
    sql_03,
    encoding="utf-8"
)

print(
    "Created:",
    SQL_DIR / "03_dark_store_operations.sql"
)

Created: /content/quick_commerce_intelligence/sql/03_dark_store_operations.sql


In [49]:
# ============================================================
# 38. SQL ASSET VALIDATION
# ============================================================

generated_sql_files = sorted(
    SQL_DIR.glob("*.sql")
)

print("=" * 100)
print(" SQL PORTFOLIO ASSETS")
print("=" * 100)

for sql_file in generated_sql_files:

    file_size_kb = (
        sql_file.stat().st_size /
        1024
    )

    line_count = len(
        sql_file.read_text(
            encoding="utf-8"
        ).splitlines()
    )

    print(
        f"✓ {sql_file.name:<38} "
        f"{line_count:>4} lines   "
        f"{file_size_kb:>7.2f} KB"
    )

print(
    f"\nSQL files created: "
    f"{len(generated_sql_files)}"
)

 SQL PORTFOLIO ASSETS
✓ 01_schema_and_quality.sql               192 lines      3.30 KB
✓ 02_commercial_performance.sql           239 lines      4.07 KB
✓ 03_dark_store_operations.sql            281 lines      5.09 KB

SQL files created: 3


In [50]:
# ============================================================
# 39. SQL ANALYTICS EXECUTION TEST
# ============================================================

print("=" * 105)
print(" SQL ANALYTICS EXECUTION TEST")
print("=" * 105)


# ------------------------------------------------------------
# A. Monthly commercial performance
# ------------------------------------------------------------

monthly_sql_test = con.execute(
    """
    WITH monthly_sales AS (

        SELECT

            DATE_TRUNC(
                'month',
                order_date
            ) AS month,

            COUNT(*) FILTER (
                WHERE completed_sales_eligible = TRUE
            ) AS completed_orders,

            SUM(
                CASE
                    WHEN completed_sales_eligible = TRUE
                    THEN gross_merchandise_value
                    ELSE 0
                END
            ) AS completed_gmv

        FROM fact_order

        GROUP BY 1
    ),

    monthly_growth AS (

        SELECT

            *,

            LAG(completed_gmv) OVER (
                ORDER BY month
            ) AS previous_month_gmv

        FROM monthly_sales
    )

    SELECT

        month,
        completed_orders,

        ROUND(
            completed_gmv,
            2
        ) AS completed_gmv,

        ROUND(
            100.0 *
            (
                completed_gmv -
                previous_month_gmv
            )
            /
            NULLIF(
                previous_month_gmv,
                0
            ),
            2
        ) AS mom_gmv_growth_pct

    FROM monthly_growth

    ORDER BY month
    """
).df()


print("\nMonthly commercial performance:")

display(
    monthly_sql_test.head(12)
)


# ------------------------------------------------------------
# B. Dark store rankings
# ------------------------------------------------------------

dark_store_sql_test = con.execute(
    """
    WITH store_performance AS (

        SELECT

            dark_store_id,
            dark_store_name,

            SUM(
                CASE
                    WHEN completed_sales_eligible = TRUE
                    THEN gross_merchandise_value
                    ELSE 0
                END
            ) AS completed_gmv,

            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END
            ) AS avg_delivery_minutes,

            100.0 *
            COUNT(*) FILTER (
                WHERE
                    sla_eligible = TRUE
                    AND
                    derived_sla_status = 'On Time'
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE sla_eligible = TRUE
                ),
                0
            ) AS sla_on_time_pct

        FROM fact_order

        GROUP BY
            dark_store_id,
            dark_store_name
    )

    SELECT

        dark_store_id,
        dark_store_name,

        ROUND(
            completed_gmv,
            2
        ) AS completed_gmv,

        ROUND(
            avg_delivery_minutes,
            2
        ) AS avg_delivery_minutes,

        ROUND(
            sla_on_time_pct,
            2
        ) AS sla_on_time_pct,

        RANK() OVER (
            ORDER BY completed_gmv DESC
        ) AS gmv_rank,

        RANK() OVER (
            ORDER BY sla_on_time_pct DESC
        ) AS sla_rank

    FROM store_performance

    ORDER BY gmv_rank
    """
).df()


print("\nDark store performance ranking:")

display(
    dark_store_sql_test.head(10)
)


print(
    "\nSQL analytics execution: PASS ✅"
)

 SQL ANALYTICS EXECUTION TEST

Monthly commercial performance:


,month,completed_orders,completed_gmv,mom_gmv_growth_pct
0,2023-01-01,631,"1,239,870.00",NaN
1,2023-02-01,535,"1,035,340.00",-16.50
2,2023-03-01,626,"1,292,880.00",24.87
3,2023-04-01,603,"1,184,650.00",-8.37
4,2023-05-01,628,"1,290,800.00",8.96
5,2023-06-01,598,"1,197,520.00",-7.23
6,2023-07-01,599,"1,209,700.00",1.02
7,2023-08-01,595,"1,166,070.00",-3.61
8,2023-09-01,606,"1,169,010.00",0.25
9,2023-10-01,647,"1,330,050.00",13.78



Dark store performance ranking:


,dark_store_id,dark_store_name,completed_gmv,avg_delivery_minutes,sla_on_time_pct,gmv_rank,sla_rank
0,DS_02_02,Delhi Dark Store 02,"1,534,750.00",41.81,42.44,1,12
1,DS_05_01,Pune Dark Store 01,"1,504,950.00",40.56,43.30,2,8
2,DS_07_02,Jaipur Dark Store 02,"1,502,010.00",40.71,43.08,3,10
3,DS_06_01,Hyderabad Dark Store 01,"1,500,460.00",42.20,41.34,4,15
4,DS_02_01,Delhi Dark Store 01,"1,474,580.00",40.38,43.36,5,6
5,DS_09_02,Lucknow Dark Store 02,"1,458,320.00",41.72,41.05,6,16
6,DS_03_01,Ahmedabad Dark Store 01,"1,450,020.00",42.10,39.13,7,18
7,DS_01_02,Kolkata Dark Store 02,"1,446,140.00",41.93,40.63,8,17
8,DS_03_02,Ahmedabad Dark Store 02,"1,439,020.00",42.09,38.72,9,20
9,DS_08_01,Bengaluru Dark Store 01,"1,437,450.00",40.84,43.85,10,5



SQL analytics execution: PASS ✅


In [51]:
# ============================================================
# 40. SQL FILE 04 — INVENTORY INTELLIGENCE
# ============================================================

sql_04 = r"""
-- ==========================================================
-- 04_inventory_intelligence.sql
-- Quick-Commerce Profitability & Fulfilment Intelligence
-- ==========================================================
--
-- Purpose:
-- Analyse inventory position, stockout exposure,
-- replenishment priorities and weekly demand behaviour.
-- ==========================================================


-- ----------------------------------------------------------
-- 1. NETWORK INVENTORY CONTROL TOWER
-- ----------------------------------------------------------

SELECT

    COUNT(*) AS inventory_positions,

    SUM(modelled_inventory_on_hand)
        AS modelled_inventory_units,

    SUM(target_inventory_units)
        AS target_inventory_units,

    COUNT(*) FILTER (
        WHERE reorder_required = TRUE
    ) AS reorder_positions,

    SUM(recommended_replenishment_units)
        AS recommended_replenishment_units,

    ROUND(
        AVG(days_of_supply),
        2
    ) AS avg_days_of_supply,

    ROUND(
        AVG(stockout_probability_pct),
        2
    ) AS avg_stockout_probability_pct

FROM inventory_intelligence
;


-- ----------------------------------------------------------
-- 2. STOCKOUT RISK DISTRIBUTION
-- ----------------------------------------------------------

SELECT

    stockout_risk_band,

    COUNT(*) AS positions,

    SUM(modelled_inventory_on_hand)
        AS inventory_units,

    SUM(recommended_replenishment_units)
        AS replenishment_units,

    ROUND(
        SUM(total_revenue),
        2
    ) AS attributed_revenue,

    ROUND(
        AVG(stockout_probability_pct),
        2
    ) AS avg_stockout_probability_pct,

    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS position_share_pct,

    ROUND(
        100.0 * SUM(total_revenue) /
        SUM(SUM(total_revenue)) OVER (),
        2
    ) AS revenue_share_pct

FROM inventory_intelligence

GROUP BY
    stockout_risk_band

ORDER BY
    avg_stockout_probability_pct DESC
;


-- ----------------------------------------------------------
-- 3. MANAGEMENT REPLENISHMENT PRIORITY
-- ----------------------------------------------------------

WITH ranked_inventory AS (

    SELECT

        dark_store_id,
        category,
        value_tier,
        xyz_class,

        total_revenue,

        modelled_inventory_on_hand,
        safety_stock_units,
        reorder_point_units,
        target_inventory_units,

        days_of_supply,
        stockout_probability_pct,
        stockout_risk_band,

        recommended_replenishment_units,
        replenishment_priority_score,
        recommended_action,

        RANK() OVER (
            ORDER BY
                replenishment_priority_score DESC,
                total_revenue DESC
        ) AS network_priority_rank,

        RANK() OVER (
            PARTITION BY dark_store_id
            ORDER BY
                replenishment_priority_score DESC,
                total_revenue DESC
        ) AS store_priority_rank

    FROM inventory_intelligence

    WHERE
        reorder_required = TRUE
)

SELECT *
FROM ranked_inventory

ORDER BY
    network_priority_rank
;


-- ----------------------------------------------------------
-- 4. CATEGORY INVENTORY RISK
-- ----------------------------------------------------------

SELECT

    category,

    COUNT(*) AS store_positions,

    COUNT(*) FILTER (
        WHERE reorder_required = TRUE
    ) AS reorder_positions,

    COUNT(*) FILTER (
        WHERE stockout_risk_band = 'High'
    ) AS high_risk_positions,

    COUNT(*) FILTER (
        WHERE stockout_risk_band = 'Critical'
    ) AS critical_risk_positions,

    SUM(recommended_replenishment_units)
        AS replenishment_units,

    ROUND(
        AVG(days_of_supply),
        2
    ) AS avg_days_of_supply,

    ROUND(
        AVG(stockout_probability_pct),
        2
    ) AS avg_stockout_probability_pct,

    ROUND(
        SUM(total_revenue),
        2
    ) AS attributed_revenue

FROM inventory_intelligence

GROUP BY category

ORDER BY
    avg_stockout_probability_pct DESC
;


-- ----------------------------------------------------------
-- 5. HIGH-VALUE × VOLATILE INVENTORY
-- ----------------------------------------------------------

SELECT

    dark_store_id,
    category,

    ROUND(
        total_revenue,
        2
    ) AS attributed_revenue,

    avg_weekly_demand,
    demand_cv,

    modelled_inventory_on_hand,
    reorder_point_units,
    target_inventory_units,

    ROUND(
        stockout_probability_pct,
        2
    ) AS stockout_probability_pct,

    recommended_replenishment_units,
    recommended_action

FROM inventory_intelligence

WHERE
    value_tier = 'High'
    AND xyz_class = 'Z'

ORDER BY
    replenishment_priority_score DESC,
    total_revenue DESC
;


-- ----------------------------------------------------------
-- 6. FOUR-WEEK ROLLING DEMAND
-- Demonstrates window functions on the complete weekly panel.
-- ----------------------------------------------------------

WITH demand_windows AS (

    SELECT

        week_index,
        week_start,
        dark_store_id,
        category,
        weekly_units,
        weekly_revenue,

        LAG(weekly_units) OVER (
            PARTITION BY
                dark_store_id,
                category
            ORDER BY week_index
        ) AS previous_week_units,

        AVG(weekly_units) OVER (
            PARTITION BY
                dark_store_id,
                category
            ORDER BY week_index
            ROWS BETWEEN
                3 PRECEDING
                AND CURRENT ROW
        ) AS rolling_4w_units,

        SUM(weekly_revenue) OVER (
            PARTITION BY
                dark_store_id,
                category
            ORDER BY week_index
            ROWS BETWEEN
                3 PRECEDING
                AND CURRENT ROW
        ) AS rolling_4w_revenue

    FROM weekly_store_category
)

SELECT

    week_index,
    week_start,
    dark_store_id,
    category,
    weekly_units,
    previous_week_units,

    ROUND(
        rolling_4w_units,
        2
    ) AS rolling_4w_avg_units,

    ROUND(
        rolling_4w_revenue,
        2
    ) AS rolling_4w_revenue,

    ROUND(
        100.0 *
        (
            weekly_units -
            previous_week_units
        )
        /
        NULLIF(
            previous_week_units,
            0
        ),
        2
    ) AS wow_unit_growth_pct

FROM demand_windows

ORDER BY
    dark_store_id,
    category,
    week_index
;
"""

sql_files[
    "04_inventory_intelligence.sql"
] = sql_04

(
    SQL_DIR /
    "04_inventory_intelligence.sql"
).write_text(
    sql_04,
    encoding="utf-8"
)

print(
    "Created:",
    SQL_DIR /
    "04_inventory_intelligence.sql"
)

Created: /content/quick_commerce_intelligence/sql/04_inventory_intelligence.sql


In [52]:
# ============================================================
# 41. SQL FILE 05 — FULFILMENT & SLA
# ============================================================

sql_05 = r"""
-- ==========================================================
-- 05_fulfilment_sla.sql
-- Quick-Commerce Profitability & Fulfilment Intelligence
-- ==========================================================
--
-- Purpose:
-- Analyse delivery speed, SLA attainment, distance effects
-- and fulfilment trends across the modelled network.
-- ==========================================================


-- ----------------------------------------------------------
-- 1. EXECUTIVE SLA KPIs
-- ----------------------------------------------------------

SELECT

    COUNT(*) FILTER (
        WHERE sla_eligible = TRUE
    ) AS sla_eligible_orders,

    COUNT(*) FILTER (
        WHERE
            sla_eligible = TRUE
            AND derived_sla_status = 'On Time'
    ) AS on_time_orders,

    COUNT(*) FILTER (
        WHERE
            sla_eligible = TRUE
            AND derived_sla_status = 'At Risk'
    ) AS at_risk_orders,

    COUNT(*) FILTER (
        WHERE
            sla_eligible = TRUE
            AND derived_sla_status = 'Late'
    ) AS late_orders,

    ROUND(
        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ),
        2
    ) AS avg_delivery_minutes,

    ROUND(
        MEDIAN(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ),
        2
    ) AS median_delivery_minutes,

    ROUND(
        QUANTILE_CONT(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END,
            0.90
        ),
        2
    ) AS p90_delivery_minutes,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE
                sla_eligible = TRUE
                AND derived_sla_status = 'On Time'
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE sla_eligible = TRUE
            ),
            0
        ),
        2
    ) AS sla_on_time_pct

FROM fact_order
;


-- ----------------------------------------------------------
-- 2. MONTHLY SLA TREND
-- ----------------------------------------------------------

WITH monthly_sla AS (

    SELECT

        DATE_TRUNC(
            'month',
            order_date
        ) AS month,

        COUNT(*) FILTER (
            WHERE sla_eligible = TRUE
        ) AS sla_orders,

        COUNT(*) FILTER (
            WHERE
                sla_eligible = TRUE
                AND derived_sla_status = 'On Time'
        ) AS on_time_orders,

        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ) AS avg_delivery_minutes,

        QUANTILE_CONT(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END,
            0.90
        ) AS p90_delivery_minutes

    FROM fact_order

    GROUP BY 1
),

monthly_metrics AS (

    SELECT

        *,

        100.0 *
        on_time_orders /
        NULLIF(
            sla_orders,
            0
        ) AS sla_on_time_pct

    FROM monthly_sla
),

trend AS (

    SELECT

        *,

        LAG(
            sla_on_time_pct
        ) OVER (
            ORDER BY month
        ) AS previous_month_sla

    FROM monthly_metrics
)

SELECT

    month,
    sla_orders,

    ROUND(
        avg_delivery_minutes,
        2
    ) AS avg_delivery_minutes,

    ROUND(
        p90_delivery_minutes,
        2
    ) AS p90_delivery_minutes,

    ROUND(
        sla_on_time_pct,
        2
    ) AS sla_on_time_pct,

    ROUND(
        sla_on_time_pct -
        previous_month_sla,
        2
    ) AS mom_sla_change_pp

FROM trend

ORDER BY month
;


-- ----------------------------------------------------------
-- 3. DELIVERY DISTANCE BANDS
-- ----------------------------------------------------------

WITH distance_bands AS (

    SELECT

        CASE

            WHEN distance_km <= 3
                THEN '0–3 km'

            WHEN distance_km <= 5
                THEN '3–5 km'

            WHEN distance_km <= 7
                THEN '5–7 km'

            ELSE '7+ km'

        END AS distance_band,

        CASE

            WHEN distance_km <= 3 THEN 1
            WHEN distance_km <= 5 THEN 2
            WHEN distance_km <= 7 THEN 3
            ELSE 4

        END AS distance_band_order,

        delivery_time_mins,
        derived_sla_status

    FROM fact_order

    WHERE sla_eligible = TRUE
)

SELECT

    distance_band,

    COUNT(*) AS orders,

    ROUND(
        AVG(delivery_time_mins),
        2
    ) AS avg_delivery_minutes,

    ROUND(
        MEDIAN(delivery_time_mins),
        2
    ) AS median_delivery_minutes,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE derived_sla_status = 'On Time'
        )
        /
        COUNT(*),
        2
    ) AS sla_on_time_pct

FROM distance_bands

GROUP BY
    distance_band,
    distance_band_order

ORDER BY
    distance_band_order
;


-- ----------------------------------------------------------
-- 4. DARK STORE SLA RANKING
-- ----------------------------------------------------------

WITH store_sla AS (

    SELECT

        dark_store_id,
        dark_store_name,
        city,

        COUNT(*) FILTER (
            WHERE sla_eligible = TRUE
        ) AS sla_orders,

        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ) AS avg_delivery_minutes,

        QUANTILE_CONT(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END,
            0.90
        ) AS p90_delivery_minutes,

        100.0 *
        COUNT(*) FILTER (
            WHERE
                sla_eligible = TRUE
                AND derived_sla_status = 'On Time'
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE sla_eligible = TRUE
            ),
            0
        ) AS sla_on_time_pct

    FROM fact_order

    GROUP BY
        dark_store_id,
        dark_store_name,
        city
)

SELECT

    dark_store_id,
    dark_store_name,
    city,
    sla_orders,

    ROUND(
        avg_delivery_minutes,
        2
    ) AS avg_delivery_minutes,

    ROUND(
        p90_delivery_minutes,
        2
    ) AS p90_delivery_minutes,

    ROUND(
        sla_on_time_pct,
        2
    ) AS sla_on_time_pct,

    RANK() OVER (
        ORDER BY
            sla_on_time_pct DESC
    ) AS sla_rank,

    RANK() OVER (
        ORDER BY
            p90_delivery_minutes ASC
    ) AS p90_speed_rank

FROM store_sla

ORDER BY sla_rank
;


-- ----------------------------------------------------------
-- 5. CITY FULFILMENT PERFORMANCE
-- ----------------------------------------------------------

SELECT

    city,
    state,

    COUNT(*) FILTER (
        WHERE sla_eligible = TRUE
    ) AS sla_orders,

    ROUND(
        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ),
        2
    ) AS avg_delivery_minutes,

    ROUND(
        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN distance_km
            END
        ),
        2
    ) AS avg_distance_km,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE
                sla_eligible = TRUE
                AND derived_sla_status = 'On Time'
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE sla_eligible = TRUE
            ),
            0
        ),
        2
    ) AS sla_on_time_pct

FROM fact_order

GROUP BY
    city,
    state

ORDER BY
    sla_on_time_pct DESC
;
"""

sql_files[
    "05_fulfilment_sla.sql"
] = sql_05

(
    SQL_DIR /
    "05_fulfilment_sla.sql"
).write_text(
    sql_05,
    encoding="utf-8"
)

print(
    "Created:",
    SQL_DIR /
    "05_fulfilment_sla.sql"
)

Created: /content/quick_commerce_intelligence/sql/05_fulfilment_sla.sql


In [53]:
# ============================================================
# 42. SQL PORTFOLIO ASSET VALIDATION — UPDATED
# ============================================================

generated_sql_files = sorted(
    SQL_DIR.glob("*.sql")
)

print("=" * 105)
print(" SQL PORTFOLIO ASSETS — UPDATED")
print("=" * 105)

for sql_file in generated_sql_files:

    file_size_kb = (
        sql_file.stat().st_size /
        1024
    )

    line_count = len(
        sql_file.read_text(
            encoding="utf-8"
        ).splitlines()
    )

    print(
        f"✓ {sql_file.name:<40}"
        f"{line_count:>5} lines   "
        f"{file_size_kb:>7.2f} KB"
    )

print(
    f"\nSQL files created: "
    f"{len(generated_sql_files)}"
)

 SQL PORTFOLIO ASSETS — UPDATED
✓ 01_schema_and_quality.sql                 192 lines      3.30 KB
✓ 02_commercial_performance.sql             239 lines      4.07 KB
✓ 03_dark_store_operations.sql              281 lines      5.09 KB
✓ 04_inventory_intelligence.sql             326 lines      6.38 KB
✓ 05_fulfilment_sla.sql                     425 lines      7.54 KB

SQL files created: 5


In [54]:
# ============================================================
# 43. INVENTORY SQL EXECUTION TEST
# ============================================================

inventory_sql_test = con.execute(
    """
    WITH ranked_inventory AS (

        SELECT

            dark_store_id,
            category,
            value_tier,
            xyz_class,

            total_revenue,
            modelled_inventory_on_hand,
            reorder_point_units,
            target_inventory_units,

            days_of_supply,
            stockout_probability_pct,

            recommended_replenishment_units,
            replenishment_priority_score,
            recommended_action,

            RANK() OVER (
                ORDER BY
                    replenishment_priority_score DESC,
                    total_revenue DESC
            ) AS network_priority_rank

        FROM inventory_intelligence

        WHERE
            reorder_required = TRUE
    )

    SELECT *
    FROM ranked_inventory

    ORDER BY
        network_priority_rank

    LIMIT 15
    """
).df()


print("=" * 108)
print(" INVENTORY SQL EXECUTION TEST")
print("=" * 108)

display(
    inventory_sql_test
)

print(
    "\nInventory SQL execution: PASS ✅"
)

 INVENTORY SQL EXECUTION TEST


,dark_store_id,category,value_tier,xyz_class,total_revenue,modelled_inventory_on_hand,reorder_point_units,target_inventory_units,days_of_supply,stockout_probability_pct,recommended_replenishment_units,replenishment_priority_score,recommended_action,network_priority_rank
0,DS_05_02,Household,High,Z,"144,090.00",7,15,20,7.77,31.60,13,65.80,Priority Replenishment,1
1,DS_06_02,Baby Care,High,Z,"310,440.00",6,13,17,6.41,30.57,11,65.29,Priority Replenishment,2
2,DS_10_01,Grocery & Staples,High,Z,"218,530.00",5,11,15,5.28,29.47,10,64.74,Planned Replenishment,3
3,DS_04_02,Baby Care,High,Z,"361,980.00",8,15,19,7.12,25.13,11,62.56,Planned Replenishment,4
4,DS_01_01,Baby Care,High,Z,"306,240.00",7,14,18,7.58,25.02,11,62.51,Planned Replenishment,5
5,DS_09_02,Baby Care,High,Y,"357,070.00",6,14,18,5.47,36.13,12,60.07,Priority Replenishment,6
6,DS_09_01,Baby Care,High,Y,"339,720.00",6,14,18,5.91,33.86,12,58.93,Priority Replenishment,7
7,DS_02_01,Grocery & Staples,High,Y,"239,330.00",5,11,15,4.83,30.38,10,57.19,Priority Replenishment,8
8,DS_10_02,Dairy & Bakery,High,Z,"126,270.00",4,8,10,3.63,13.85,6,56.92,Planned Replenishment,9
9,DS_07_02,Baby Care,High,Y,"410,890.00",8,16,21,6.55,29.05,13,56.52,Planned Replenishment,10



Inventory SQL execution: PASS ✅


In [55]:
# ============================================================
# 44. FULFILMENT SQL EXECUTION TEST
# ============================================================

fulfilment_sql_test = con.execute(
    """
    WITH monthly_sla AS (

        SELECT

            DATE_TRUNC(
                'month',
                order_date
            ) AS month,

            COUNT(*) FILTER (
                WHERE sla_eligible = TRUE
            ) AS sla_orders,

            COUNT(*) FILTER (
                WHERE
                    sla_eligible = TRUE
                    AND
                    derived_sla_status = 'On Time'
            ) AS on_time_orders,

            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END
            ) AS avg_delivery_minutes,

            QUANTILE_CONT(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END,
                0.90
            ) AS p90_delivery_minutes

        FROM fact_order

        GROUP BY 1
    ),

    monthly_metrics AS (

        SELECT

            *,

            100.0 *
            on_time_orders /
            NULLIF(
                sla_orders,
                0
            ) AS sla_on_time_pct

        FROM monthly_sla
    ),

    trend AS (

        SELECT

            *,

            LAG(
                sla_on_time_pct
            ) OVER (
                ORDER BY month
            ) AS previous_month_sla

        FROM monthly_metrics
    )

    SELECT

        month,
        sla_orders,

        ROUND(
            avg_delivery_minutes,
            2
        ) AS avg_delivery_minutes,

        ROUND(
            p90_delivery_minutes,
            2
        ) AS p90_delivery_minutes,

        ROUND(
            sla_on_time_pct,
            2
        ) AS sla_on_time_pct,

        ROUND(
            sla_on_time_pct -
            previous_month_sla,
            2
        ) AS mom_sla_change_pp

    FROM trend

    ORDER BY month
    """
).df()


print("=" * 108)
print(" FULFILMENT SQL EXECUTION TEST")
print("=" * 108)

display(
    fulfilment_sql_test.head(12)
)


# ------------------------------------------------------------
# Network reconciliation
# ------------------------------------------------------------

network_sla_sql = con.execute(
    """
    SELECT

        COUNT(*) FILTER (
            WHERE sla_eligible = TRUE
        ) AS sla_orders,

        ROUND(
            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END
            ),
            2
        ) AS avg_delivery_minutes,

        ROUND(
            QUANTILE_CONT(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END,
                0.90
            ),
            2
        ) AS p90_delivery_minutes,

        ROUND(
            100.0 *
            COUNT(*) FILTER (
                WHERE
                    sla_eligible = TRUE
                    AND
                    derived_sla_status = 'On Time'
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE sla_eligible = TRUE
                ),
                0
            ),
            2
        ) AS sla_on_time_pct

    FROM fact_order
    """
).df()


print("\nNetwork SLA reconciliation:")

display(
    network_sla_sql
)

print(
    "\nFulfilment SQL execution: PASS ✅"
)

 FULFILMENT SQL EXECUTION TEST


,month,sla_orders,avg_delivery_minutes,p90_delivery_minutes,sla_on_time_pct,mom_sla_change_pp
0,2023-01-01,676,42.51,73.50,41.12,NaN
1,2023-02-01,592,39.87,69.00,46.11,4.99
2,2023-03-01,675,41.97,71.00,40.74,-5.37
3,2023-04-01,643,40.56,70.80,43.23,2.49
4,2023-05-01,685,41.58,73.00,43.94,0.71
5,2023-06-01,654,39.69,70.00,45.26,1.32
6,2023-07-01,666,42.77,74.00,40.09,-5.17
7,2023-08-01,649,40.21,71.00,45.45,5.36
8,2023-09-01,663,40.98,72.00,41.78,-3.67
9,2023-10-01,707,41.09,70.00,42.15,0.37



Network SLA reconciliation:


,sla_orders,avg_delivery_minutes,p90_delivery_minutes,sla_on_time_pct
0,15618,41.30,71.00,42.50



Fulfilment SQL execution: PASS ✅


In [56]:
# ============================================================
# 45. SQL FILE 06 — CUSTOMER EXPERIENCE
# ============================================================

sql_06 = r"""
-- ==========================================================
-- 06_customer_experience.sql
-- Quick-Commerce Profitability & Fulfilment Intelligence
-- ==========================================================
--
-- Purpose:
-- Analyse delivered-order customer experience using the
-- canonical order-level rating layer.
--
-- Important:
-- CX metrics include only delivered orders with valid ratings.
-- Multiple source ratings per order were aggregated upstream
-- into one order-level average rating.
-- ==========================================================


-- ----------------------------------------------------------
-- 1. EXECUTIVE CUSTOMER EXPERIENCE KPIs
-- ----------------------------------------------------------

SELECT

    COUNT(*) FILTER (
        WHERE completed_sales_eligible = TRUE
    ) AS delivered_commercial_orders,

    COUNT(*) FILTER (
        WHERE cx_eligible = TRUE
    ) AS cx_eligible_orders,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE cx_eligible = TRUE
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE completed_sales_eligible = TRUE
            ),
            0
        ),
        2
    ) AS rating_coverage_pct,

    ROUND(
        AVG(
            CASE
                WHEN cx_eligible = TRUE
                THEN avg_rating
            END
        ),
        2
    ) AS avg_customer_rating,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE
                cx_eligible = TRUE
                AND avg_rating >= 4
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE cx_eligible = TRUE
            ),
            0
        ),
        2
    ) AS high_rating_order_pct

FROM fact_order
;


-- ----------------------------------------------------------
-- 2. CUSTOMER RATING DISTRIBUTION
-- ----------------------------------------------------------

WITH rating_bands AS (

    SELECT

        CASE
            WHEN avg_rating < 2
                THEN 'Below 2.0'

            WHEN avg_rating < 3
                THEN '2.0–2.9'

            WHEN avg_rating < 4
                THEN '3.0–3.9'

            ELSE '4.0–5.0'
        END AS rating_band,

        CASE
            WHEN avg_rating < 2 THEN 1
            WHEN avg_rating < 3 THEN 2
            WHEN avg_rating < 4 THEN 3
            ELSE 4
        END AS rating_band_order

    FROM fact_order

    WHERE
        cx_eligible = TRUE
)

SELECT

    rating_band,

    COUNT(*) AS orders,

    ROUND(
        100.0 *
        COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS order_share_pct

FROM rating_bands

GROUP BY
    rating_band,
    rating_band_order

ORDER BY
    rating_band_order
;


-- ----------------------------------------------------------
-- 3. DARK STORE CUSTOMER EXPERIENCE
-- ----------------------------------------------------------

WITH store_cx AS (

    SELECT

        dark_store_id,
        dark_store_name,
        city,

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        ) AS completed_orders,

        COUNT(*) FILTER (
            WHERE cx_eligible = TRUE
        ) AS rated_orders,

        AVG(
            CASE
                WHEN cx_eligible = TRUE
                THEN avg_rating
            END
        ) AS avg_rating,

        AVG(
            CASE
                WHEN cx_eligible = TRUE
                THEN delivery_time_mins
            END
        ) AS avg_delivery_minutes,

        100.0 *
        COUNT(*) FILTER (
            WHERE
                cx_eligible = TRUE
                AND derived_sla_status = 'On Time'
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE cx_eligible = TRUE
            ),
            0
        ) AS rated_order_sla_pct

    FROM fact_order

    GROUP BY
        dark_store_id,
        dark_store_name,
        city
)

SELECT

    dark_store_id,
    dark_store_name,
    city,
    completed_orders,
    rated_orders,

    ROUND(
        100.0 *
        rated_orders /
        NULLIF(completed_orders, 0),
        2
    ) AS rating_coverage_pct,

    ROUND(
        avg_rating,
        2
    ) AS avg_rating,

    ROUND(
        avg_delivery_minutes,
        2
    ) AS avg_delivery_minutes,

    ROUND(
        rated_order_sla_pct,
        2
    ) AS rated_order_sla_pct,

    RANK() OVER (
        ORDER BY avg_rating DESC
    ) AS customer_rating_rank

FROM store_cx

ORDER BY
    customer_rating_rank,
    dark_store_id
;


-- ----------------------------------------------------------
-- 4. RATING VS SLA PERFORMANCE
-- ----------------------------------------------------------

SELECT

    derived_sla_status,

    COUNT(*) AS rated_orders,

    ROUND(
        AVG(avg_rating),
        2
    ) AS avg_rating,

    ROUND(
        AVG(delivery_time_mins),
        2
    ) AS avg_delivery_minutes,

    ROUND(
        100.0 *
        COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS rated_order_share_pct

FROM fact_order

WHERE
    cx_eligible = TRUE

GROUP BY
    derived_sla_status

ORDER BY
    CASE derived_sla_status
        WHEN 'On Time' THEN 1
        WHEN 'At Risk' THEN 2
        WHEN 'Late' THEN 3
        ELSE 4
    END
;


-- ----------------------------------------------------------
-- 5. MONTHLY CUSTOMER EXPERIENCE TREND
-- ----------------------------------------------------------

WITH monthly_cx AS (

    SELECT

        DATE_TRUNC(
            'month',
            order_date
        ) AS month,

        COUNT(*) FILTER (
            WHERE cx_eligible = TRUE
        ) AS rated_orders,

        AVG(
            CASE
                WHEN cx_eligible = TRUE
                THEN avg_rating
            END
        ) AS avg_rating

    FROM fact_order

    GROUP BY 1
),

cx_trend AS (

    SELECT

        *,

        LAG(avg_rating) OVER (
            ORDER BY month
        ) AS previous_month_rating,

        AVG(avg_rating) OVER (
            ORDER BY month
            ROWS BETWEEN
                2 PRECEDING
                AND CURRENT ROW
        ) AS rolling_3m_avg_rating

    FROM monthly_cx
)

SELECT

    month,
    rated_orders,

    ROUND(
        avg_rating,
        3
    ) AS avg_rating,

    ROUND(
        avg_rating -
        previous_month_rating,
        3
    ) AS mom_rating_change,

    ROUND(
        rolling_3m_avg_rating,
        3
    ) AS rolling_3m_avg_rating

FROM cx_trend

ORDER BY month
;
"""

sql_files[
    "06_customer_experience.sql"
] = sql_06

(
    SQL_DIR /
    "06_customer_experience.sql"
).write_text(
    sql_06,
    encoding="utf-8"
)

print(
    "Created:",
    SQL_DIR /
    "06_customer_experience.sql"
)

Created: /content/quick_commerce_intelligence/sql/06_customer_experience.sql


In [57]:
# ============================================================
# 46. SQL FILE 07 — MANAGEMENT PRIORITY
# ============================================================

sql_07 = r"""
-- ==========================================================
-- 07_management_priority.sql
-- Quick-Commerce Profitability & Fulfilment Intelligence
-- ==========================================================
--
-- Purpose:
-- Combine commercial, fulfilment and inventory signals into
-- a management-level dark-store intervention framework.
--
-- This is a portfolio decision-support score.
-- Weights are analytical assumptions and do not represent
-- proprietary operating policies.
-- ==========================================================


-- ----------------------------------------------------------
-- 1. DARK STORE MANAGEMENT PRIORITY
-- ----------------------------------------------------------

WITH store_operations AS (

    SELECT

        dark_store_id,
        dark_store_name,
        city,
        state,

        COUNT(*) AS total_orders,

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        ) AS completed_orders,

        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN gross_merchandise_value
                ELSE 0
            END
        ) AS completed_gmv,

        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ) AS avg_delivery_minutes,

        100.0 *
        COUNT(*) FILTER (
            WHERE
                sla_eligible = TRUE
                AND derived_sla_status = 'On Time'
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE sla_eligible = TRUE
            ),
            0
        ) AS sla_on_time_pct,

        100.0 *
        COUNT(*) FILTER (
            WHERE order_status = 'Cancelled'
        )
        /
        NULLIF(
            COUNT(*),
            0
        ) AS cancellation_rate_pct,

        AVG(
            CASE
                WHEN cx_eligible = TRUE
                THEN avg_rating
            END
        ) AS avg_customer_rating

    FROM fact_order

    GROUP BY
        dark_store_id,
        dark_store_name,
        city,
        state
),


store_inventory AS (

    SELECT

        dark_store_id,

        COUNT(*) AS inventory_positions,

        COUNT(*) FILTER (
            WHERE reorder_required = TRUE
        ) AS reorder_positions,

        COUNT(*) FILTER (
            WHERE stockout_risk_band IN (
                'High',
                'Critical'
            )
        ) AS high_risk_inventory_positions,

        SUM(
            recommended_replenishment_units
        ) AS replenishment_units,

        AVG(
            stockout_probability_pct
        ) AS avg_stockout_probability_pct,

        MAX(
            replenishment_priority_score
        ) AS highest_inventory_priority_score

    FROM inventory_intelligence

    GROUP BY
        dark_store_id
),


combined AS (

    SELECT

        o.*,

        i.inventory_positions,
        i.reorder_positions,
        i.high_risk_inventory_positions,
        i.replenishment_units,
        i.avg_stockout_probability_pct,
        i.highest_inventory_priority_score

    FROM store_operations o

    LEFT JOIN store_inventory i
        ON o.dark_store_id =
           i.dark_store_id
),


normalized AS (

    SELECT

        *,

        -- High GMV = larger commercial exposure
        PERCENT_RANK() OVER (
            ORDER BY completed_gmv ASC
        ) AS commercial_exposure_score,

        -- Lower SLA = higher operational risk
        PERCENT_RANK() OVER (
            ORDER BY sla_on_time_pct DESC
        ) AS sla_risk_score,

        -- Slower delivery = higher operational risk
        PERCENT_RANK() OVER (
            ORDER BY avg_delivery_minutes ASC
        ) AS delivery_risk_score,

        -- Higher cancellation rate = higher risk
        PERCENT_RANK() OVER (
            ORDER BY cancellation_rate_pct ASC
        ) AS cancellation_risk_score,

        -- Higher stockout probability = higher risk
        PERCENT_RANK() OVER (
            ORDER BY avg_stockout_probability_pct ASC
        ) AS inventory_risk_score

    FROM combined
),


scored AS (

    SELECT

        *,

        (
            0.20 * commercial_exposure_score
            +
            0.25 * sla_risk_score
            +
            0.15 * delivery_risk_score
            +
            0.10 * cancellation_risk_score
            +
            0.30 * inventory_risk_score
        ) * 100 AS management_intervention_score

    FROM normalized
),


ranked AS (

    SELECT

        *,

        RANK() OVER (
            ORDER BY
                management_intervention_score DESC
        ) AS management_priority_rank

    FROM scored
)

SELECT

    dark_store_id,
    dark_store_name,
    city,
    state,

    total_orders,
    completed_orders,

    ROUND(
        completed_gmv,
        2
    ) AS completed_gmv,

    ROUND(
        sla_on_time_pct,
        2
    ) AS sla_on_time_pct,

    ROUND(
        avg_delivery_minutes,
        2
    ) AS avg_delivery_minutes,

    ROUND(
        cancellation_rate_pct,
        2
    ) AS cancellation_rate_pct,

    ROUND(
        avg_customer_rating,
        2
    ) AS avg_customer_rating,

    inventory_positions,
    reorder_positions,
    high_risk_inventory_positions,
    replenishment_units,

    ROUND(
        avg_stockout_probability_pct,
        2
    ) AS avg_stockout_probability_pct,

    ROUND(
        commercial_exposure_score * 100,
        2
    ) AS commercial_exposure_score,

    ROUND(
        sla_risk_score * 100,
        2
    ) AS sla_risk_score,

    ROUND(
        delivery_risk_score * 100,
        2
    ) AS delivery_risk_score,

    ROUND(
        cancellation_risk_score * 100,
        2
    ) AS cancellation_risk_score,

    ROUND(
        inventory_risk_score * 100,
        2
    ) AS inventory_risk_score,

    ROUND(
        management_intervention_score,
        2
    ) AS management_intervention_score,

    management_priority_rank,

    CASE

        WHEN management_intervention_score >= 70
            THEN 'Immediate Review'

        WHEN management_intervention_score >= 55
            THEN 'High Attention'

        WHEN management_intervention_score >= 40
            THEN 'Monitor Closely'

        ELSE 'Standard Monitoring'

    END AS management_action

FROM ranked

ORDER BY
    management_priority_rank
;


-- ----------------------------------------------------------
-- 2. CITY MANAGEMENT SUMMARY
-- ----------------------------------------------------------

WITH city_operations AS (

    SELECT

        city,
        state,

        COUNT(*) AS total_orders,

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        ) AS completed_orders,

        SUM(
            CASE
                WHEN completed_sales_eligible = TRUE
                THEN gross_merchandise_value
                ELSE 0
            END
        ) AS completed_gmv,

        AVG(
            CASE
                WHEN sla_eligible = TRUE
                THEN delivery_time_mins
            END
        ) AS avg_delivery_minutes,

        100.0 *
        COUNT(*) FILTER (
            WHERE
                sla_eligible = TRUE
                AND derived_sla_status = 'On Time'
        )
        /
        NULLIF(
            COUNT(*) FILTER (
                WHERE sla_eligible = TRUE
            ),
            0
        ) AS sla_on_time_pct

    FROM fact_order

    GROUP BY
        city,
        state
),

city_inventory AS (

    SELECT

        d.city,
        d.state,

        SUM(
            i.recommended_replenishment_units
        ) AS replenishment_units,

        COUNT(*) FILTER (
            WHERE i.reorder_required = TRUE
        ) AS reorder_positions,

        AVG(
            i.stockout_probability_pct
        ) AS avg_stockout_probability_pct

    FROM inventory_intelligence i

    INNER JOIN dim_dark_store d
        ON i.dark_store_id =
           d.dark_store_id

    GROUP BY
        d.city,
        d.state
)

SELECT

    o.city,
    o.state,
    o.total_orders,
    o.completed_orders,

    ROUND(
        o.completed_gmv,
        2
    ) AS completed_gmv,

    ROUND(
        o.avg_delivery_minutes,
        2
    ) AS avg_delivery_minutes,

    ROUND(
        o.sla_on_time_pct,
        2
    ) AS sla_on_time_pct,

    i.reorder_positions,
    i.replenishment_units,

    ROUND(
        i.avg_stockout_probability_pct,
        2
    ) AS avg_stockout_probability_pct

FROM city_operations o

LEFT JOIN city_inventory i
    ON o.city = i.city
    AND o.state = i.state

ORDER BY
    o.completed_gmv DESC
;
"""

sql_files[
    "07_management_priority.sql"
] = sql_07

(
    SQL_DIR /
    "07_management_priority.sql"
).write_text(
    sql_07,
    encoding="utf-8"
)

print(
    "Created:",
    SQL_DIR /
    "07_management_priority.sql"
)

Created: /content/quick_commerce_intelligence/sql/07_management_priority.sql


In [58]:
# ============================================================
# 47. FINAL SQL ASSET INVENTORY — STAGE 1
# ============================================================

generated_sql_files = sorted(
    SQL_DIR.glob("*.sql")
)

print("=" * 110)
print(" SQL PORTFOLIO ASSETS")
print("=" * 110)

sql_asset_records = []

for sql_file in generated_sql_files:

    file_size_kb = (
        sql_file.stat().st_size /
        1024
    )

    line_count = len(
        sql_file.read_text(
            encoding="utf-8"
        ).splitlines()
    )

    sql_asset_records.append(
        {
            "sql_file": sql_file.name,
            "lines": line_count,
            "size_kb": round(
                file_size_kb,
                2
            )
        }
    )


sql_asset_manifest = pd.DataFrame(
    sql_asset_records
)

display(
    sql_asset_manifest
)

print(
    f"\nSQL files created: "
    f"{len(sql_asset_manifest)}"
)


expected_sql_files = {
    "01_schema_and_quality.sql",
    "02_commercial_performance.sql",
    "03_dark_store_operations.sql",
    "04_inventory_intelligence.sql",
    "05_fulfilment_sla.sql",
    "06_customer_experience.sql",
    "07_management_priority.sql"
}


actual_sql_files = set(
    sql_asset_manifest[
        "sql_file"
    ]
)


if expected_sql_files.issubset(
    actual_sql_files
):

    print(
        "SQL asset validation: PASS ✅"
    )

else:

    print(
        "SQL asset validation: REVIEW ⚠️"
    )

    print(
        "Missing:",
        expected_sql_files -
        actual_sql_files
    )

 SQL PORTFOLIO ASSETS


,sql_file,lines,size_kb
0,01_schema_and_quality.sql,192,3.30
1,02_commercial_performance.sql,239,4.07
2,03_dark_store_operations.sql,281,5.09
3,04_inventory_intelligence.sql,326,6.38
4,05_fulfilment_sla.sql,425,7.54
5,06_customer_experience.sql,352,6.31
6,07_management_priority.sql,442,8.38



SQL files created: 7
SQL asset validation: PASS ✅


In [59]:
# ============================================================
# 48. CUSTOMER EXPERIENCE SQL EXECUTION TEST
# ============================================================

cx_sql_test = con.execute(
    """
    SELECT

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        ) AS delivered_commercial_orders,

        COUNT(*) FILTER (
            WHERE cx_eligible = TRUE
        ) AS cx_eligible_orders,

        ROUND(
            100.0 *
            COUNT(*) FILTER (
                WHERE cx_eligible = TRUE
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE completed_sales_eligible = TRUE
                ),
                0
            ),
            2
        ) AS rating_coverage_pct,

        ROUND(
            AVG(
                CASE
                    WHEN cx_eligible = TRUE
                    THEN avg_rating
                END
            ),
            2
        ) AS avg_customer_rating

    FROM fact_order
    """
).df()


print("=" * 110)
print(" CUSTOMER EXPERIENCE SQL EXECUTION TEST")
print("=" * 110)

display(
    cx_sql_test
)


cx_store_test = con.execute(
    """
    SELECT

        dark_store_id,
        dark_store_name,

        COUNT(*) FILTER (
            WHERE cx_eligible = TRUE
        ) AS rated_orders,

        ROUND(
            AVG(
                CASE
                    WHEN cx_eligible = TRUE
                    THEN avg_rating
                END
            ),
            2
        ) AS avg_rating,

        RANK() OVER (
            ORDER BY
                AVG(
                    CASE
                        WHEN cx_eligible = TRUE
                        THEN avg_rating
                    END
                ) DESC
        ) AS rating_rank

    FROM fact_order

    GROUP BY
        dark_store_id,
        dark_store_name

    ORDER BY
        rating_rank

    LIMIT 10
    """
).df()


print("\nTop dark stores by customer rating:")

display(
    cx_store_test
)


print(
    "\nCustomer Experience SQL execution: PASS ✅"
)

 CUSTOMER EXPERIENCE SQL EXECUTION TEST


,delivered_commercial_orders,cx_eligible_orders,rating_coverage_pct,avg_customer_rating
0,14333,9826,68.56,3.88



Top dark stores by customer rating:


,dark_store_id,dark_store_name,rated_orders,avg_rating,rating_rank
0,DS_05_02,Pune Dark Store 02,475,3.97,1
1,DS_08_01,Bengaluru Dark Store 01,504,3.96,2
2,DS_02_02,Delhi Dark Store 02,493,3.94,3
3,DS_09_01,Lucknow Dark Store 01,472,3.93,4
4,DS_06_02,Hyderabad Dark Store 02,482,3.92,5
5,DS_03_01,Ahmedabad Dark Store 01,525,3.92,6
6,DS_04_01,Chennai Dark Store 01,523,3.91,7
7,DS_01_01,Kolkata Dark Store 01,492,3.91,8
8,DS_04_02,Chennai Dark Store 02,514,3.89,9
9,DS_07_01,Jaipur Dark Store 01,464,3.88,10



Customer Experience SQL execution: PASS ✅


In [60]:
# ============================================================
# 49. MANAGEMENT PRIORITY SQL EXECUTION TEST
# ============================================================

management_sql_test = con.execute(
    """
    WITH store_operations AS (

        SELECT

            dark_store_id,
            dark_store_name,
            city,

            SUM(
                CASE
                    WHEN completed_sales_eligible = TRUE
                    THEN gross_merchandise_value
                    ELSE 0
                END
            ) AS completed_gmv,

            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END
            ) AS avg_delivery_minutes,

            100.0 *
            COUNT(*) FILTER (
                WHERE
                    sla_eligible = TRUE
                    AND
                    derived_sla_status = 'On Time'
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE sla_eligible = TRUE
                ),
                0
            ) AS sla_on_time_pct,

            100.0 *
            COUNT(*) FILTER (
                WHERE order_status = 'Cancelled'
            )
            /
            NULLIF(
                COUNT(*),
                0
            ) AS cancellation_rate_pct

        FROM fact_order

        GROUP BY
            dark_store_id,
            dark_store_name,
            city
    ),


    store_inventory AS (

        SELECT

            dark_store_id,

            COUNT(*) FILTER (
                WHERE reorder_required = TRUE
            ) AS reorder_positions,

            SUM(
                recommended_replenishment_units
            ) AS replenishment_units,

            AVG(
                stockout_probability_pct
            ) AS avg_stockout_probability_pct

        FROM inventory_intelligence

        GROUP BY
            dark_store_id
    ),


    combined AS (

        SELECT
            o.*,
            i.reorder_positions,
            i.replenishment_units,
            i.avg_stockout_probability_pct

        FROM store_operations o

        LEFT JOIN store_inventory i
            ON o.dark_store_id =
               i.dark_store_id
    ),


    normalized AS (

        SELECT

            *,

            PERCENT_RANK() OVER (
                ORDER BY completed_gmv ASC
            ) AS commercial_score,

            PERCENT_RANK() OVER (
                ORDER BY sla_on_time_pct DESC
            ) AS sla_risk_score,

            PERCENT_RANK() OVER (
                ORDER BY avg_delivery_minutes ASC
            ) AS delivery_risk_score,

            PERCENT_RANK() OVER (
                ORDER BY cancellation_rate_pct ASC
            ) AS cancellation_risk_score,

            PERCENT_RANK() OVER (
                ORDER BY avg_stockout_probability_pct ASC
            ) AS inventory_risk_score

        FROM combined
    ),


    scored AS (

        SELECT

            *,

            (
                0.20 * commercial_score
                +
                0.25 * sla_risk_score
                +
                0.15 * delivery_risk_score
                +
                0.10 * cancellation_risk_score
                +
                0.30 * inventory_risk_score
            ) * 100
                AS intervention_score

        FROM normalized
    )

    SELECT

        dark_store_id,
        dark_store_name,
        city,

        ROUND(
            completed_gmv,
            2
        ) AS completed_gmv,

        ROUND(
            sla_on_time_pct,
            2
        ) AS sla_on_time_pct,

        ROUND(
            avg_delivery_minutes,
            2
        ) AS avg_delivery_minutes,

        reorder_positions,
        replenishment_units,

        ROUND(
            avg_stockout_probability_pct,
            2
        ) AS avg_stockout_probability_pct,

        ROUND(
            intervention_score,
            2
        ) AS management_intervention_score,

        RANK() OVER (
            ORDER BY
                intervention_score DESC
        ) AS priority_rank

    FROM scored

    ORDER BY
        priority_rank

    LIMIT 10
    """
).df()


print("=" * 112)
print(" MANAGEMENT PRIORITY SQL EXECUTION TEST")
print("=" * 112)

display(
    management_sql_test
)

print(
    "\nManagement Priority SQL execution: PASS ✅"
)

 MANAGEMENT PRIORITY SQL EXECUTION TEST


,dark_store_id,dark_store_name,city,completed_gmv,sla_on_time_pct,avg_delivery_minutes,reorder_positions,replenishment_units,avg_stockout_probability_pct,management_intervention_score,priority_rank
0,DS_05_02,Pune Dark Store 02,Pune,"1,365,340.00",41.88,42.31,7,53.00,16.81,76.05,1
1,DS_03_01,Ahmedabad Dark Store 01,Ahmedabad,"1,450,020.00",39.13,42.10,6,42.00,13.44,72.89,2
2,DS_01_02,Kolkata Dark Store 02,Kolkata,"1,446,140.00",40.63,41.93,7,46.00,12.19,68.95,3
3,DS_07_02,Jaipur Dark Store 02,Jaipur,"1,502,010.00",43.08,40.71,7,51.00,14.32,66.32,4
4,DS_04_02,Chennai Dark Store 02,Chennai,"1,430,920.00",42.63,41.81,7,51.00,15.69,63.95,5
5,DS_02_02,Delhi Dark Store 02,Delhi,"1,534,750.00",42.44,41.81,8,46.00,10.47,60.79,6
6,DS_09_01,Lucknow Dark Store 01,Lucknow,"1,341,510.00",42.24,41.15,8,51.00,14.49,60.53,7
7,DS_02_01,Delhi Dark Store 01,Delhi,"1,474,580.00",43.36,40.38,8,54.00,15.07,58.68,8
8,DS_03_02,Ahmedabad Dark Store 02,Ahmedabad,"1,439,020.00",38.72,42.09,5,30.00,7.59,57.89,9
9,DS_06_01,Hyderabad Dark Store 01,Hyderabad,"1,500,460.00",41.34,42.20,5,29.00,4.65,57.11,10



Management Priority SQL execution: PASS ✅


## 50. Profitability & Unit Economics Assumptions

The source dataset contains observed order revenue, product category, quantity, delivery distance and fulfilment outcomes, but does not contain actual procurement costs, supplier margins, rider costs, warehouse labour costs, payment-processing charges or corporate overhead.

Therefore, profitability is modelled as a transparent **contribution-margin scenario**, not reported company profit.

Observed inputs:
- Completed-order revenue
- Product category
- Units sold
- Delivery distance
- Delivery time
- SLA outcome

Modelled assumptions:
- Category gross-margin rates
- Picking and packing cost per completed order
- Packaging cost per completed order
- Last-mile base delivery cost
- Distance-based delivery cost
- Payment-processing fee

The resulting metric is labelled **Modelled Contribution Margin** and must not be interpreted as actual Zepto profitability.

In [61]:
# ============================================================
# 50. PROFITABILITY & UNIT ECONOMICS ASSUMPTIONS
# ============================================================


# ------------------------------------------------------------
# Category gross-margin scenario assumptions
#
# These are analytical portfolio assumptions only.
# They are NOT actual Zepto procurement or margin data.
# ------------------------------------------------------------

profitability_category_assumptions = pd.DataFrame([
    {
        "category": "Fruits & Vegetables",
        "gross_margin_rate": 0.24
    },
    {
        "category": "Dairy & Bakery",
        "gross_margin_rate": 0.20
    },
    {
        "category": "Beverages",
        "gross_margin_rate": 0.18
    },
    {
        "category": "Snacks & Branded Foods",
        "gross_margin_rate": 0.21
    },
    {
        "category": "Grocery & Staples",
        "gross_margin_rate": 0.16
    },
    {
        "category": "Baby Care",
        "gross_margin_rate": 0.18
    },
    {
        "category": "Personal Care",
        "gross_margin_rate": 0.25
    },
    {
        "category": "Household",
        "gross_margin_rate": 0.22
    }
])


# ------------------------------------------------------------
# Order-level variable fulfilment assumptions
# ------------------------------------------------------------

PICK_PACK_COST_PER_ORDER = 20.00
PACKAGING_COST_PER_ORDER = 8.00

LAST_MILE_BASE_COST = 25.00
LAST_MILE_COST_PER_KM = 4.00

PAYMENT_PROCESSING_RATE = 0.012


print("=" * 110)
print(" PROFITABILITY & UNIT ECONOMICS ASSUMPTIONS")
print("=" * 110)

print("\nCategory gross-margin assumptions:")

display(
    profitability_category_assumptions.assign(
        gross_margin_pct=lambda x:
            x["gross_margin_rate"] * 100
    )
)


print("\nOrder-level fulfilment assumptions:")

print(
    f"Picking & packing cost/order : "
    f"₹{PICK_PACK_COST_PER_ORDER:,.2f}"
)

print(
    f"Packaging cost/order         : "
    f"₹{PACKAGING_COST_PER_ORDER:,.2f}"
)

print(
    f"Last-mile base cost          : "
    f"₹{LAST_MILE_BASE_COST:,.2f}"
)

print(
    f"Last-mile cost/km            : "
    f"₹{LAST_MILE_COST_PER_KM:,.2f}"
)

print(
    f"Payment processing rate      : "
    f"{PAYMENT_PROCESSING_RATE * 100:.2f}%"
)


# ------------------------------------------------------------
# Category validation
# ------------------------------------------------------------

source_categories = set(
    dim_product["category"]
    .dropna()
    .unique()
)

profitability_categories = set(
    profitability_category_assumptions[
        "category"
    ]
)


missing_profit_assumptions = (
    source_categories
    -
    profitability_categories
)

unused_profit_assumptions = (
    profitability_categories
    -
    source_categories
)


print("\nValidation:")

print(
    "Source categories               :",
    len(source_categories)
)

print(
    "Categories with margin assumptions:",
    len(profitability_categories)
)

print(
    "Missing margin assumptions      :",
    missing_profit_assumptions
)

print(
    "Unused margin assumptions       :",
    unused_profit_assumptions
)


if (
    len(missing_profit_assumptions) == 0
    and
    len(unused_profit_assumptions) == 0
):

    print(
        "\nProfitability assumption validation: PASS ✅"
    )

else:

    print(
        "\nProfitability assumption validation: REVIEW ⚠️"
    )

 PROFITABILITY & UNIT ECONOMICS ASSUMPTIONS

Category gross-margin assumptions:


,category,gross_margin_rate,gross_margin_pct
0,Fruits & Vegetables,0.24,24.00
1,Dairy & Bakery,0.20,20.00
2,Beverages,0.18,18.00
3,Snacks & Branded Foods,0.21,21.00
4,Grocery & Staples,0.16,16.00
5,Baby Care,0.18,18.00
6,Personal Care,0.25,25.00
7,Household,0.22,22.00



Order-level fulfilment assumptions:
Picking & packing cost/order : ₹20.00
Packaging cost/order         : ₹8.00
Last-mile base cost          : ₹25.00
Last-mile cost/km            : ₹4.00
Payment processing rate      : 1.20%

Validation:
Source categories               : 8
Categories with margin assumptions: 8
Missing margin assumptions      : set()
Unused margin assumptions       : set()

Profitability assumption validation: PASS ✅


## 51. Merchandise Gross-Profit Model

Category-specific gross-margin assumptions are applied at transaction-line level.

For each completed order item:

**Modelled COGS**

Revenue × (1 − Gross Margin Rate)

**Modelled Merchandise Gross Profit**

Revenue × Gross Margin Rate

The line-level calculation is then aggregated to order level, allowing mixed-category baskets to retain their appropriate category economics.

In [62]:
# ============================================================
# 51. MERCHANDISE GROSS-PROFIT MODEL
# ============================================================


# ------------------------------------------------------------
# Completed order-item base
# ------------------------------------------------------------

profit_order_items = (
    fact_order_item[
        fact_order_item[
            "completed_sales_eligible"
        ] == True
    ]
    .copy()
)


# ------------------------------------------------------------
# Safe rerun
# ------------------------------------------------------------

profit_order_items = (
    profit_order_items
    .drop(
        columns=[
            "gross_margin_rate",
            "modelled_cogs",
            "modelled_merchandise_gross_profit"
        ],
        errors="ignore"
    )
)


# ------------------------------------------------------------
# Add category margin assumptions
# ------------------------------------------------------------

profit_order_items = (
    profit_order_items
    .merge(
        profitability_category_assumptions,
        on="category",
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# Economics
# ------------------------------------------------------------

profit_order_items[
    "modelled_cogs"
] = (
    profit_order_items[
        "line_revenue"
    ]
    *
    (
        1
        -
        profit_order_items[
            "gross_margin_rate"
        ]
    )
)


profit_order_items[
    "modelled_merchandise_gross_profit"
] = (
    profit_order_items[
        "line_revenue"
    ]
    *
    profit_order_items[
        "gross_margin_rate"
    ]
)


# ------------------------------------------------------------
# Order-level aggregation
# ------------------------------------------------------------

order_merchandise_economics = (
    profit_order_items
    .groupby(
        "order_id",
        as_index=False
    )
    .agg(
        merchandise_revenue=(
            "line_revenue",
            "sum"
        ),

        modelled_cogs=(
            "modelled_cogs",
            "sum"
        ),

        modelled_merchandise_gross_profit=(
            "modelled_merchandise_gross_profit",
            "sum"
        ),

        units=(
            "quantity",
            "sum"
        ),

        categories_in_basket=(
            "category",
            "nunique"
        )
    )
)


order_merchandise_economics[
    "effective_gross_margin_pct"
] = (
    100
    *
    order_merchandise_economics[
        "modelled_merchandise_gross_profit"
    ]
    /
    order_merchandise_economics[
        "merchandise_revenue"
    ]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

missing_margin_rows = int(
    profit_order_items[
        "gross_margin_rate"
    ]
    .isna()
    .sum()
)


revenue_reconciliation = (
    order_merchandise_economics[
        "merchandise_revenue"
    ].sum()
)


source_completed_gmv = (
    fact_order.loc[
        fact_order[
            "completed_sales_eligible"
        ] == True,
        "gross_merchandise_value"
    ]
    .sum()
)


revenue_difference = (
    revenue_reconciliation
    -
    source_completed_gmv
)


print("=" * 112)
print(" MERCHANDISE GROSS-PROFIT MODEL")
print("=" * 112)

print(
    f"Completed transaction lines     : "
    f"{len(profit_order_items):,}"
)

print(
    f"Completed orders modelled        : "
    f"{len(order_merchandise_economics):,}"
)

print(
    f"Missing margin assumptions       : "
    f"{missing_margin_rows:,}"
)

print(
    f"Transaction-line revenue         : "
    f"₹{revenue_reconciliation:,.2f}"
)

print(
    f"Canonical completed GMV          : "
    f"₹{source_completed_gmv:,.2f}"
)

print(
    f"Revenue reconciliation difference: "
    f"₹{revenue_difference:,.2f}"
)


if (
    missing_margin_rows == 0
    and
    abs(revenue_difference) < 0.01
):

    print(
        "\nMerchandise economics validation: PASS ✅"
    )

else:

    print(
        "\nMerchandise economics validation: REVIEW ⚠️"
    )


print("\nNetwork merchandise economics:")

network_merchandise_summary = pd.DataFrame({
    "metric": [
        "Completed GMV",
        "Modelled COGS",
        "Modelled Merchandise Gross Profit",
        "Effective Gross Margin %"
    ],

    "value": [
        order_merchandise_economics[
            "merchandise_revenue"
        ].sum(),

        order_merchandise_economics[
            "modelled_cogs"
        ].sum(),

        order_merchandise_economics[
            "modelled_merchandise_gross_profit"
        ].sum(),

        order_merchandise_economics[
            "modelled_merchandise_gross_profit"
        ].sum()
        /
        order_merchandise_economics[
            "merchandise_revenue"
        ].sum()
        * 100
    ]
})

display(
    network_merchandise_summary
)

 MERCHANDISE GROSS-PROFIT MODEL
Completed transaction lines     : 38,986
Completed orders modelled        : 14,333
Missing margin assumptions       : 0
Transaction-line revenue         : ₹28,224,720.00
Canonical completed GMV          : ₹28,224,720.00
Revenue reconciliation difference: ₹0.00

Merchandise economics validation: PASS ✅

Network merchandise economics:


,metric,value
0,Completed GMV,"28,224,720.00"
1,Modelled COGS,"22,524,757.40"
2,Modelled Merchandise Gross Profit,"5,699,962.60"
3,Effective Gross Margin %,20.19


## 52. Order-Level Cost-to-Serve & Contribution Margin

For each completed commercial order, modelled merchandise gross profit is combined with scenario-based fulfilment costs.

**Last-Mile Cost**

Base Delivery Cost + (Distance × Cost per km)

**Payment Processing Cost**

Order Revenue × Payment Processing Rate

**Variable Cost-to-Serve**

Picking & Packing + Packaging + Last-Mile + Payment Processing

**Modelled Contribution Margin**

Merchandise Gross Profit − Variable Cost-to-Serve

**Contribution Margin %**

Contribution Margin / Order Revenue

This is a contribution-margin model before fixed corporate overhead and does not represent reported company net profit.

In [63]:
# ============================================================
# 52. ORDER-LEVEL COST-TO-SERVE & CONTRIBUTION MARGIN
# ============================================================


# ------------------------------------------------------------
# Completed canonical orders
# ------------------------------------------------------------

profitability_orders = (
    fact_order[
        fact_order[
            "completed_sales_eligible"
        ] == True
    ]
    .copy()
)


# ------------------------------------------------------------
# Merge merchandise economics
# ------------------------------------------------------------

profitability_orders = (
    profitability_orders
    .merge(
        order_merchandise_economics,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Cost-to-serve components
# ------------------------------------------------------------

profitability_orders[
    "pick_pack_cost"
] = PICK_PACK_COST_PER_ORDER


profitability_orders[
    "packaging_cost"
] = PACKAGING_COST_PER_ORDER


profitability_orders[
    "modelled_last_mile_cost"
] = (
    LAST_MILE_BASE_COST
    +
    (
        profitability_orders[
            "distance_km"
        ]
        *
        LAST_MILE_COST_PER_KM
    )
)


profitability_orders[
    "payment_processing_cost"
] = (
    profitability_orders[
        "merchandise_revenue"
    ]
    *
    PAYMENT_PROCESSING_RATE
)


profitability_orders[
    "modelled_variable_cost_to_serve"
] = (
    profitability_orders[
        "pick_pack_cost"
    ]
    +
    profitability_orders[
        "packaging_cost"
    ]
    +
    profitability_orders[
        "modelled_last_mile_cost"
    ]
    +
    profitability_orders[
        "payment_processing_cost"
    ]
)


# ------------------------------------------------------------
# Contribution economics
# ------------------------------------------------------------

profitability_orders[
    "modelled_contribution_margin"
] = (
    profitability_orders[
        "modelled_merchandise_gross_profit"
    ]
    -
    profitability_orders[
        "modelled_variable_cost_to_serve"
    ]
)


profitability_orders[
    "contribution_margin_pct"
] = (
    100
    *
    profitability_orders[
        "modelled_contribution_margin"
    ]
    /
    profitability_orders[
        "merchandise_revenue"
    ]
)


profitability_orders[
    "cost_to_serve_pct"
] = (
    100
    *
    profitability_orders[
        "modelled_variable_cost_to_serve"
    ]
    /
    profitability_orders[
        "merchandise_revenue"
    ]
)


# ------------------------------------------------------------
# Contribution classification
# ------------------------------------------------------------

profitability_orders[
    "profitability_band"
] = pd.cut(
    profitability_orders[
        "contribution_margin_pct"
    ],

    bins=[
        -np.inf,
        0,
        10,
        15,
        np.inf
    ],

    labels=[
        "Negative Contribution",
        "Low Margin",
        "Moderate Margin",
        "Strong Margin"
    ]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

missing_economics = int(
    profitability_orders[
        [
            "merchandise_revenue",
            "modelled_merchandise_gross_profit",
            "modelled_variable_cost_to_serve",
            "modelled_contribution_margin"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)


reconciled_profitability_revenue = (
    profitability_orders[
        "merchandise_revenue"
    ].sum()
)


profitability_revenue_difference = (
    reconciled_profitability_revenue
    -
    source_completed_gmv
)


print("=" * 116)
print(" ORDER-LEVEL UNIT ECONOMICS")
print("=" * 116)

print(
    f"Completed orders modelled      : "
    f"{len(profitability_orders):,}"
)

print(
    f"Missing economics rows         : "
    f"{missing_economics:,}"
)

print(
    f"Revenue reconciliation gap     : "
    f"₹{profitability_revenue_difference:,.2f}"
)


if (
    missing_economics == 0
    and
    abs(
        profitability_revenue_difference
    ) < 0.01
):

    print(
        "\nUnit-economics validation: PASS ✅"
    )

else:

    print(
        "\nUnit-economics validation: REVIEW ⚠️"
    )


# ------------------------------------------------------------
# Network unit economics
# ------------------------------------------------------------

network_profitability = pd.DataFrame({
    "metric": [

        "Completed GMV",

        "Modelled Merchandise Gross Profit",

        "Modelled Variable Cost-to-Serve",

        "Modelled Contribution Margin",

        "Contribution Margin %",

        "Average Contribution / Order",

        "Average Cost-to-Serve / Order"
    ],

    "value": [

        profitability_orders[
            "merchandise_revenue"
        ].sum(),

        profitability_orders[
            "modelled_merchandise_gross_profit"
        ].sum(),

        profitability_orders[
            "modelled_variable_cost_to_serve"
        ].sum(),

        profitability_orders[
            "modelled_contribution_margin"
        ].sum(),

        profitability_orders[
            "modelled_contribution_margin"
        ].sum()
        /
        profitability_orders[
            "merchandise_revenue"
        ].sum()
        * 100,

        profitability_orders[
            "modelled_contribution_margin"
        ].mean(),

        profitability_orders[
            "modelled_variable_cost_to_serve"
        ].mean()
    ]
})


print("\nNetwork unit economics:")

display(
    network_profitability
)


# ------------------------------------------------------------
# Profitability-band distribution
# ------------------------------------------------------------

profitability_band_summary = (
    profitability_orders
    .groupby(
        "profitability_band",
        observed=True
    )
    .agg(
        orders=(
            "order_id",
            "count"
        ),

        revenue=(
            "merchandise_revenue",
            "sum"
        ),

        contribution_margin=(
            "modelled_contribution_margin",
            "sum"
        )
    )
    .reset_index()
)


profitability_band_summary[
    "order_share_pct"
] = (
    profitability_band_summary[
        "orders"
    ]
    /
    profitability_band_summary[
        "orders"
    ].sum()
    * 100
)


print(
    "\nProfitability-band distribution:"
)

display(
    profitability_band_summary
)

 ORDER-LEVEL UNIT ECONOMICS
Completed orders modelled      : 14,333
Missing economics rows         : 0
Revenue reconciliation gap     : ₹0.00

Unit-economics validation: PASS ✅

Network unit economics:


,metric,value
0,Completed GMV,"28,224,720.00"
1,Modelled Merchandise Gross Profit,"5,699,962.60"
2,Modelled Variable Cost-to-Serve,"1,398,378.04"
3,Modelled Contribution Margin,"4,301,584.56"
4,Contribution Margin %,15.24
5,Average Contribution / Order,300.12
6,Average Cost-to-Serve / Order,97.56



Profitability-band distribution:


,profitability_band,orders,revenue,contribution_margin,order_share_pct
0,Negative Contribution,1478,340180,"-48,010.66",10.31
1,Low Margin,2188,1509380,"104,721.84",15.27
2,Moderate Margin,4818,8737740,"1,167,467.02",33.61
3,Strong Margin,5849,17637420,"3,077,406.36",40.81


## 53. Dark Store & Category Profitability

The order-level contribution model is aggregated across operational dimensions to identify where commercial scale does — and does not — translate into attractive unit economics.

The analysis evaluates:

- Dark store contribution margin
- Contribution margin percentage
- Average contribution per completed order
- Negative-contribution order exposure
- Category-level gross margin and contribution economics

All profitability values remain scenario-based contribution metrics rather than actual reported company profit.

In [64]:
# ============================================================
# 53. DARK STORE & CATEGORY PROFITABILITY
# ============================================================


# ------------------------------------------------------------
# DARK STORE PROFITABILITY
# ------------------------------------------------------------

dark_store_profitability = (
    profitability_orders
    .groupby(
        [
            "dark_store_id",
            "dark_store_name",
            "city",
            "state"
        ],
        as_index=False
    )
    .agg(
        completed_orders=(
            "order_id",
            "count"
        ),

        completed_gmv=(
            "merchandise_revenue",
            "sum"
        ),

        merchandise_gross_profit=(
            "modelled_merchandise_gross_profit",
            "sum"
        ),

        variable_cost_to_serve=(
            "modelled_variable_cost_to_serve",
            "sum"
        ),

        contribution_margin=(
            "modelled_contribution_margin",
            "sum"
        ),

        avg_delivery_distance_km=(
            "distance_km",
            "mean"
        ),

        avg_delivery_minutes=(
            "delivery_time_mins",
            "mean"
        )
    )
)


dark_store_profitability[
    "contribution_margin_pct"
] = (
    100
    *
    dark_store_profitability[
        "contribution_margin"
    ]
    /
    dark_store_profitability[
        "completed_gmv"
    ]
)


dark_store_profitability[
    "avg_contribution_per_order"
] = (
    dark_store_profitability[
        "contribution_margin"
    ]
    /
    dark_store_profitability[
        "completed_orders"
    ]
)


# ------------------------------------------------------------
# Negative-contribution exposure
# ------------------------------------------------------------

negative_store_orders = (
    profitability_orders[
        profitability_orders[
            "modelled_contribution_margin"
        ] < 0
    ]
    .groupby(
        "dark_store_id",
        as_index=False
    )
    .agg(
        negative_contribution_orders=(
            "order_id",
            "count"
        ),

        negative_contribution_revenue=(
            "merchandise_revenue",
            "sum"
        ),

        negative_contribution_loss=(
            "modelled_contribution_margin",
            "sum"
        )
    )
)


dark_store_profitability = (
    dark_store_profitability
    .merge(
        negative_store_orders,
        on="dark_store_id",
        how="left",
        validate="one_to_one"
    )
)


dark_store_profitability[
    [
        "negative_contribution_orders",
        "negative_contribution_revenue",
        "negative_contribution_loss"
    ]
] = (
    dark_store_profitability[
        [
            "negative_contribution_orders",
            "negative_contribution_revenue",
            "negative_contribution_loss"
        ]
    ]
    .fillna(0)
)


dark_store_profitability[
    "negative_order_share_pct"
] = (
    100
    *
    dark_store_profitability[
        "negative_contribution_orders"
    ]
    /
    dark_store_profitability[
        "completed_orders"
    ]
)


# ------------------------------------------------------------
# Store ranking
# ------------------------------------------------------------

dark_store_profitability[
    "contribution_rank"
] = (
    dark_store_profitability[
        "contribution_margin"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


dark_store_profitability[
    "margin_pct_rank"
] = (
    dark_store_profitability[
        "contribution_margin_pct"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


print("=" * 118)
print(" DARK STORE PROFITABILITY")
print("=" * 118)

display(
    dark_store_profitability[
        [
            "dark_store_id",
            "dark_store_name",
            "city",
            "completed_orders",
            "completed_gmv",
            "variable_cost_to_serve",
            "contribution_margin",
            "contribution_margin_pct",
            "avg_contribution_per_order",
            "negative_contribution_orders",
            "negative_order_share_pct",
            "contribution_rank",
            "margin_pct_rank"
        ]
    ]
    .sort_values(
        "contribution_margin",
        ascending=False
    )
)


# ------------------------------------------------------------
# CATEGORY PROFITABILITY
# ------------------------------------------------------------

category_profitability = (
    profit_order_items
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        completed_orders=(
            "order_id",
            "nunique"
        ),

        units_sold=(
            "quantity",
            "sum"
        ),

        revenue=(
            "line_revenue",
            "sum"
        ),

        modelled_cogs=(
            "modelled_cogs",
            "sum"
        ),

        merchandise_gross_profit=(
            "modelled_merchandise_gross_profit",
            "sum"
        )
    )
)


category_profitability[
    "gross_margin_pct"
] = (
    100
    *
    category_profitability[
        "merchandise_gross_profit"
    ]
    /
    category_profitability[
        "revenue"
    ]
)


category_profitability[
    "revenue_share_pct"
] = (
    100
    *
    category_profitability[
        "revenue"
    ]
    /
    category_profitability[
        "revenue"
    ].sum()
)


print("\n" + "=" * 118)
print(" CATEGORY MERCHANDISE PROFITABILITY")
print("=" * 118)

display(
    category_profitability
    .sort_values(
        "merchandise_gross_profit",
        ascending=False
    )
)

 DARK STORE PROFITABILITY


,dark_store_id,dark_store_name,city,completed_orders,completed_gmv,variable_cost_to_serve,contribution_margin,contribution_margin_pct,avg_contribution_per_order,negative_contribution_orders,negative_order_share_pct,contribution_rank,margin_pct_rank
3,DS_02_02,Delhi Dark Store 02,Delhi,760,1534750,"74,615.00","235,310.30",15.33,309.62,67,8.82,1,6
8,DS_05_01,Pune Dark Store 01,Pune,738,1504950,"72,623.80","231,031.60",15.35,313.05,78,10.57,2,4
13,DS_07_02,Jaipur Dark Store 02,Jaipur,755,1502010,"73,562.32","227,657.08",15.16,301.53,75,9.93,3,14
10,DS_06_01,Hyderabad Dark Store 01,Hyderabad,762,1500460,"74,367.12","227,382.98",15.15,298.40,74,9.71,4,15
2,DS_02_01,Delhi Dark Store 01,Delhi,737,1474580,"72,183.56","226,130.44",15.34,306.83,83,11.26,5,5
17,DS_09_02,Lucknow Dark Store 02,Lucknow,696,1458320,"69,392.24","224,575.56",15.40,322.67,66,9.48,6,3
1,DS_01_02,Kolkata Dark Store 02,Kolkata,760,1446140,"73,508.48","219,401.32",15.17,288.69,89,11.71,7,13
14,DS_08_01,Bengaluru Dark Store 01,Bengaluru,725,1437450,"70,771.20","219,188.70",15.25,302.33,80,11.03,8,10
7,DS_04_02,Chennai Dark Store 02,Chennai,727,1430920,"71,147.64","219,140.66",15.31,301.43,81,11.14,9,7
4,DS_03_01,Ahmedabad Dark Store 01,Ahmedabad,756,1450020,"73,667.04","219,049.26",15.11,289.75,79,10.45,10,16



 CATEGORY MERCHANDISE PROFITABILITY


,category,completed_orders,units_sold,revenue,modelled_cogs,merchandise_gross_profit,gross_margin_pct,revenue_share_pct
0,Baby Care,4401,15569,7179820,"5,887,452.40","1,292,367.60",18.00,25.44
6,Personal Care,4080,14318,4688810,"3,516,607.50","1,172,202.50",25.00,16.61
4,Grocery & Staples,4222,14986,4596430,"3,861,001.20","735,428.80",16.00,16.29
3,Fruits & Vegetables,4547,15871,2740140,"2,082,506.40","657,633.60",24.00,9.71
5,Household,3911,13668,2863030,"2,233,163.40","629,866.60",22.00,10.14
2,Dairy & Bakery,4610,16678,2657910,"2,126,328.00","531,582.00",20.00,9.42
7,Snacks & Branded Foods,3769,12947,1704570,"1,346,610.30","357,959.70",21.00,6.04
1,Beverages,3921,13431,1794010,"1,471,088.20","322,921.80",18.00,6.36


## 54. Cost-to-Serve Drivers

Contribution economics are segmented by delivery distance, SLA outcome and basket size to identify structural drivers of low-margin and negative-contribution orders.

This analysis helps distinguish:

- commercially healthy orders,
- expensive long-distance fulfilment,
- low-value baskets that cannot absorb fulfilment costs,
- and operational SLA pressure.

In [65]:
# ============================================================
# 54. DISTANCE, SLA & BASKET ECONOMICS
# ============================================================


# ------------------------------------------------------------
# Distance bands
# ------------------------------------------------------------

profitability_orders[
    "distance_band"
] = pd.cut(
    profitability_orders[
        "distance_km"
    ],
    bins=[
        -np.inf,
        3,
        5,
        7,
        np.inf
    ],
    labels=[
        "0–3 km",
        "3–5 km",
        "5–7 km",
        "7+ km"
    ]
)


distance_profitability = (
    profitability_orders
    .groupby(
        "distance_band",
        observed=True
    )
    .agg(
        orders=(
            "order_id",
            "count"
        ),

        avg_order_value=(
            "merchandise_revenue",
            "mean"
        ),

        avg_last_mile_cost=(
            "modelled_last_mile_cost",
            "mean"
        ),

        avg_cost_to_serve=(
            "modelled_variable_cost_to_serve",
            "mean"
        ),

        avg_contribution=(
            "modelled_contribution_margin",
            "mean"
        ),

        contribution_margin=(
            "modelled_contribution_margin",
            "sum"
        ),

        revenue=(
            "merchandise_revenue",
            "sum"
        )
    )
    .reset_index()
)


distance_profitability[
    "contribution_margin_pct"
] = (
    100
    *
    distance_profitability[
        "contribution_margin"
    ]
    /
    distance_profitability[
        "revenue"
    ]
)


print("=" * 116)
print(" PROFITABILITY BY DELIVERY DISTANCE")
print("=" * 116)

display(
    distance_profitability
)


# ------------------------------------------------------------
# SLA economics
# ------------------------------------------------------------

sla_profitability = (
    profitability_orders
    .groupby(
        "derived_sla_status",
        as_index=False
    )
    .agg(
        orders=(
            "order_id",
            "count"
        ),

        avg_order_value=(
            "merchandise_revenue",
            "mean"
        ),

        avg_delivery_minutes=(
            "delivery_time_mins",
            "mean"
        ),

        avg_distance_km=(
            "distance_km",
            "mean"
        ),

        avg_cost_to_serve=(
            "modelled_variable_cost_to_serve",
            "mean"
        ),

        contribution_margin=(
            "modelled_contribution_margin",
            "sum"
        ),

        revenue=(
            "merchandise_revenue",
            "sum"
        )
    )
)


sla_profitability[
    "contribution_margin_pct"
] = (
    100
    *
    sla_profitability[
        "contribution_margin"
    ]
    /
    sla_profitability[
        "revenue"
    ]
)


print("\n" + "=" * 116)
print(" PROFITABILITY BY SLA STATUS")
print("=" * 116)

display(
    sla_profitability
)


# ------------------------------------------------------------
# Basket-value quartiles
# ------------------------------------------------------------

profitability_orders[
    "basket_value_segment"
] = pd.qcut(
    profitability_orders[
        "merchandise_revenue"
    ],
    q=4,
    labels=[
        "Q1 Low Basket",
        "Q2",
        "Q3",
        "Q4 High Basket"
    ],
    duplicates="drop"
)


basket_profitability = (
    profitability_orders
    .groupby(
        "basket_value_segment",
        observed=True
    )
    .agg(
        orders=(
            "order_id",
            "count"
        ),

        avg_order_value=(
            "merchandise_revenue",
            "mean"
        ),

        avg_cost_to_serve=(
            "modelled_variable_cost_to_serve",
            "mean"
        ),

        avg_contribution=(
            "modelled_contribution_margin",
            "mean"
        ),

        contribution_margin=(
            "modelled_contribution_margin",
            "sum"
        ),

        negative_orders=(
            "modelled_contribution_margin",
            lambda x:
                int((x < 0).sum())
        )
    )
    .reset_index()
)


basket_profitability[
    "negative_order_share_pct"
] = (
    100
    *
    basket_profitability[
        "negative_orders"
    ]
    /
    basket_profitability[
        "orders"
    ]
)


print("\n" + "=" * 116)
print(" PROFITABILITY BY BASKET VALUE")
print("=" * 116)

display(
    basket_profitability
)

 PROFITABILITY BY DELIVERY DISTANCE


,distance_band,orders,avg_order_value,avg_last_mile_cost,avg_cost_to_serve,avg_contribution,contribution_margin,revenue,contribution_margin_pct
0,0–3 km,3859,"1,980.84",32.08,83.85,314.25,"1,212,685.08",7644060,15.86
1,3–5 km,3047,"1,960.18",41.24,92.76,303.88,"925,915.08",5972660,15.50
2,5–7 km,3038,"1,960.74",49.24,100.77,296.83,"901,777.24",5956730,15.14
3,7+ km,4389,"1,971.13",59.08,110.73,287.36,"1,261,207.16",8651270,14.58



 PROFITABILITY BY SLA STATUS


,derived_sla_status,orders,avg_order_value,avg_delivery_minutes,avg_distance_km,avg_cost_to_serve,contribution_margin,revenue,contribution_margin_pct
0,At Risk,3339,"1,976.21",42.82,5.69,99.48,"999,319.06",6598570,15.14
1,Late,4894,"1,964.64",65.99,7.94,108.35,"1,414,377.86",9614970,14.71
2,On Time,6100,"1,969.05",20.59,2.81,87.86,"1,887,887.64",12011180,15.72



 PROFITABILITY BY BASKET VALUE


,basket_value_segment,orders,avg_order_value,avg_cost_to_serve,avg_contribution,contribution_margin,negative_orders,negative_order_share_pct
0,Q1 Low Basket,3612,449.68,79.31,13.35,"48,229.20",1477,40.89
1,Q2,3571,"1,213.46",88.16,162.11,"578,890.66",0,0.00
2,Q3,3569,"2,149.04",100.09,336.99,"1,202,724.72",0,0.00
3,Q4 High Basket,3581,"4,076.31",122.83,690.24,"2,471,739.98",0,0.00


In [66]:
# ============================================================
# 55A. MATERIALISE PROFITABILITY TABLES INTO DUCKDB
# ============================================================

profitability_database_tables = {
    "fact_order_profitability":
        profitability_orders,

    "fact_order_item_profitability":
        profit_order_items,

    "dark_store_profitability":
        dark_store_profitability,

    "category_profitability":
        category_profitability
}


for table_name, dataframe in (
    profitability_database_tables.items()
):

    temp_name = (
        f"df_{table_name}"
    )

    con.register(
        temp_name,
        dataframe
    )

    con.execute(
        f"""
        CREATE OR REPLACE TABLE
        {table_name}
        AS
        SELECT *
        FROM {temp_name}
        """
    )

    con.unregister(
        temp_name
    )


print("=" * 112)
print(" PROFITABILITY DATABASE TABLES")
print("=" * 112)

profitability_db_validation = []

for table_name, dataframe in (
    profitability_database_tables.items()
):

    db_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM {table_name}
        """
    ).fetchone()[0]

    profitability_db_validation.append({
        "table_name":
            table_name,

        "expected_rows":
            len(dataframe),

        "database_rows":
            db_rows,

        "row_count_match":
            len(dataframe) == db_rows
    })


profitability_db_validation = pd.DataFrame(
    profitability_db_validation
)

display(
    profitability_db_validation
)


if (
    profitability_db_validation[
        "row_count_match"
    ].all()
):

    print(
        "\nProfitability database validation: PASS ✅"
    )

else:

    print(
        "\nProfitability database validation: REVIEW ⚠️"
    )

 PROFITABILITY DATABASE TABLES


,table_name,expected_rows,database_rows,row_count_match
0,fact_order_profitability,14333,14333,True
1,fact_order_item_profitability,38986,38986,True
2,dark_store_profitability,20,20,True
3,category_profitability,8,8,True



Profitability database validation: PASS ✅


In [67]:
# ============================================================
# 55B. SQL FILE 08 — PROFITABILITY & UNIT ECONOMICS
# ============================================================

sql_08 = r"""
-- ==========================================================
-- 08_profitability_unit_economics.sql
-- Quick-Commerce Profitability & Fulfilment Intelligence
-- ==========================================================
--
-- Profitability values are modelled contribution economics
-- based on explicit portfolio assumptions.
-- They are NOT reported company profits.
-- ==========================================================


-- ----------------------------------------------------------
-- 1. NETWORK UNIT ECONOMICS
-- ----------------------------------------------------------

SELECT

    COUNT(*) AS completed_orders,

    ROUND(
        SUM(merchandise_revenue),
        2
    ) AS completed_gmv,

    ROUND(
        SUM(modelled_merchandise_gross_profit),
        2
    ) AS merchandise_gross_profit,

    ROUND(
        SUM(modelled_variable_cost_to_serve),
        2
    ) AS variable_cost_to_serve,

    ROUND(
        SUM(modelled_contribution_margin),
        2
    ) AS contribution_margin,

    ROUND(
        100.0 *
        SUM(modelled_contribution_margin)
        /
        NULLIF(
            SUM(merchandise_revenue),
            0
        ),
        2
    ) AS contribution_margin_pct,

    ROUND(
        AVG(modelled_contribution_margin),
        2
    ) AS avg_contribution_per_order,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE modelled_contribution_margin < 0
        )
        /
        COUNT(*),
        2
    ) AS negative_contribution_order_pct

FROM fact_order_profitability
;


-- ----------------------------------------------------------
-- 2. DARK STORE PROFITABILITY RANKING
-- ----------------------------------------------------------

SELECT

    dark_store_id,
    dark_store_name,
    city,

    completed_orders,

    ROUND(
        completed_gmv,
        2
    ) AS completed_gmv,

    ROUND(
        contribution_margin,
        2
    ) AS contribution_margin,

    ROUND(
        contribution_margin_pct,
        2
    ) AS contribution_margin_pct,

    ROUND(
        avg_contribution_per_order,
        2
    ) AS avg_contribution_per_order,

    ROUND(
        negative_order_share_pct,
        2
    ) AS negative_order_share_pct,

    RANK() OVER (
        ORDER BY
            contribution_margin DESC
    ) AS contribution_rank,

    RANK() OVER (
        ORDER BY
            contribution_margin_pct DESC
    ) AS margin_efficiency_rank

FROM dark_store_profitability

ORDER BY
    contribution_rank
;


-- ----------------------------------------------------------
-- 3. CATEGORY GROSS PROFITABILITY
-- ----------------------------------------------------------

SELECT

    category,

    completed_orders,
    units_sold,

    ROUND(
        revenue,
        2
    ) AS revenue,

    ROUND(
        merchandise_gross_profit,
        2
    ) AS merchandise_gross_profit,

    ROUND(
        gross_margin_pct,
        2
    ) AS gross_margin_pct,

    ROUND(
        revenue_share_pct,
        2
    ) AS revenue_share_pct,

    RANK() OVER (
        ORDER BY
            merchandise_gross_profit DESC
    ) AS gross_profit_rank

FROM category_profitability

ORDER BY
    gross_profit_rank
;


-- ----------------------------------------------------------
-- 4. DISTANCE ECONOMICS
-- ----------------------------------------------------------

WITH distance_economics AS (

    SELECT

        CASE

            WHEN distance_km <= 3
                THEN '0–3 km'

            WHEN distance_km <= 5
                THEN '3–5 km'

            WHEN distance_km <= 7
                THEN '5–7 km'

            ELSE '7+ km'

        END AS distance_band,

        CASE
            WHEN distance_km <= 3 THEN 1
            WHEN distance_km <= 5 THEN 2
            WHEN distance_km <= 7 THEN 3
            ELSE 4
        END AS band_order,

        merchandise_revenue,
        modelled_last_mile_cost,
        modelled_variable_cost_to_serve,
        modelled_contribution_margin

    FROM fact_order_profitability
)

SELECT

    distance_band,

    COUNT(*) AS orders,

    ROUND(
        AVG(merchandise_revenue),
        2
    ) AS avg_order_value,

    ROUND(
        AVG(modelled_last_mile_cost),
        2
    ) AS avg_last_mile_cost,

    ROUND(
        AVG(modelled_variable_cost_to_serve),
        2
    ) AS avg_cost_to_serve,

    ROUND(
        AVG(modelled_contribution_margin),
        2
    ) AS avg_contribution,

    ROUND(
        100.0 *
        SUM(modelled_contribution_margin)
        /
        NULLIF(
            SUM(merchandise_revenue),
            0
        ),
        2
    ) AS contribution_margin_pct

FROM distance_economics

GROUP BY
    distance_band,
    band_order

ORDER BY
    band_order
;


-- ----------------------------------------------------------
-- 5. LOW-VALUE / LOSS-MAKING ORDER DIAGNOSTIC
-- ----------------------------------------------------------

WITH order_segments AS (

    SELECT

        *,

        NTILE(4) OVER (
            ORDER BY merchandise_revenue
        ) AS basket_value_quartile

    FROM fact_order_profitability
)

SELECT

    basket_value_quartile,

    COUNT(*) AS orders,

    ROUND(
        AVG(merchandise_revenue),
        2
    ) AS avg_order_value,

    ROUND(
        AVG(modelled_variable_cost_to_serve),
        2
    ) AS avg_cost_to_serve,

    ROUND(
        AVG(modelled_contribution_margin),
        2
    ) AS avg_contribution,

    COUNT(*) FILTER (
        WHERE modelled_contribution_margin < 0
    ) AS negative_contribution_orders,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE modelled_contribution_margin < 0
        )
        /
        COUNT(*),
        2
    ) AS negative_contribution_order_pct

FROM order_segments

GROUP BY
    basket_value_quartile

ORDER BY
    basket_value_quartile
;
"""


sql_files[
    "08_profitability_unit_economics.sql"
] = sql_08


(
    SQL_DIR /
    "08_profitability_unit_economics.sql"
).write_text(
    sql_08,
    encoding="utf-8"
)


print(
    "Created:",
    SQL_DIR /
    "08_profitability_unit_economics.sql"
)

Created: /content/quick_commerce_intelligence/sql/08_profitability_unit_economics.sql


In [68]:
# ============================================================
# 55C. SQL ASSET VALIDATION — 8 FILES
# ============================================================

generated_sql_files = sorted(
    SQL_DIR.glob("*.sql")
)

print("=" * 110)
print(" SQL PORTFOLIO ASSETS")
print("=" * 110)

for sql_file in generated_sql_files:

    print(
        "✓",
        sql_file.name
    )


print(
    f"\nSQL files created: "
    f"{len(generated_sql_files)}"
)


if len(generated_sql_files) >= 8:

    print(
        "SQL asset validation: PASS ✅"
    )

else:

    print(
        "SQL asset validation: REVIEW ⚠️"
    )

 SQL PORTFOLIO ASSETS
✓ 01_schema_and_quality.sql
✓ 02_commercial_performance.sql
✓ 03_dark_store_operations.sql
✓ 04_inventory_intelligence.sql
✓ 05_fulfilment_sla.sql
✓ 06_customer_experience.sql
✓ 07_management_priority.sql
✓ 08_profitability_unit_economics.sql

SQL files created: 8
SQL asset validation: PASS ✅


## 56. Final Analytics Export Package

Validated analytical outputs are exported into a curated reporting layer for:

- GitHub project documentation
- Power BI semantic modelling
- KPI reconciliation
- Dashboard development
- Portfolio reproducibility

The export layer contains management-ready summary tables rather than the original raw workbook.

Observed metrics and scenario-modelled metrics remain explicitly separated throughout the project.

In [69]:
# ============================================================
# 56. FINAL ANALYTICS EXPORT PACKAGE
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# Export directory
# ------------------------------------------------------------

POWERBI_EXPORT_DIR = (
    PROJECT_ROOT /
    "powerbi_exports"
)

POWERBI_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. EXECUTIVE KPI TABLE
# ============================================================

executive_kpis = con.execute(
    """
    WITH commercial AS (

        SELECT

            COUNT(*) AS total_orders,

            COUNT(*) FILTER (
                WHERE commercial_eligible = TRUE
            ) AS commercial_orders,

            COUNT(*) FILTER (
                WHERE completed_sales_eligible = TRUE
            ) AS completed_orders,

            SUM(
                CASE
                    WHEN completed_sales_eligible = TRUE
                    THEN gross_merchandise_value
                    ELSE 0
                END
            ) AS completed_gmv,

            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END
            ) AS avg_delivery_minutes,

            QUANTILE_CONT(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END,
                0.90
            ) AS p90_delivery_minutes,

            100.0 *
            COUNT(*) FILTER (
                WHERE
                    sla_eligible = TRUE
                    AND
                    derived_sla_status = 'On Time'
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE sla_eligible = TRUE
                ),
                0
            ) AS sla_on_time_pct,

            COUNT(*) FILTER (
                WHERE cx_eligible = TRUE
            ) AS cx_eligible_orders,

            AVG(
                CASE
                    WHEN cx_eligible = TRUE
                    THEN avg_rating
                END
            ) AS avg_customer_rating

        FROM fact_order
    ),

    profitability AS (

        SELECT

            SUM(
                modelled_merchandise_gross_profit
            ) AS merchandise_gross_profit,

            SUM(
                modelled_variable_cost_to_serve
            ) AS variable_cost_to_serve,

            SUM(
                modelled_contribution_margin
            ) AS contribution_margin,

            100.0 *
            SUM(
                modelled_contribution_margin
            )
            /
            NULLIF(
                SUM(merchandise_revenue),
                0
            ) AS contribution_margin_pct,

            100.0 *
            COUNT(*) FILTER (
                WHERE
                    modelled_contribution_margin < 0
            )
            /
            COUNT(*) AS negative_order_share_pct

        FROM fact_order_profitability
    ),

    inventory AS (

        SELECT

            COUNT(*) FILTER (
                WHERE reorder_required = TRUE
            ) AS reorder_positions,

            SUM(
                recommended_replenishment_units
            ) AS recommended_replenishment_units,

            AVG(
                stockout_probability_pct
            ) AS avg_stockout_probability_pct

        FROM inventory_intelligence
    )

    SELECT

        c.total_orders,
        c.commercial_orders,
        c.completed_orders,

        ROUND(
            c.completed_gmv,
            2
        ) AS completed_gmv,

        ROUND(
            c.completed_gmv /
            NULLIF(c.completed_orders, 0),
            2
        ) AS completed_aov,

        ROUND(
            c.avg_delivery_minutes,
            2
        ) AS avg_delivery_minutes,

        ROUND(
            c.p90_delivery_minutes,
            2
        ) AS p90_delivery_minutes,

        ROUND(
            c.sla_on_time_pct,
            2
        ) AS sla_on_time_pct,

        c.cx_eligible_orders,

        ROUND(
            c.avg_customer_rating,
            2
        ) AS avg_customer_rating,

        ROUND(
            p.merchandise_gross_profit,
            2
        ) AS modelled_merchandise_gross_profit,

        ROUND(
            p.variable_cost_to_serve,
            2
        ) AS modelled_variable_cost_to_serve,

        ROUND(
            p.contribution_margin,
            2
        ) AS modelled_contribution_margin,

        ROUND(
            p.contribution_margin_pct,
            2
        ) AS contribution_margin_pct,

        ROUND(
            p.negative_order_share_pct,
            2
        ) AS negative_order_share_pct,

        i.reorder_positions,

        i.recommended_replenishment_units,

        ROUND(
            i.avg_stockout_probability_pct,
            2
        ) AS avg_stockout_probability_pct

    FROM commercial c
    CROSS JOIN profitability p
    CROSS JOIN inventory i
    """
).df()


# ============================================================
# 2. MONTHLY PERFORMANCE
# ============================================================

monthly_performance = con.execute(
    """
    WITH order_month AS (

        SELECT

            DATE_TRUNC(
                'month',
                order_date
            ) AS month,

            COUNT(*) AS total_orders,

            COUNT(*) FILTER (
                WHERE completed_sales_eligible = TRUE
            ) AS completed_orders,

            SUM(
                CASE
                    WHEN completed_sales_eligible = TRUE
                    THEN gross_merchandise_value
                    ELSE 0
                END
            ) AS completed_gmv,

            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END
            ) AS avg_delivery_minutes,

            100.0 *
            COUNT(*) FILTER (
                WHERE
                    sla_eligible = TRUE
                    AND derived_sla_status = 'On Time'
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE sla_eligible = TRUE
                ),
                0
            ) AS sla_on_time_pct,

            AVG(
                CASE
                    WHEN cx_eligible = TRUE
                    THEN avg_rating
                END
            ) AS avg_customer_rating

        FROM fact_order

        GROUP BY 1
    ),

    profit_month AS (

        SELECT

            DATE_TRUNC(
                'month',
                order_date
            ) AS month,

            SUM(
                modelled_contribution_margin
            ) AS contribution_margin,

            SUM(
                modelled_variable_cost_to_serve
            ) AS variable_cost_to_serve,

            100.0 *
            SUM(
                modelled_contribution_margin
            )
            /
            NULLIF(
                SUM(merchandise_revenue),
                0
            ) AS contribution_margin_pct

        FROM fact_order_profitability

        GROUP BY 1
    )

    SELECT

        o.month,
        o.total_orders,
        o.completed_orders,

        ROUND(
            o.completed_gmv,
            2
        ) AS completed_gmv,

        ROUND(
            o.completed_gmv /
            NULLIF(o.completed_orders, 0),
            2
        ) AS completed_aov,

        ROUND(
            o.avg_delivery_minutes,
            2
        ) AS avg_delivery_minutes,

        ROUND(
            o.sla_on_time_pct,
            2
        ) AS sla_on_time_pct,

        ROUND(
            o.avg_customer_rating,
            2
        ) AS avg_customer_rating,

        ROUND(
            p.variable_cost_to_serve,
            2
        ) AS variable_cost_to_serve,

        ROUND(
            p.contribution_margin,
            2
        ) AS contribution_margin,

        ROUND(
            p.contribution_margin_pct,
            2
        ) AS contribution_margin_pct

    FROM order_month o

    LEFT JOIN profit_month p
        ON o.month = p.month

    ORDER BY
        o.month
    """
).df()


# ============================================================
# 3. DARK STORE SCORECARD
# ============================================================

dark_store_scorecard = con.execute(
    """
    WITH operational AS (

        SELECT

            dark_store_id,
            dark_store_name,
            city,
            state,

            COUNT(*) AS total_orders,

            COUNT(*) FILTER (
                WHERE completed_sales_eligible = TRUE
            ) AS completed_orders,

            SUM(
                CASE
                    WHEN completed_sales_eligible = TRUE
                    THEN gross_merchandise_value
                    ELSE 0
                END
            ) AS completed_gmv,

            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN delivery_time_mins
                END
            ) AS avg_delivery_minutes,

            AVG(
                CASE
                    WHEN sla_eligible = TRUE
                    THEN distance_km
                END
            ) AS avg_distance_km,

            100.0 *
            COUNT(*) FILTER (
                WHERE
                    sla_eligible = TRUE
                    AND derived_sla_status = 'On Time'
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE sla_eligible = TRUE
                ),
                0
            ) AS sla_on_time_pct,

            AVG(
                CASE
                    WHEN cx_eligible = TRUE
                    THEN avg_rating
                END
            ) AS avg_customer_rating

        FROM fact_order

        GROUP BY
            dark_store_id,
            dark_store_name,
            city,
            state
    ),

    inventory AS (

        SELECT

            dark_store_id,

            COUNT(*) FILTER (
                WHERE reorder_required = TRUE
            ) AS reorder_positions,

            SUM(
                recommended_replenishment_units
            ) AS replenishment_units,

            AVG(
                stockout_probability_pct
            ) AS avg_stockout_probability_pct,

            MAX(
                replenishment_priority_score
            ) AS max_inventory_priority_score

        FROM inventory_intelligence

        GROUP BY
            dark_store_id
    )

    SELECT

        o.dark_store_id,
        o.dark_store_name,
        o.city,
        o.state,

        o.total_orders,
        o.completed_orders,

        ROUND(
            o.completed_gmv,
            2
        ) AS completed_gmv,

        ROUND(
            o.avg_delivery_minutes,
            2
        ) AS avg_delivery_minutes,

        ROUND(
            o.avg_distance_km,
            2
        ) AS avg_distance_km,

        ROUND(
            o.sla_on_time_pct,
            2
        ) AS sla_on_time_pct,

        ROUND(
            o.avg_customer_rating,
            2
        ) AS avg_customer_rating,

        ROUND(
            p.contribution_margin,
            2
        ) AS contribution_margin,

        ROUND(
            p.contribution_margin_pct,
            2
        ) AS contribution_margin_pct,

        ROUND(
            p.avg_contribution_per_order,
            2
        ) AS avg_contribution_per_order,

        ROUND(
            p.negative_order_share_pct,
            2
        ) AS negative_order_share_pct,

        i.reorder_positions,
        i.replenishment_units,

        ROUND(
            i.avg_stockout_probability_pct,
            2
        ) AS avg_stockout_probability_pct,

        ROUND(
            i.max_inventory_priority_score,
            2
        ) AS max_inventory_priority_score

    FROM operational o

    LEFT JOIN dark_store_profitability p
        ON o.dark_store_id =
           p.dark_store_id

    LEFT JOIN inventory i
        ON o.dark_store_id =
           i.dark_store_id

    ORDER BY
        o.completed_gmv DESC
    """
).df()


# ============================================================
# 4. CUSTOMER EXPERIENCE BY STORE
# ============================================================

customer_experience_store = con.execute(
    """
    SELECT

        dark_store_id,
        dark_store_name,
        city,

        COUNT(*) FILTER (
            WHERE completed_sales_eligible = TRUE
        ) AS completed_orders,

        COUNT(*) FILTER (
            WHERE cx_eligible = TRUE
        ) AS rated_orders,

        ROUND(
            100.0 *
            COUNT(*) FILTER (
                WHERE cx_eligible = TRUE
            )
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE completed_sales_eligible = TRUE
                ),
                0
            ),
            2
        ) AS rating_coverage_pct,

        ROUND(
            AVG(
                CASE
                    WHEN cx_eligible = TRUE
                    THEN avg_rating
                END
            ),
            2
        ) AS avg_rating

    FROM fact_order

    GROUP BY
        dark_store_id,
        dark_store_name,
        city

    ORDER BY
        avg_rating DESC
    """
).df()


# ============================================================
# 5. EXPORT TABLE COLLECTION
# ============================================================

export_tables = {

    "00_executive_kpis.csv":
        executive_kpis,

    "01_monthly_performance.csv":
        monthly_performance,

    "02_dark_store_scorecard.csv":
        dark_store_scorecard,

    "03_inventory_control_tower.csv":
        inventory_model,

    "04_inventory_priority.csv":
        inventory_priority,

    "05_weekly_store_category.csv":
        weekly_store_category,

    "06_customer_experience_store.csv":
        customer_experience_store,

    "07_dark_store_profitability.csv":
        dark_store_profitability,

    "08_category_profitability.csv":
        category_profitability,

    "09_distance_profitability.csv":
        distance_profitability,

    "10_sla_profitability.csv":
        sla_profitability,

    "11_basket_profitability.csv":
        basket_profitability,

    "12_dim_dark_store.csv":
        dim_dark_store,

    "13_dim_date.csv":
        dim_date,

    "14_inventory_assumptions.csv":
        inventory_assumptions,

    "15_profitability_assumptions.csv":
        profitability_category_assumptions
}


# ============================================================
# 6. WRITE CSV FILES
# ============================================================

export_manifest_records = []


for file_name, dataframe in export_tables.items():

    export_path = (
        POWERBI_EXPORT_DIR /
        file_name
    )

    dataframe.to_csv(
        export_path,
        index=False
    )

    export_manifest_records.append({
        "file_name":
            file_name,

        "rows":
            len(dataframe),

        "columns":
            len(dataframe.columns),

        "size_kb":
            round(
                export_path.stat().st_size /
                1024,
                2
            )
    })


export_manifest = pd.DataFrame(
    export_manifest_records
)


# ============================================================
# 7. EXPORT MANIFEST
# ============================================================

manifest_path = (
    POWERBI_EXPORT_DIR /
    "99_export_manifest.csv"
)

export_manifest.to_csv(
    manifest_path,
    index=False
)


print("=" * 116)
print(" FINAL ANALYTICS EXPORT PACKAGE")
print("=" * 116)

print(
    f"Export directory : "
    f"{POWERBI_EXPORT_DIR}"
)

print(
    f"Tables exported  : "
    f"{len(export_tables):,}"
)

display(
    export_manifest
)


# ============================================================
# 8. BASIC VALIDATION
# ============================================================

zero_row_files = (
    export_manifest[
        export_manifest["rows"] <= 0
    ]
)


missing_files = [
    file_name
    for file_name
    in export_tables.keys()
    if not (
        POWERBI_EXPORT_DIR /
        file_name
    ).exists()
]


print("\nValidation:")

print(
    f"Zero-row exports : "
    f"{len(zero_row_files)}"
)

print(
    f"Missing files    : "
    f"{len(missing_files)}"
)


if (
    len(zero_row_files) == 0
    and
    len(missing_files) == 0
):

    print(
        "\nAnalytics export validation: PASS ✅"
    )

else:

    print(
        "\nAnalytics export validation: REVIEW ⚠️"
    )

 FINAL ANALYTICS EXPORT PACKAGE
Export directory : /content/quick_commerce_intelligence/powerbi_exports
Tables exported  : 16


,file_name,rows,columns,size_kb
0,00_executive_kpis.csv,1,18,0.50
1,01_monthly_performance.csv,24,11,2.01
2,02_dark_store_scorecard.csv,20,19,3.00
3,03_inventory_control_tower.csv,160,42,70.62
4,04_inventory_priority.csv,160,17,23.55
5,05_weekly_store_category.csv,16800,8,927.19
6,06_customer_experience_store.csv,20,7,1.22
7,07_dark_store_profitability.csv,20,19,4.41
8,08_category_profitability.csv,8,8,0.75
9,09_distance_profitability.csv,4,9,0.62



Validation:
Zero-row exports : 0
Missing files    : 0

Analytics export validation: PASS ✅


## 57. Final Project Validation & Analytics Bundle

Before GitHub packaging, the complete analytical pipeline is validated across its major layers:

- Commercial KPIs
- Fulfilment SLA
- Customer experience
- Inventory intelligence
- Profitability and unit economics
- DuckDB analytical warehouse
- SQL portfolio assets
- Power BI export tables

A portable analytics bundle is then created containing generated SQL, reporting outputs, database assets and validation reports.

Raw source files are intentionally excluded from the portfolio bundle.

In [70]:
# ============================================================
# 57. FINAL PROJECT VALIDATION & ANALYTICS BUNDLE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import zipfile


# ============================================================
# 1. PROJECT PATHS
# ============================================================

REPORTS_DIR = (
    PROJECT_ROOT /
    "reports"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ANALYTICS_BUNDLE_PATH = (
    PROJECT_ROOT /
    "quick_commerce_intelligence_analytics_bundle.zip"
)


# ============================================================
# 2. CHECKPOINT DUCKDB
# ============================================================

con.execute(
    "CHECKPOINT"
)


# ============================================================
# 3. RECOUNT DATABASE TABLES
# ============================================================

final_database_inventory = con.execute(
    """
    SELECT
        table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type = 'BASE TABLE'
    ORDER BY table_name
    """
).df()


final_database_table_count = len(
    final_database_inventory
)


# ============================================================
# 4. SQL ASSET COUNT
# ============================================================

final_sql_files = sorted(
    SQL_DIR.glob("*.sql")
)

final_sql_count = len(
    final_sql_files
)


# ============================================================
# 5. POWER BI EXPORT COUNT
# ============================================================

final_export_files = sorted(
    POWERBI_EXPORT_DIR.glob("*.csv")
)

final_export_count = len(
    final_export_files
)


# ============================================================
# 6. KPI VALIDATION
# ============================================================

validation_records = []


def add_validation(
    check_name,
    actual,
    expected,
    tolerance=0
):

    if isinstance(actual, (float, np.floating)):

        passed = (
            abs(
                float(actual)
                -
                float(expected)
            )
            <= tolerance
        )

    else:

        passed = (
            actual == expected
        )

    validation_records.append({
        "validation_check":
            check_name,

        "actual_value":
            actual,

        "expected_value":
            expected,

        "status":
            "PASS"
            if passed
            else "REVIEW"
    })


# ------------------------------------------------------------
# Commercial
# ------------------------------------------------------------

add_validation(
    "Total orders",
    int(
        executive_kpis.loc[
            0,
            "total_orders"
        ]
    ),
    20000
)


add_validation(
    "Commercial eligible orders",
    int(
        executive_kpis.loc[
            0,
            "commercial_orders"
        ]
    ),
    18372
)


add_validation(
    "Completed orders",
    int(
        executive_kpis.loc[
            0,
            "completed_orders"
        ]
    ),
    14333
)


add_validation(
    "Completed GMV",
    float(
        executive_kpis.loc[
            0,
            "completed_gmv"
        ]
    ),
    28224720.00,
    tolerance=0.01
)


# ------------------------------------------------------------
# Fulfilment
# ------------------------------------------------------------

add_validation(
    "Average delivery minutes",
    float(
        executive_kpis.loc[
            0,
            "avg_delivery_minutes"
        ]
    ),
    41.30,
    tolerance=0.01
)


add_validation(
    "P90 delivery minutes",
    float(
        executive_kpis.loc[
            0,
            "p90_delivery_minutes"
        ]
    ),
    71.00,
    tolerance=0.01
)


add_validation(
    "SLA on-time %",
    float(
        executive_kpis.loc[
            0,
            "sla_on_time_pct"
        ]
    ),
    42.50,
    tolerance=0.01
)


# ------------------------------------------------------------
# Customer Experience
# ------------------------------------------------------------

add_validation(
    "CX eligible orders",
    int(
        executive_kpis.loc[
            0,
            "cx_eligible_orders"
        ]
    ),
    9826
)


add_validation(
    "Average customer rating",
    float(
        executive_kpis.loc[
            0,
            "avg_customer_rating"
        ]
    ),
    3.88,
    tolerance=0.01
)


# ------------------------------------------------------------
# Profitability
# ------------------------------------------------------------

add_validation(
    "Modelled contribution margin",
    float(
        executive_kpis.loc[
            0,
            "modelled_contribution_margin"
        ]
    ),
    4301584.56,
    tolerance=0.01
)


add_validation(
    "Contribution margin %",
    float(
        executive_kpis.loc[
            0,
            "contribution_margin_pct"
        ]
    ),
    15.24,
    tolerance=0.01
)


add_validation(
    "Negative contribution order %",
    float(
        executive_kpis.loc[
            0,
            "negative_order_share_pct"
        ]
    ),
    10.31,
    tolerance=0.01
)


# ------------------------------------------------------------
# Inventory
# ------------------------------------------------------------

add_validation(
    "Inventory reorder positions",
    int(
        executive_kpis.loc[
            0,
            "reorder_positions"
        ]
    ),
    130
)


add_validation(
    "Recommended replenishment units",
    int(
        executive_kpis.loc[
            0,
            "recommended_replenishment_units"
        ]
    ),
    800
)


add_validation(
    "Average stockout probability %",
    float(
        executive_kpis.loc[
            0,
            "avg_stockout_probability_pct"
        ]
    ),
    10.64,
    tolerance=0.01
)


# ------------------------------------------------------------
# Architecture
# ------------------------------------------------------------

add_validation(
    "DuckDB base tables",
    final_database_table_count,
    14
)


add_validation(
    "SQL portfolio files",
    final_sql_count,
    8
)


# 16 analytical exports + 99_export_manifest.csv
add_validation(
    "CSV reporting files",
    final_export_count,
    17
)


# ============================================================
# 7. FINAL VALIDATION REPORT
# ============================================================

final_validation_report = pd.DataFrame(
    validation_records
)


print("=" * 118)
print(" FINAL PROJECT VALIDATION")
print("=" * 118)

display(
    final_validation_report
)


failed_checks = (
    final_validation_report[
        final_validation_report[
            "status"
        ] != "PASS"
    ]
)


print(
    f"\nValidation checks : "
    f"{len(final_validation_report):,}"
)

print(
    f"Checks passed     : "
    f"{(
        final_validation_report['status']
        == 'PASS'
    ).sum():,}"
)

print(
    f"Checks requiring review : "
    f"{len(failed_checks):,}"
)


if len(failed_checks) == 0:

    print(
        "\nFINAL PROJECT VALIDATION: PASS ✅"
    )

else:

    print(
        "\nFINAL PROJECT VALIDATION: REVIEW ⚠️"
    )


# ============================================================
# 8. SAVE VALIDATION REPORT
# ============================================================

validation_report_path = (
    REPORTS_DIR /
    "final_validation_report.csv"
)

final_validation_report.to_csv(
    validation_report_path,
    index=False
)


# ============================================================
# 9. PROJECT ASSET MANIFEST
# ============================================================

project_manifest_records = []


asset_directories = {
    "SQL":
        SQL_DIR,

    "Power BI Export":
        POWERBI_EXPORT_DIR,

    "Database":
        DATABASE_DIR,

    "Report":
        REPORTS_DIR
}


for asset_type, directory in (
    asset_directories.items()
):

    for path in sorted(
        directory.rglob("*")
    ):

        if not path.is_file():
            continue

        project_manifest_records.append({
            "asset_type":
                asset_type,

            "relative_path":
                str(
                    path.relative_to(
                        PROJECT_ROOT
                    )
                ),

            "size_kb":
                round(
                    path.stat().st_size
                    /
                    1024,
                    2
                )
        })


project_manifest = pd.DataFrame(
    project_manifest_records
)


project_manifest_path = (
    REPORTS_DIR /
    "project_asset_manifest.csv"
)

project_manifest.to_csv(
    project_manifest_path,
    index=False
)


print("\n" + "=" * 118)
print(" PROJECT ASSET MANIFEST")
print("=" * 118)

print(
    f"Generated assets tracked : "
    f"{len(project_manifest):,}"
)

display(
    project_manifest
)


# ============================================================
# 10. BUILD ANALYTICS ZIP BUNDLE
#
# Excludes:
# - raw source ZIP
# - source Excel workbook
#
# Includes:
# - SQL
# - Power BI exports
# - DuckDB database
# - validation reports
# ============================================================

if ANALYTICS_BUNDLE_PATH.exists():

    ANALYTICS_BUNDLE_PATH.unlink()


with zipfile.ZipFile(
    ANALYTICS_BUNDLE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    bundle_directories = [
        SQL_DIR,
        POWERBI_EXPORT_DIR,
        DATABASE_DIR,
        REPORTS_DIR
    ]

    for directory in bundle_directories:

        for path in directory.rglob("*"):

            if not path.is_file():
                continue

            archive_name = (
                Path(
                    "quick_commerce_intelligence"
                )
                /
                path.relative_to(
                    PROJECT_ROOT
                )
            )

            zip_file.write(
                path,
                archive_name
            )


# ============================================================
# 11. ZIP VALIDATION
# ============================================================

with zipfile.ZipFile(
    ANALYTICS_BUNDLE_PATH,
    "r"
) as zip_file:

    bundled_files = (
        zip_file.namelist()
    )


bundle_size_mb = (
    ANALYTICS_BUNDLE_PATH.stat().st_size
    /
    (1024 ** 2)
)


print("\n" + "=" * 118)
print(" ANALYTICS BUNDLE")
print("=" * 118)

print(
    f"Bundle path : "
    f"{ANALYTICS_BUNDLE_PATH}"
)

print(
    f"Files inside: "
    f"{len(bundled_files):,}"
)

print(
    f"Bundle size : "
    f"{bundle_size_mb:.2f} MB"
)


# ------------------------------------------------------------
# Confirm raw source exclusion
# ------------------------------------------------------------

raw_files_in_bundle = [
    name
    for name in bundled_files
    if (
        name.lower().endswith(".xlsx")
        or
        name.lower().endswith("archive.zip")
    )
]


print(
    f"Raw source files included: "
    f"{len(raw_files_in_bundle)}"
)


if len(raw_files_in_bundle) == 0:

    print(
        "Raw source exclusion: PASS ✅"
    )

else:

    print(
        "Raw source exclusion: REVIEW ⚠️"
    )


if (
    len(failed_checks) == 0
    and
    len(raw_files_in_bundle) == 0
    and
    ANALYTICS_BUNDLE_PATH.exists()
):

    print(
        "\nANALYTICS BUNDLE VALIDATION: PASS ✅"
    )

else:

    print(
        "\nANALYTICS BUNDLE VALIDATION: REVIEW ⚠️"
    )

 FINAL PROJECT VALIDATION


,validation_check,actual_value,expected_value,status
0,Total orders,"20,000.00","20,000.00",PASS
1,Commercial eligible orders,"18,372.00","18,372.00",PASS
2,Completed orders,"14,333.00","14,333.00",PASS
3,Completed GMV,"28,224,720.00","28,224,720.00",PASS
4,Average delivery minutes,41.30,41.30,PASS
5,P90 delivery minutes,71.00,71.00,PASS
6,SLA on-time %,42.50,42.50,PASS
7,CX eligible orders,"9,826.00","9,826.00",PASS
8,Average customer rating,3.88,3.88,PASS
9,Modelled contribution margin,"4,301,584.56","4,301,584.56",PASS



Validation checks : 18
Checks passed     : 18
Checks requiring review : 0

FINAL PROJECT VALIDATION: PASS ✅

 PROJECT ASSET MANIFEST
Generated assets tracked : 27


,asset_type,relative_path,size_kb
0,SQL,sql/01_schema_and_quality.sql,3.30
1,SQL,sql/02_commercial_performance.sql,4.07
2,SQL,sql/03_dark_store_operations.sql,5.09
3,SQL,sql/04_inventory_intelligence.sql,6.38
4,SQL,sql/05_fulfilment_sla.sql,7.54
5,SQL,sql/06_customer_experience.sql,6.31
6,SQL,sql/07_management_priority.sql,8.38
7,SQL,sql/08_profitability_unit_economics.sql,5.67
8,Power BI Export,powerbi_exports/00_executive_kpis.csv,0.50
9,Power BI Export,powerbi_exports/01_monthly_performance.csv,2.01



 ANALYTICS BUNDLE
Bundle path : /content/quick_commerce_intelligence/quick_commerce_intelligence_analytics_bundle.zip
Files inside: 28
Bundle size : 3.47 MB
Raw source files included: 0
Raw source exclusion: PASS ✅

ANALYTICS BUNDLE VALIDATION: PASS ✅


## 58. GitHub Portfolio Documentation

A recruiter-facing documentation layer is created to explain the project in business terms while preserving technical reproducibility.

The documentation covers:

- Business problem
- Analytical architecture
- Dataset disclosure
- Validated KPIs
- Key business findings
- Inventory methodology
- Profitability assumptions
- SQL capabilities
- KPI definitions
- Project limitations
- Planned Power BI dashboard structure

In [72]:
# ============================================================
# 58. PROFESSIONAL GITHUB DOCUMENTATION — CORRECTED
# ============================================================

from pathlib import Path
import pandas as pd


# ============================================================
# 1. DOCUMENTATION DIRECTORIES
# ============================================================

DOCS_DIR = PROJECT_ROOT / "docs"
DATA_DOCS_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
POWERBI_DIR = PROJECT_ROOT / "powerbi"

for directory in [
    DOCS_DIR,
    DATA_DOCS_DIR,
    NOTEBOOKS_DIR,
    POWERBI_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. MAIN README
# ============================================================

README_CONTENT = r'''
# India Quick-Commerce Profitability & Fulfilment Intelligence

## Dark Store, Inventory, Last-Mile & Unit Economics Analytics

An end-to-end analytics portfolio project that transforms a public
quick-commerce practice dataset into a management decision-support
system covering:

- Commercial performance
- Dark-store operations
- Inventory intelligence
- Fulfilment SLA
- Customer experience
- Last-mile economics
- Contribution-margin modelling
- Management prioritisation

> **Important disclosure**
>
> This project does not use proprietary operational or financial data
> from Zepto, Blinkit, Swiggy Instamart or any other quick-commerce
> company.
>
> The source dataset is a public Zepto-inspired practice dataset.
> Dark stores, inventory positions, replenishment policies and
> profitability assumptions are explicitly modelled for portfolio
> analytics.

---

# Business Objective

Quick-commerce businesses must simultaneously:

- grow order volume and GMV,
- maintain fast delivery,
- position inventory close to demand,
- minimise stockouts,
- control last-mile cost,
- and protect contribution margin.

This project develops an analytical system to answer:

1. Which dark stores generate the strongest commercial performance?
2. Which stores and categories face the highest inventory risk?
3. Where should replenishment be prioritised?
4. How does delivery distance affect cost-to-serve?
5. Which basket sizes create weak or negative contribution economics?
6. Which dark stores require management intervention?
7. How do commercial, inventory and fulfilment metrics interact?

---

# Technology Stack

| Layer | Technology |
|---|---|
| Data Engineering | Python, Pandas, NumPy |
| Statistical Modelling | SciPy |
| Analytical Database | DuckDB |
| SQL Analytics | SQL, CTEs, Window Functions |
| BI & Reporting | Power BI |
| Source Format | Excel |
| Development | Google Colab |
| Version Control | Git & GitHub |

---

# Analytical Architecture

Source Dataset
→ Python Data Quality & Transformation
→ Canonical Analytical Model
→ Dark Store Simulation
→ Inventory Intelligence
→ Fulfilment & Last-Mile Analytics
→ Profitability & Unit Economics
→ DuckDB Analytical Warehouse
→ SQL Analytics Layer
→ Power BI Reporting Tables
→ Management Decision Dashboard

---

# Dataset

**Dataset:** Zepto-Inspired Online Grocery & Delivery Dataset
**Source:** Kaggle

Dataset page:

https://www.kaggle.com/datasets/ashishjangir6494/zepto-inspired-online-grocery-and-delivery-dataset

The source workbook contains six relational tables:

| Table | Rows |
|---|---:|
| Customers | 10,000 |
| Products | 1,200 |
| Orders | 20,000 |
| Transactions | 50,000 |
| Ratings | 20,000 |
| Deliveries | 20,000 |

**Total source records: 121,200**

The raw workbook is intentionally not redistributed in this repository.
Users should obtain the dataset directly from its original Kaggle page.

---

# Data Quality Treatment

The raw source tables are preserved.

Important analytical rules include:

- `customer_id` is the authoritative customer identifier.
- Orders without transaction details remain in operational counts.
- Orders without transaction details are excluded from revenue analytics.
- Completed-sales analytics require a Delivered order and transaction detail.
- Multiple ratings are aggregated to one order-level average.
- Customer-experience KPIs use delivered rated orders only.
- Source delivery status is retained for lineage.
- A separate analytical SLA is derived from delivery minutes.
- Customer creation dates are excluded from lifecycle analysis because of temporal inconsistencies.
- Payment mode is treated as transaction-line metadata.

---

# Analytical SLA

For portfolio analysis:

| SLA Status | Delivery Time |
|---|---:|
| On Time | <= 35 minutes |
| At Risk | 36–50 minutes |
| Late | > 50 minutes |

Only delivered orders are SLA eligible.

This is an analytical SLA definition and not a claimed Zepto SLA policy.

---

# Dark Store Network

The source dataset does not contain dark-store identifiers.

A deterministic analytical network of:

**20 modelled dark stores across 10 cities**

was created.

Orders are assigned consistently using a deterministic customer-based
hash.

These locations are analytical operating nodes and do not represent
actual company dark stores.

---

# Validated Executive KPIs

| KPI | Result |
|---|---:|
| Total Orders | 20,000 |
| Commercial Eligible Orders | 18,372 |
| Completed Orders | 14,333 |
| Completed GMV | ₹28,224,720 |
| Completed AOV | ₹1,969.21 |
| Average Delivery Time | 41.30 min |
| P90 Delivery Time | 71.00 min |
| SLA On-Time Rate | 42.50% |
| CX Eligible Orders | 9,826 |
| Average Customer Rating | 3.88 |
| Modelled Contribution Margin | ₹4,301,584.56 |
| Contribution Margin % | 15.24% |
| Negative Contribution Orders | ~10.3% |
| Inventory Positions | 160 |
| Reorder Positions | 130 |
| Recommended Replenishment | 800 units |
| Average Stockout Probability | 10.64% |

Core metrics are reconciled across Python, DuckDB and SQL.

---

# Key Business Findings

## 1. Low-value baskets create the greatest profitability pressure

The lowest basket-value quartile averages approximately:

**₹449.68 per order**

Within this segment:

- 40.89% of orders are negative contribution
- average cost-to-serve is approximately ₹79.31
- average contribution is approximately ₹13.35/order

No negative-contribution orders appear in Q2–Q4 under the scenario model.

### Decision implication

Potential interventions include:

- basket-building incentives,
- minimum-order economics,
- product bundling,
- targeted cross-sell,
- and delivery-fee optimisation.

---

## 2. Delivery distance increases cost-to-serve

| Distance | Avg Cost-to-Serve | Contribution Margin |
|---|---:|---:|
| 0–3 km | ₹83.85 | 15.86% |
| 3–5 km | ₹92.76 | 15.50% |
| 5–7 km | ₹100.77 | 15.14% |
| 7+ km | ₹110.73 | 14.58% |

Contribution efficiency declines as delivery distance increases within
the modelled scenario.

This is an analytical association and should not be interpreted as
causal proof.

---

## 3. Fulfilment performance and economics interact

| SLA Status | Contribution Margin |
|---|---:|
| On Time | 15.72% |
| At Risk | 15.14% |
| Late | 14.71% |

Late deliveries are also associated with greater delivery distance and
higher modelled fulfilment cost.

---

## 4. Inventory intervention is highly targeted

Across 160 Dark Store × Category positions:

- 92 are Low stockout risk
- 56 are Medium risk
- 12 are High risk
- 130 have reached or fallen below reorder point
- 800 units are recommended for replenishment

---

## 5. Commercial importance and demand variability are combined

Commercial Value:

- High
- Medium
- Low

Demand Variability:

- X = comparatively predictable
- Y = moderate variability
- Z = comparatively volatile

The **High Value × Z** combination represents a strategically important
inventory segment.

---

# Inventory Methodology

The source dataset does not contain actual inventory-on-hand values.

The project therefore models inventory decision intelligence using:

- weekly observed demand,
- ABC/XYZ-style segmentation,
- commercial value,
- lead-time assumptions,
- coverage assumptions,
- service levels,
- safety stock,
- reorder points,
- simulated inventory on hand,
- stockout probability,
- replenishment priority.

## Core formulas

Average Daily Demand:

Average Weekly Demand / 7

Safety Stock:

Service Z Score × Lead-Time Demand Standard Deviation

Reorder Point:

Expected Lead-Time Demand + Safety Stock

Recommended Replenishment:

Target Inventory − Modelled Inventory On Hand

Replenishment is recommended only after the calculated reorder threshold
is reached.

---

# Inventory Scenario Assumptions

| Category | Lead Time | Coverage |
|---|---:|---:|
| Fruits & Vegetables | 1 | 3 |
| Dairy & Bakery | 1 | 3 |
| Beverages | 2 | 5 |
| Snacks & Branded Foods | 2 | 6 |
| Grocery & Staples | 3 | 7 |
| Baby Care | 4 | 8 |
| Personal Care | 4 | 8 |
| Household | 5 | 10 |

These are portfolio modelling assumptions.

---

# Profitability & Unit Economics

Actual procurement costs and corporate cost structures are not present
in the source dataset.

The project therefore creates a transparent scenario-based
contribution-margin model.

## Modelled category gross margins

| Category | Margin |
|---|---:|
| Fruits & Vegetables | 24% |
| Dairy & Bakery | 20% |
| Beverages | 18% |
| Snacks & Branded Foods | 21% |
| Grocery & Staples | 16% |
| Baby Care | 18% |
| Personal Care | 25% |
| Household | 22% |

## Fulfilment assumptions

- Picking & packing: ₹20/order
- Packaging: ₹8/order
- Last-mile base cost: ₹25/order
- Distance cost: ₹4/km
- Payment processing: 1.2% of revenue

## Contribution formula

Modelled Contribution Margin

= Merchandise Gross Profit
− Picking & Packing
− Packaging
− Last-Mile Cost
− Payment Processing

The metric represents contribution before fixed overhead and is not
actual company profit.

---

# Dark Store Profitability

The project distinguishes between:

- total contribution generation,
- margin efficiency,
- GMV scale,
- and negative-contribution exposure.

For example, a store can rank highly in revenue but lower in
operational or inventory performance.

This prevents management prioritisation from becoming a simple GMV
leaderboard.

---

# SQL Analytics

Eight standalone SQL scripts are included.

| SQL File | Domain |
|---|---|
| 01_schema_and_quality.sql | Data quality and integrity |
| 02_commercial_performance.sql | Commercial analytics |
| 03_dark_store_operations.sql | Store operations |
| 04_inventory_intelligence.sql | Inventory |
| 05_fulfilment_sla.sql | SLA and fulfilment |
| 06_customer_experience.sql | Customer experience |
| 07_management_priority.sql | Management ranking |
| 08_profitability_unit_economics.sql | Profitability |

SQL capabilities demonstrated include:

- CTEs
- joins
- conditional aggregation
- LAG()
- RANK()
- PERCENT_RANK()
- NTILE()
- rolling windows
- partitioned rankings
- percentile calculations
- management scoring

---

# Management Intervention Framework

The modelled management score combines:

| Component | Weight |
|---|---:|
| Commercial Exposure | 20% |
| SLA Risk | 25% |
| Delivery Risk | 15% |
| Cancellation Risk | 10% |
| Inventory Risk | 30% |

These weights are analytical assumptions.

---

# DuckDB Warehouse

The analytical warehouse contains 14 validated base tables.

Dimensions:

- dim_customer
- dim_product
- dim_date
- dim_dark_store

Core facts:

- fact_order
- fact_order_item

Operational tables:

- weekly_store_category
- inventory_intelligence
- sku_network_profile
- inventory_assumptions

Profitability tables:

- fact_order_profitability
- fact_order_item_profitability
- dark_store_profitability
- category_profitability

---

# Power BI Dashboard

The planned dashboard consists of five pages:

1. Executive Overview
2. Dark Store Performance
3. Inventory Control Tower
4. Fulfilment & Last Mile
5. Profitability & Decision Intelligence

The analytical engine is completed and validated before dashboard
development so presentation changes cannot alter the underlying
business logic.

---

# Repository Structure

README.md

requirements.txt

.gitignore

data/
- README.md

notebooks/
- India_Quick_Commerce_Profitability_Fulfilment_Intelligence.ipynb
- README.md

sql/
- 01_schema_and_quality.sql
- 02_commercial_performance.sql
- 03_dark_store_operations.sql
- 04_inventory_intelligence.sql
- 05_fulfilment_sla.sql
- 06_customer_experience.sql
- 07_management_priority.sql
- 08_profitability_unit_economics.sql

database/
- quick_commerce_intelligence.duckdb

powerbi/
- README.md
- exports/

docs/
- methodology.md
- kpi_dictionary.md

reports/
- final_validation_report.csv
- project_asset_manifest.csv

---

# Reproducibility

The notebook executes the complete workflow:

1. Data ingestion
2. Data-quality audit
3. Canonical modelling
4. Dark-store simulation
5. Demand analysis
6. Inventory modelling
7. Stockout-risk modelling
8. Profitability modelling
9. DuckDB materialisation
10. SQL analytics
11. Power BI reporting exports
12. Final project validation

---

# Important Limitations

This is a portfolio analytics project rather than an operational system
deployed by a quick-commerce company.

Important limitations:

- dark stores are simulated,
- inventory levels are simulated,
- lead times are assumptions,
- gross margins are assumptions,
- cost-to-serve parameters are assumptions,
- no rider-level operational data is available,
- source customer creation dates are not used for lifecycle analysis,
- some synthetic product names/categories may not be semantically realistic.

Modelled inventory and financial metrics demonstrate methodology rather
than actual company performance.

---

# Skills Demonstrated

## Analytics
- Data quality
- KPI modelling
- Business analysis
- Decision intelligence

## Python
- Pandas
- NumPy
- SciPy
- Feature engineering

## SQL
- DuckDB
- CTEs
- Window functions
- Ranking
- Analytical SQL

## Supply Chain
- ABC/XYZ
- Safety stock
- Reorder point
- Stockout risk

## Operations
- Dark stores
- SLA
- Delivery distance
- Last-mile analytics

## Finance
- Gross margin
- Cost-to-serve
- Contribution economics

## BI
- Power BI
- Semantic modelling
- Management dashboards

---

# Project Status

Analytics Engine: COMPLETE
Data Validation: COMPLETE
DuckDB Warehouse: COMPLETE
SQL Analytics: COMPLETE
Power BI Export Layer: COMPLETE
Power BI Dashboard: NEXT PHASE
'''


README_PATH = PROJECT_ROOT / "README.md"

README_PATH.write_text(
    README_CONTENT.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 3. DATA README
# ============================================================

DATA_README = r'''
# Data Source

This project uses the public:

**Zepto-Inspired Online Grocery & Delivery Dataset**

Source:

https://www.kaggle.com/datasets/ashishjangir6494/zepto-inspired-online-grocery-and-delivery-dataset

## Raw Data Policy

Raw source files are intentionally not redistributed in this repository.

Users who want to reproduce the analysis should download the dataset
from its original Kaggle listing and follow the applicable usage terms.

## Disclosure

This is a Zepto-inspired practice dataset.

It does not represent proprietary Zepto operating data.

Dark stores, inventory positions, replenishment assumptions and
profitability assumptions in this project are analytical simulations.
'''

(DATA_DOCS_DIR / "README.md").write_text(
    DATA_README.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 4. METHODOLOGY
# ============================================================

METHODOLOGY = r'''
# Methodology

## Data Treatment

Raw source records are retained.

Analytical eligibility flags determine which records are valid for
commercial, SLA and customer-experience calculations.

## Commercial Analytics

Orders with transaction information are commercially eligible.

Completed-sales analytics require:

- transaction detail
- Delivered order status

## Customer Experience

Multiple ratings are aggregated to one order-level mean rating.

CX KPIs use delivered rated orders only.

## Analytical SLA

Delivered orders are classified using delivery minutes:

- On Time: <= 35 minutes
- At Risk: 36–50 minutes
- Late: > 50 minutes

## Dark Store Simulation

The dataset contains cities but no real dark-store IDs.

Twenty analytical dark stores are generated deterministically across
ten cities.

## Inventory Grain

Store × SKU daily demand was too sparse for reliable variability
modelling.

Therefore:

- Network × SKU is used for ABC analysis.
- Dark Store × Category × Week is used for inventory variability.

## XYZ Classification

Relative quartiles of coefficient of variation are used.

- X = lowest 25% variability
- Y = middle 50%
- Z = highest 25%

## Inventory

Safety stock and reorder points use:

- weekly demand
- lead-time assumptions
- service-level assumptions
- demand uncertainty

Inventory-on-hand values are simulated deterministically.

## Profitability

Category gross margins and fulfilment costs are scenario assumptions.

Contribution Margin:

Merchandise Gross Profit
minus Variable Cost-to-Serve.

The resulting metrics are modelled contribution economics rather than
actual company financial results.

## Validation

Core metrics are reconciled across:

Python → DuckDB → SQL → Reporting Exports.

The final project contains 18 validation checks.
'''

(DOCS_DIR / "methodology.md").write_text(
    METHODOLOGY.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 5. KPI DICTIONARY
# ============================================================

KPI_DICTIONARY = r'''
# KPI Dictionary

| KPI | Definition |
|---|---|
| Total Orders | All source orders |
| Commercial Orders | Orders containing transaction detail |
| Completed Orders | Delivered orders containing transaction detail |
| Completed GMV | Revenue from completed orders |
| Completed AOV | Completed GMV / Completed Orders |
| SLA Eligible Orders | Delivered orders with delivery-time information |
| SLA On-Time % | SLA-eligible orders delivered within 35 minutes |
| Average Delivery Time | Mean delivery time for SLA-eligible orders |
| P90 Delivery Time | 90th percentile delivery time |
| CX Eligible Orders | Delivered orders with an aggregated rating |
| Average Customer Rating | Mean order-level rating |
| Safety Stock | Demand uncertainty buffer |
| Reorder Point | Lead-time demand + Safety Stock |
| Days of Supply | Inventory on hand / Average Daily Demand |
| Stockout Probability | Probability lead-time demand exceeds inventory |
| Recommended Replenishment | Units required to restore target inventory |
| Merchandise Gross Profit | Revenue × modelled gross-margin rate |
| Variable Cost-to-Serve | Pick/pack + packaging + delivery + payment cost |
| Contribution Margin | Gross profit − variable cost-to-serve |
| Contribution Margin % | Contribution / Merchandise Revenue |
| Negative Contribution Order | Order with modelled contribution below zero |
| Management Intervention Score | Weighted operational decision score |
'''

(DOCS_DIR / "kpi_dictionary.md").write_text(
    KPI_DICTIONARY.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 6. NOTEBOOK README
# ============================================================

NOTEBOOK_README = r'''
# Notebook

Main notebook:

India_Quick_Commerce_Profitability_Fulfilment_Intelligence.ipynb

The notebook covers:

1. Data ingestion
2. Schema and quality audit
3. Canonical modelling
4. Dark-store simulation
5. Inventory analysis
6. Safety stock and reorder points
7. Stockout-risk modelling
8. DuckDB database
9. SQL analytics
10. Profitability and unit economics
11. Reporting exports
12. Final validation

Raw data should be downloaded separately from the original Kaggle
dataset page.
'''

(NOTEBOOKS_DIR / "README.md").write_text(
    NOTEBOOK_README.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 7. POWER BI README
# ============================================================

POWERBI_README = r'''
# Power BI Dashboard

The Power BI dashboard is the final presentation layer.

Planned pages:

1. Executive Overview
2. Dark Store Performance
3. Inventory Control Tower
4. Fulfilment & Last Mile
5. Profitability & Decision Intelligence

The reporting CSV files are generated from the validated analytical
pipeline.

The PBIX file and dashboard screenshots will be added after dashboard
development.
'''

(POWERBI_DIR / "README.md").write_text(
    POWERBI_README.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 8. REQUIREMENTS.TXT
# ============================================================

REQUIREMENTS = r'''
pandas>=2.2
numpy>=2.1
duckdb>=1.3
scipy>=1.14
openpyxl>=3.1
pyarrow>=18.0
'''

(PROJECT_ROOT / "requirements.txt").write_text(
    REQUIREMENTS.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 9. .GITIGNORE
# ============================================================

GITIGNORE = r'''
# Raw source datasets
raw/
data/raw/
*.xlsx

# Source archives
archive.zip

# Python
__pycache__/
*.py[cod]
.ipynb_checkpoints/

# OS files
.DS_Store
Thumbs.db

# Temporary files
*.tmp
*.temp

# Environments
.venv/
venv/
env/
'''

(PROJECT_ROOT / ".gitignore").write_text(
    GITIGNORE.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 10. DOCUMENTATION VALIDATION
# ============================================================

required_documentation = [
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / ".gitignore",
    DATA_DOCS_DIR / "README.md",
    DOCS_DIR / "methodology.md",
    DOCS_DIR / "kpi_dictionary.md",
    NOTEBOOKS_DIR / "README.md",
    POWERBI_DIR / "README.md"
]


documentation_records = []


for path in required_documentation:

    documentation_records.append({
        "file":
            str(
                path.relative_to(PROJECT_ROOT)
            ),

        "exists":
            path.exists(),

        "size_kb":
            round(
                path.stat().st_size / 1024,
                2
            )
            if path.exists()
            else 0
    })


documentation_manifest = pd.DataFrame(
    documentation_records
)


print("=" * 112)
print(" GITHUB DOCUMENTATION ASSETS")
print("=" * 112)

display(
    documentation_manifest
)


if documentation_manifest["exists"].all():

    print(
        "\nGitHub documentation validation: PASS ✅"
    )

else:

    print(
        "\nGitHub documentation validation: REVIEW ⚠️"
    )


print(
    f"\nMain README size: "
    f"{README_PATH.stat().st_size / 1024:.2f} KB"
)

 GITHUB DOCUMENTATION ASSETS


,file,exists,size_kb
0,README.md,True,13.29
1,requirements.txt,True,0.07
2,.gitignore,True,0.22
3,data/README.md,True,0.68
4,docs/methodology.md,True,1.76
5,docs/kpi_dictionary.md,True,1.41
6,notebooks/README.md,True,0.48
7,powerbi/README.md,True,0.39



GitHub documentation validation: PASS ✅

Main README size: 13.29 KB


## 59. GitHub Repository Packaging

A clean public-repository structure is generated from the validated project assets.

The repository includes:

- recruiter-facing documentation,
- standalone SQL analytics,
- DuckDB analytical warehouse,
- validated reporting exports,
- methodology and KPI documentation,
- validation reports,
- and the Power BI project structure.

Raw source data is excluded.

The main notebook is added separately because the active Google Colab notebook is not automatically stored as a normal file inside the runtime filesystem.

In [73]:
# ============================================================
# 59. BUILD CLEAN GITHUB REPOSITORY PACKAGE
# ============================================================

from pathlib import Path
import shutil
import zipfile
import pandas as pd


# ============================================================
# 1. REPOSITORY CONFIGURATION
# ============================================================

REPO_NAME = (
    "india-quick-commerce-profitability-fulfilment-intelligence"
)

GITHUB_REPO_DIR = (
    Path("/content") /
    REPO_NAME
)

GITHUB_ZIP_PATH = (
    Path("/content") /
    f"{REPO_NAME}.zip"
)


# ------------------------------------------------------------
# Start clean
# ------------------------------------------------------------

if GITHUB_REPO_DIR.exists():
    shutil.rmtree(
        GITHUB_REPO_DIR
    )

if GITHUB_ZIP_PATH.exists():
    GITHUB_ZIP_PATH.unlink()


# ============================================================
# 2. CREATE REPOSITORY STRUCTURE
# ============================================================

repo_directories = [
    GITHUB_REPO_DIR / "data",
    GITHUB_REPO_DIR / "notebooks",
    GITHUB_REPO_DIR / "sql",
    GITHUB_REPO_DIR / "database",
    GITHUB_REPO_DIR / "powerbi",
    GITHUB_REPO_DIR / "powerbi" / "exports",
    GITHUB_REPO_DIR / "docs",
    GITHUB_REPO_DIR / "reports"
]


for directory in repo_directories:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 3. COPY ROOT DOCUMENTATION
# ============================================================

root_files = [
    "README.md",
    "requirements.txt",
    ".gitignore"
]


for file_name in root_files:

    source = (
        PROJECT_ROOT /
        file_name
    )

    destination = (
        GITHUB_REPO_DIR /
        file_name
    )

    shutil.copy2(
        source,
        destination
    )


# ============================================================
# 4. COPY DOCUMENTATION
# ============================================================

documentation_copies = {

    DATA_DOCS_DIR / "README.md":
        GITHUB_REPO_DIR /
        "data" /
        "README.md",

    NOTEBOOKS_DIR / "README.md":
        GITHUB_REPO_DIR /
        "notebooks" /
        "README.md",

    POWERBI_DIR / "README.md":
        GITHUB_REPO_DIR /
        "powerbi" /
        "README.md",

    DOCS_DIR / "methodology.md":
        GITHUB_REPO_DIR /
        "docs" /
        "methodology.md",

    DOCS_DIR / "kpi_dictionary.md":
        GITHUB_REPO_DIR /
        "docs" /
        "kpi_dictionary.md"
}


for source, destination in (
    documentation_copies.items()
):

    shutil.copy2(
        source,
        destination
    )


# ============================================================
# 5. COPY SQL FILES
# ============================================================

for sql_file in sorted(
    SQL_DIR.glob("*.sql")
):

    shutil.copy2(
        sql_file,
        GITHUB_REPO_DIR /
        "sql" /
        sql_file.name
    )


# ============================================================
# 6. COPY DUCKDB DATABASE
# ============================================================

DATABASE_SOURCE = (
    DATABASE_DIR /
    "quick_commerce_intelligence.duckdb"
)

DATABASE_DESTINATION = (
    GITHUB_REPO_DIR /
    "database" /
    "quick_commerce_intelligence.duckdb"
)

shutil.copy2(
    DATABASE_SOURCE,
    DATABASE_DESTINATION
)


# ============================================================
# 7. COPY POWER BI EXPORTS
# ============================================================

for csv_file in sorted(
    POWERBI_EXPORT_DIR.glob("*.csv")
):

    shutil.copy2(
        csv_file,
        GITHUB_REPO_DIR /
        "powerbi" /
        "exports" /
        csv_file.name
    )


# ============================================================
# 8. COPY VALIDATION REPORTS
# ============================================================

for report_file in sorted(
    REPORTS_DIR.glob("*.csv")
):

    shutil.copy2(
        report_file,
        GITHUB_REPO_DIR /
        "reports" /
        report_file.name
    )


# ============================================================
# 9. CREATE NOTEBOOK PLACEHOLDER
#
# This makes the required notebook filename explicit.
# We will replace this placeholder with the actual .ipynb.
# ============================================================

NOTEBOOK_TARGET_NAME = (
    "India_Quick_Commerce_Profitability_"
    "Fulfilment_Intelligence.ipynb"
)

NOTEBOOK_TARGET_PATH = (
    GITHUB_REPO_DIR /
    "notebooks" /
    NOTEBOOK_TARGET_NAME
)


NOTEBOOK_INSTRUCTION_PATH = (
    GITHUB_REPO_DIR /
    "notebooks" /
    "ADD_NOTEBOOK_HERE.txt"
)


NOTEBOOK_INSTRUCTION_PATH.write_text(
    (
        "Add the final Google Colab notebook here using the "
        "following exact filename:\n\n"
        f"{NOTEBOOK_TARGET_NAME}\n\n"
        "Do not upload the raw source workbook to the repository.\n"
    ),
    encoding="utf-8"
)


# ============================================================
# 10. VALIDATE RAW-DATA EXCLUSION
# ============================================================

forbidden_extensions = {
    ".xlsx",
    ".xls"
}


raw_source_files = []


for path in GITHUB_REPO_DIR.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() in forbidden_extensions:

        raw_source_files.append(
            str(
                path.relative_to(
                    GITHUB_REPO_DIR
                )
            )
        )


# ============================================================
# 11. BUILD REPOSITORY MANIFEST
# ============================================================

repo_manifest_records = []


for path in sorted(
    GITHUB_REPO_DIR.rglob("*")
):

    if not path.is_file():
        continue

    repo_manifest_records.append({

        "relative_path":
            str(
                path.relative_to(
                    GITHUB_REPO_DIR
                )
            ),

        "extension":
            path.suffix.lower(),

        "size_kb":
            round(
                path.stat().st_size /
                1024,
                2
            )
    })


repo_manifest = pd.DataFrame(
    repo_manifest_records
)


# ============================================================
# 12. REPOSITORY VALIDATION
# ============================================================

required_repo_files = [

    GITHUB_REPO_DIR /
    "README.md",

    GITHUB_REPO_DIR /
    "requirements.txt",

    GITHUB_REPO_DIR /
    ".gitignore",

    GITHUB_REPO_DIR /
    "data" /
    "README.md",

    GITHUB_REPO_DIR /
    "docs" /
    "methodology.md",

    GITHUB_REPO_DIR /
    "docs" /
    "kpi_dictionary.md",

    GITHUB_REPO_DIR /
    "notebooks" /
    "README.md",

    GITHUB_REPO_DIR /
    "powerbi" /
    "README.md",

    GITHUB_REPO_DIR /
    "database" /
    "quick_commerce_intelligence.duckdb",

    GITHUB_REPO_DIR /
    "reports" /
    "final_validation_report.csv"
]


missing_required_files = [
    str(
        path.relative_to(
            GITHUB_REPO_DIR
        )
    )
    for path in required_repo_files
    if not path.exists()
]


repo_sql_files = list(
    (
        GITHUB_REPO_DIR /
        "sql"
    ).glob("*.sql")
)


repo_export_files = list(
    (
        GITHUB_REPO_DIR /
        "powerbi" /
        "exports"
    ).glob("*.csv")
)


# ============================================================
# 13. PRINT VALIDATION
# ============================================================

print("=" * 118)
print(" GITHUB REPOSITORY PACKAGE")
print("=" * 118)

print(
    f"Repository name : "
    f"{REPO_NAME}"
)

print(
    f"Repository path : "
    f"{GITHUB_REPO_DIR}"
)

print(
    f"Files packaged  : "
    f"{len(repo_manifest):,}"
)

print(
    f"SQL files       : "
    f"{len(repo_sql_files):,}"
)

print(
    f"Reporting CSVs  : "
    f"{len(repo_export_files):,}"
)

print(
    f"Raw Excel files : "
    f"{len(raw_source_files):,}"
)

print(
    f"Missing required files : "
    f"{len(missing_required_files):,}"
)


print("\nRepository manifest:")

display(
    repo_manifest
)


# ============================================================
# 14. VALIDATION STATUS
# ============================================================

package_pass = (
    len(repo_sql_files) == 8
    and
    len(repo_export_files) == 17
    and
    len(raw_source_files) == 0
    and
    len(missing_required_files) == 0
)


if package_pass:

    print(
        "\nGitHub repository package validation: PASS ✅"
    )

else:

    print(
        "\nGitHub repository package validation: REVIEW ⚠️"
    )

    if missing_required_files:

        print(
            "\nMissing required files:"
        )

        for item in missing_required_files:
            print(
                "-",
                item
            )

    if raw_source_files:

        print(
            "\nRaw data files found:"
        )

        for item in raw_source_files:
            print(
                "-",
                item
            )


# ============================================================
# 15. CREATE ZIP PACKAGE
# ============================================================

with zipfile.ZipFile(
    GITHUB_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for path in sorted(
        GITHUB_REPO_DIR.rglob("*")
    ):

        if not path.is_file():
            continue

        archive_name = (
            Path(REPO_NAME)
            /
            path.relative_to(
                GITHUB_REPO_DIR
            )
        )

        zip_file.write(
            path,
            archive_name
        )


# ============================================================
# 16. ZIP VALIDATION
# ============================================================

with zipfile.ZipFile(
    GITHUB_ZIP_PATH,
    "r"
) as zip_file:

    zip_contents = (
        zip_file.namelist()
    )


github_zip_size_mb = (
    GITHUB_ZIP_PATH.stat().st_size /
    (1024 ** 2)
)


print("\n" + "=" * 118)
print(" GITHUB ZIP PACKAGE")
print("=" * 118)

print(
    f"ZIP path       : "
    f"{GITHUB_ZIP_PATH}"
)

print(
    f"Files in ZIP   : "
    f"{len(zip_contents):,}"
)

print(
    f"ZIP size       : "
    f"{github_zip_size_mb:.2f} MB"
)


if (
    GITHUB_ZIP_PATH.exists()
    and
    package_pass
):

    print(
        "\nGitHub ZIP validation: PASS ✅"
    )

else:

    print(
        "\nGitHub ZIP validation: REVIEW ⚠️"
    )


# ============================================================
# 17. NOTEBOOK STATUS
# ============================================================

print("\n" + "=" * 118)
print(" NOTEBOOK STATUS")
print("=" * 118)

print(
    "Required notebook filename:"
)

print(
    NOTEBOOK_TARGET_NAME
)

print(
    "\nCurrent status:"
)

print(
    "Notebook still needs to be added manually before "
    "the final GitHub upload."
)

 GITHUB REPOSITORY PACKAGE
Repository name : india-quick-commerce-profitability-fulfilment-intelligence
Repository path : /content/india-quick-commerce-profitability-fulfilment-intelligence
Files packaged  : 37
SQL files       : 8
Reporting CSVs  : 17
Raw Excel files : 0
Missing required files : 0

Repository manifest:


,relative_path,extension,size_kb
0,.gitignore,,0.22
1,README.md,.md,13.29
2,data/README.md,.md,0.68
3,database/quick_commerce_intelligence.duckdb,.duckdb,"9,996.00"
4,docs/kpi_dictionary.md,.md,1.41
5,docs/methodology.md,.md,1.76
6,notebooks/ADD_NOTEBOOK_HERE.txt,.txt,0.20
7,notebooks/README.md,.md,0.48
8,powerbi/README.md,.md,0.39
9,powerbi/exports/00_executive_kpis.csv,.csv,0.50



GitHub repository package validation: PASS ✅

 GITHUB ZIP PACKAGE
ZIP path       : /content/india-quick-commerce-profitability-fulfilment-intelligence.zip
Files in ZIP   : 37
ZIP size       : 3.48 MB

GitHub ZIP validation: PASS ✅

 NOTEBOOK STATUS
Required notebook filename:
India_Quick_Commerce_Profitability_Fulfilment_Intelligence.ipynb

Current status:
Notebook still needs to be added manually before the final GitHub upload.
